<a href="https://colab.research.google.com/github/kvdep/Analysing-machine-learning-citations-graph-using-neural-networks/blob/main/%D0%9A%D1%83%D1%80%D1%81%D0%BE%D0%B2%D0%B0%D1%8F_2026_%D0%9F%D1%80%D0%B8%D0%BC%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BC%D0%B5%D1%82%D0%BE%D0%B4%D0%BE%D0%B2_%D0%BC%D0%B0%D1%88%D0%B8%D0%BD%D0%BD%D0%BE%D0%B3%D0%BE_%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D1%8F_%D0%B4%D0%BB%D1%8F_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0_%D1%81%D0%B5%D1%82%D0%B5%D0%B9_%D1%86%D0%B8%D1%82%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F_%D0%BD%D0%B0%D1%83%D1%87%D0%BD%D1%8B%D1%85_%D1%80%D0%B0%D0%B1%D0%BE%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports

In [ ]:
!pip install torch_geometric catboost -qqq

In [ ]:
# @title
## 1. Общие
import os
import gc
import json
import time
import uuid
import math
import random
import pickle
import shutil
import psutil
import inspect
import datetime
import itertools
import subprocess
import collections
import numpy as np
import pandas as pd
import catboost as cb
import scipy.sparse as sp;
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
## 2. Работа с данными
import threading
import polars as pl
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
import polars.selectors as cs
import requests as rq
## 3. torch
import torch
import torch.nn as nn
import torch.optim as op
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
from torch_geometric.data import HeteroData
from torch_geometric.nn import Node2Vec
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.loader import DataLoader
from torch_geometric.utils import k_hop_subgraph
from sentence_transformers import SentenceTransformer as ST, util as ut
## 4. Sklearn
import sklearn.metrics as sm
## 5. Графы
import networkx as nx
## 6. Визуализация данных
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe

d = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def mount_drive(b_p='/content/drive/MyDrive/ВУЗ/КУРС 3/КУРСОВАЯ'):
    from google.colab import drive; import os
    drive.mount('/content/drive')
    os.makedirs(b_p, exist_ok=True)
    return b_p

b_p = mount_drive() #Путь к целевой рабочей папке проекта на диске

# Классы и функции

## Логи

In [ ]:
def update_scoreboard(met, exp_n, b_p):
  '''
  Обновляет общую таблицу результатов экспериментов.
  met (dict): Словарь рассчитанных метрик эффективности (ROC, PR, NDCG@10 и т.д.).
  exp_n (str): Строковое имя текущего эксперимента.
  b_p (str): Путь к рабочей директории проекта, где лежит файл scoreboard.csv.
  '''

  sb_p = os.path.join(b_p, 'scoreboard.csv')
  d_r = {"Дата": datetime.datetime.now().strftime('%Y-%m-%d %H:%M'), "Эксперимент": exp_n}
  d_r.update({k: round(v, 4) for k, v in met.items() if isinstance(v, float)})
  df_n = pl.DataFrame([d_r])
  if os.path.exists(sb_p):
      df_o = pl.read_csv(sb_p); df = pl.concat([df_o, df_n], how="diagonal")
  else: df = df_n
  df.write_csv(sb_p)

def save_experiment(m, kwargs_dict, met, p_path, b_p, exp_n):
  '''
  Сохраняет веса, метаданные и визуализации завершенного обучения.
  m (nn.Module): Обученный экземпляр нейросетевой модели.
  kwargs_dict (dict): Словарь параметров, с которыми инициализировалась модель m.
  met (dict): Словарь итоговых метрик на тестовой выборке.
  p_path (str): Локальный путь к графику PR-кривой (будет скопирован в архив эксперимента).
  b_p (str): Базовый путь к директории проекта.
  exp_n (str): Уникальное имя папки для сохранения текущего эксперимента.
  '''

  e_d = os.path.join(b_p, 'experiments', exp_n); os.makedirs(e_d, exist_ok=True)
  torch.save({'m': m.state_dict(), 'kw': kwargs_dict}, os.path.join(e_d, 'weights.pt'))
  with open(os.path.join(e_d, 'metrics.json'), 'w', encoding='utf-8') as f: json.dump(met, f, ensure_ascii=False, indent=4)
  if os.path.exists(p_path): shutil.copy(p_path, os.path.join(e_d, 'eval_plot.png'))
  update_scoreboard(met, exp_n, b_p)
  print(f"[{time.strftime('%H:%M:%S')}] Эксп. сохранен: {e_d}")



## Сохранение между сессиями

In [ ]:
def load_model(e_d, d_dv):
  '''
  Загружает веса и конфигурацию модели из архива.
  e_d (str): Путь к директории конкретного эксперимента.
  d_dv (torch.device)
  '''

  w_p = os.path.join(e_d, 'weights.pt')
  if not os.path.exists(w_p): raise RuntimeError(f"ОШИБКА: Веса не найдены по пути {w_p}")
  print(f"[{time.strftime('%H:%M:%S')}] Чтение контрольной точки: {w_p}")
  ckpt = torch.load(w_p, map_location=d_dv); kw = ckpt['kw']
  m = GM(**kw).to(d_dv); m.load_state_dict(ckpt['m']); m.eval()
  return m, kw


class cb_log:
  '''
  Класс-обратный вызов (callback) для логирования прогресса обучения моделей градиентного бустинга CatBoost.
  e_step (int, по умолчанию 50): Частота вывода логов на экран (шаг итераций).
  '''

  def __init__(self, e_step=50): self.e_step = e_step

  def after_iteration(self, info):
      e = info.iteration
      if e % self.e_step == 0 or e == info.iterations_count - 1:
          m_k = list(info.metrics['learn'].keys())[0]
          tr_l = info.metrics['learn'][m_k][-1]; vl_l = info.metrics['validation'][m_k][-1]
          t_p = info.passed_time; h = int(t_p // 3600); m = int((t_p % 3600) // 60); s = int(t_p % 60)
          print(f'[{h:02d}:{m:02d}:{s:02d}] epoch {e}\n L Train: {tr_l:.4f}\n L Val: {vl_l:.4f}\n Time: {t_p:.2f}')
      return True


## Сбор и парсинг

In [ ]:
def fetch_year(year_val, target_n_per_year):
  '''
  Пакетно запрашивает научные публикации за конкретный год через OpenAlex API.
  year_val (int): Год публикации, для которого собираются данные.
  target_n_per_year (int): Лимит количества собираемых статей за этот год
  '''
  u = "https://api.openalex.org/works"; results_year = []; cursor = "*"; s_year = set()
  while len(results_year) < target_n_per_year:
      print(f'year: {year_val}, {len(results_year)}')
      pm = {"filter": f"publication_year:{year_val},has_abstract:true", "per-page": 100, "cursor": cursor, "mailto": "a@a.com"}
      resp = rq.get(u, params=pm, timeout=15)
      if resp.status_code != 200: raise RuntimeError(f"API Error {resp.status_code} for year {year_val}")
      j = resp.json(); works = j.get('results', [])
      if not works: break
      for w in works:
          ai = w.get('abstract_inverted_index')
          if not ai or w['id'] in s_year: continue
          s_year.add(w['id'])
          max_idx = max([p for pos in ai.values() if isinstance(pos, list) for p in pos], default=-1)
          if max_idx == -1: continue
          ab_arr = [""] * (max_idx + 1)
          for word, positions in ai.items():
              if isinstance(positions, list):
                  for pos in positions: ab_arr[pos] = word
          ab = " ".join(ab_arr).strip()
          results_year.append({'i': w['id'], 'y': w['publication_year'], 'ti': w.get('title'), 'na': len(w['authorships']), 'c': w.get('cited_by_count', 0), 'ab': ab, 'au': [a['author']['id'] for a in w['authorships'] if 'author' in a], 'ct': " ".join([x['display_name']for x in w.get('concepts', [])]), 'rf': w.get('referenced_works', [])})
          if len(results_year) >= target_n_per_year: return results_year
      cursor = j.get('meta', {}).get('next_cursor')
      if not cursor: break
  return results_year

def f_yr(y, n_t, c_ids, f_out, lock):
  '''
  Потоковый обработчик запросов OpenAlex для одного года.
  y (int): Целевой год публикации.
  n_t (int): Лимит записей за год.
  c_ids (list): Список идентификаторов предметных концептов OpenAlex (фильтрация).
  f_out (file object): Открытый файл на диске, куда в реальном времени дописываются строки JSONL.
  lock (threading.Lock): Потоковый замок для предотвращения конфликтов одновременной записи в файл из разных потоков
  '''

  u = "https://api.openalex.org/works"; c = "*"; s = set(); r_c = 0
  flt = f"publication_year:{y},has_abstract:true"
  if c_ids: flt += f",concepts.id:{'|'.join(c_ids)}"
  while r_c < n_t:
      pm = {"filter": flt, "per-page": 100, "cursor": c, "mailto": "a@a.com"}
      p = rq.get(u, params=pm, timeout=15)
      if p.status_code != 200: raise RuntimeError(f"API Error {p.status_code} for year {y}")
      j = p.json(); w = j.get('results', [])
      if not w: break
      b_d = []
      for x in w:
          ai = x.get('abstract_inverted_index')
          if not ai or x['id'] in s: continue
          s.add(x['id'])
          max_idx = max([p for pos in ai.values() if isinstance(pos, list) for p in pos], default=-1)
          if max_idx == -1: continue
          ab_arr = [""] * (max_idx + 1)
          for word, positions in ai.items():
              if isinstance(positions, list):
                  for pos in positions: ab_arr[pos] = word
          ab = " ".join(ab_arr).strip()
          au = [a['author']['id'] for a in x.get('authorships', []) if 'author' in a]
          ct = " ".join([cn['display_name'] for cn in x.get('concepts', [])])
          rf = x.get('referenced_works', [])
          b_d.append(json.dumps({'i': x['id'], 'y': x['publication_year'], 'ti': x.get('title'), 'ab': ab, 'ct': ct, 'au': au, 'rf': rf}))
      if b_d:
          with lock: f_out.write("\n".join(b_d) + "\n")
          r_c += len(b_d)
      print(f"[{y}] Собрано: {r_c}/{n_t}")
      c = j.get('meta', {}).get('next_cursor')
      if not c: break

def g_d(n, y_range):
  '''
  Оркестрирует параллельный сбор исторических данных за промежуток лет.
  n (int): Суммарный объем выборки статей.
  y_range (int): Количество лет ретроспективы (глубина сбора назад от 2026 года)
  '''

  all_data = []; start_year = 2026; years = [start_year - i for i in range(y_range)]
  with ThreadPoolExecutor(max_workers=min(y_range, 10)) as executor:
      futures = {executor.submit(fetch_year, y, n // y_range + 100): y for y in years}
      for f in as_completed(futures):
          res = f.result(); all_data.extend(res); print(f"Год {futures[f]}: получено {len(res)}")
          if len(all_data) >= n: break
  return all_data[:n]

def f_ai(n, y_r=10, c_ids=None, p_file='raw_data.jsonl'):
  '''
  Многопоточный менеджер сбора данных с сохранением в JSONL-файл.
  n (int): Суммарное количество статей.
  y_r (int, по умолчанию 10): Глубина сбора данных в годах.
  c_ids (list, по умолчанию None): Перечень ID концептов OpenAlex для тематической фильтрации.
  p_file (str, по умолчанию 'raw_data.jsonl'): Имя результирующего JSONL-файла на диске
  '''

  ys = [2026 - i for i in range(y_r)]; lock = threading.Lock()
  with open(p_file, 'w', encoding='utf-8') as f_out:
      with ThreadPoolExecutor(max_workers=min(y_r, 10)) as ex:
          fs = {ex.submit(f_yr, y, n // y_r + 500, c_ids, f_out, lock): y for y in ys}
          for f in as_completed(fs): pass
  print(f"Сбор завершен. Данные сохранены в {p_file}")
  return p_file

def b_raw_g(p_file):
  '''
  Формирует направленный граф цитирований NetworkX из JSONL-файла.
  p_file (str): Путь к сырому JSONL-файлу
  '''

  g = nx.DiGraph(); s_i = set()
  with open(p_file, 'r', encoding='utf-8') as f:
      for ln in f:
          if ln.strip(): s_i.add(json.loads(ln)['i'])
  with open(p_file, 'r', encoding='utf-8') as f:
      for ln in f:
          if not ln.strip(): continue
          x = json.loads(ln)
          g.add_node(x['i'], y=x['y'], ct=x['ct'], au=x.get('au', []))
          for rf in x['rf']:
              if rf in s_i: g.add_edge(x['i'], rf, cit=1)
  g.remove_edges_from(list(nx.selfloop_edges(g)))
  return g

def extract_metadata(p):
  '''
  Извлекает текстовые метаданные статей для качественного анализа ошибок предсказаний.
  p (str): Путь к исходному JSONL-файлу
  '''
  m_d = {}
  with open(p, 'r', encoding='utf-8') as f:
      for ln in f:
          if not ln.strip(): continue
          x = json.loads(ln)
          m_d[x['i']] = {'ti': x.get('ti', 'Без названия'), 'ab': x.get('ab', '')[:300], 'ct': x.get('ct', ''), 'y': x.get('y', 0)}
  return m_d

def k_rw(m, i, f_d=None, t='Dataset Data', p='./k_tmp'):
  '''
  Обеспечивает чтение и запись датасетов через Kaggle API.
  m (str): Режим работы. 'r' — чтение (скачивание), 'w' — запись (создание новой версии датасета на Kaggle).
  i (str): Уникальный идентификатор Kaggle-датасета (в формате username/dataset-name).
  f_d (dict, по умолчанию None): Словарь сохраняемых объектов {имя_файла: объект} (используется только при m='w').
  t (str, по умолчанию 'Dataset Data'): Описание новой версии датасета на Kaggle.
  p (str, по умолчанию './k_tmp'): Путь к локальной временной папке для распаковки или архивирования данных.
  '''
  os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
  os.makedirs(p, exist_ok=True); i = i.lower()
  if m == 'w':
      for f_n, obj in f_d.items():
          tgt = os.path.join(p, f_n)
          if isinstance(obj, str) and os.path.exists(obj):
              if obj != tgt: shutil.copy(obj, tgt)
          elif f_n.endswith('.pt'): torch.save(obj, tgt)
          else:
              with open(tgt, 'wb') as f: pickle.dump(obj, f)
      with open(os.path.join(p, 'dataset-metadata.json'), 'w') as f: json.dump({"title": t, "id": i, "licenses": [{"name": "CC0-1.0"}]}, f)
      c = subprocess.run(["kaggle", "datasets", "status", i], capture_output=True, text=True)
      cmd = f'kaggle datasets version -p {p} -m "upd" --dir-mode zip' if c.returncode == 0 else f'kaggle datasets create -p {p} --dir-mode zip'
      res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
      if res.returncode != 0: raise RuntimeError(f"Kaggle Error: {res.stderr}")
      return True
  elif m == 'r':
      if not any(f.endswith(('.pkl', '.jsonl', '.pt')) for f in os.listdir(p)): subprocess.run(f'kaggle datasets download -d {i} -p {p} --unzip', shell=True, check=True)
      r = {}
      for f_n in os.listdir(p):
          fp = os.path.join(p, f_n)
          if f_n.endswith('.pt'): r[f_n] = torch.load(fp, weights_only=False)
          elif f_n.endswith('.jsonl') or f_n.endswith('.csv'): r[f_n] = fp
          elif f_n.endswith('.pkl'):
              with open(fp, 'rb') as f: r[f_n] = pickle.load(f)
      return r

def get_p(n): return f"./{n.split('/')[-1].replace('-', '_')}"



## Анализ графа

In [ ]:
def b_f_g(d, g, k_c=2, pref='e', n_t=1):
  '''
  Очищает граф цитирования методом k-core прунинга.
  d (str или list): Источник сырых данных (путь к JSONL или список записей).
  g (nx.DiGraph): Исходный грязный граф цитирований.
  k_c (int, по умолчанию 2): Порог отсечения вершин по степени k-core (вершины со степенью < k_c будут отброшены).
  pref (str, по умолчанию 'e'): Метрика ранжирования связных компонент графа ('e' — по количеству ребер, 'n' — по количеству узлов).
  n_t (int, по умолчанию 1): Количество топовых связных компонент, оставляемых после прунинга
  '''

  print(f"[{time.strftime('%H:%M:%S')}] До прунинга: Узлы={g.number_of_nodes()} Ребра={g.number_of_edges()}")
  g_k = nx.k_core(g, k_c).copy(); w_c = list(nx.weakly_connected_components(g_k))
  c_s = []
  for i, cmp in enumerate(w_c):
      c_s.append({'id': i, 'n': len(cmp), 'e': g_k.subgraph(cmp).number_of_edges(), 'nd': cmp})
  if pref == 'e': c_s.sort(key=lambda x: x['e'], reverse=True)
  else: c_s.sort(key=lambda x: x['n'], reverse=True)
  s_i = {x['id'] for x in c_s[:n_t]}
  df_d = []; cm_a = {}
  for x in c_s:
      st = "ВЫБРАН" if x['id'] in s_i else "Отброшен"
      df_d.append({"ID кластера": str(x['id']), "Кол-во узлов": x['n'], "Кол-во ребер": x['e'], "Статус": st})
      for nd in x['nd']: cm_a[nd] = x['id']
  display(pl.DataFrame(df_d))
  t_c = set()
  for x in c_s[:n_t]: t_c.update(x['nd'])
  g_p = g_k.subgraph(t_c).copy()

  print(f"[{time.strftime('%H:%M:%S')}] Инжиниринг признаков (CPU)...")
  s_w = {'computer', 'science', 'machine', 'learning', 'of', 'the', 'and', 'in', 'for', 'a', 'an', 'on', 'to', 'with', 'based', 'using'}
  for n, d_n in g_p.nodes(data=True):
      ct_l = [w for w in d_n.get('ct', '').lower().split() if w not in s_w]
      d_n['ct_f'] = " ".join(ct_l)
      d_n['in_d'] = g_p.in_degree(n)
      d_n.pop('ct', None)
  for u, v, e_d in g_p.edges(data=True):
      e_d['dt'] = abs(g_p.nodes[u]['y'] - g_p.nodes[v]['y'])
      au_u = set(g_p.nodes[u].get('au', [])); au_v = set(g_p.nodes[v].get('au', []))
      un = len(au_u | au_v)
      e_d['au_o'] = len(au_u & au_v) / un if un > 0 else 0.0

  p_f = 'pruned_data.jsonl'
  with open(p_f, 'w', encoding='utf-8') as f_out:
      if isinstance(d, str):
          with open(d, 'r', encoding='utf-8') as f_in:
              for ln in f_in:
                  if not ln.strip(): continue
                  if json.loads(ln)['i'] in g_p: f_out.write(ln)
      else:
          for x in d:
              if x['i'] in g_p: f_out.write(json.dumps(x) + '\n')
  print(f"[{time.strftime('%H:%M:%S')}] После прунинга (pruned-graph): Узлы={g_p.number_of_nodes()} Ребра={g_p.number_of_edges()}")
  return p_f, g_p, g_k, cm_a, s_i

def a_c(g, cm):
  '''
  Анализирует свойства сообществ Лувена.
  g (nx.DiGraph): Очищенный граф цитирований.
  cm (dict): Словарь принадлежности вершин сообществам {node_id: community_id}
  '''

  if not g.nodes or not cm: return {}
  c = collections.defaultdict(list); [c[v].append(k) for k, v in cm.items() if g.has_node(k)]
  print(f"\nВСЕГО ВЫДЕЛЕНО ПОД-СООБЩЕСТВ (Лувен): {len(c)}")
  #ks = sorted(list(c.keys()), key=lambda x: len(c[x]), reverse=True)[:15]
  ks = sorted(list(c.keys()), key=lambda x: len(c[x]), reverse=True)
  k_i = {k: i for i, k in enumerate(ks)}; m_e = np.zeros((len(ks), len(ks)), dtype=int)
  for u, v in g.edges():
      if cm.get(u) in k_i and cm.get(v) in k_i: m_e[k_i[cm[u]], k_i[cm[v]]] += 1
  r = {}
  for k in ks:
      v_n = [n for n in c[k] if n in g.nodes()]
      if not v_n: continue
      v_set = set(v_n)
      y_a = [g.nodes[n].get('y', 0) for n in v_n]; y = [yr for yr in y_a if yr > 0]
      t = [g.nodes[n].get('ct_f', '') for n in v_n if g.nodes[n].get('ct_f')]
      w = collections.Counter(" ".join(t).split()).most_common(5) if t else []
      s_g = g.subgraph(v_n)
      e_in = s_g.number_of_edges()
      e_out = sum(1 for u, v in g.edges(v_n) if v not in v_set) + sum(1 for u, v in g.in_edges(v_n) if u not in v_set)
      den = nx.density(s_g); vol = e_in + e_out; cond = e_out / vol if vol > 0 else 0.0
      r[k] = {'n': len(v_n), 'e_in': e_in, 'e_out': e_out, 'den': den, 'cond': cond, 'y': np.median(y) if y else 0, 'w': w}
  if not r: return r
  c_s_cl = [{n for n in c[k] if n in g.nodes()} for k in ks if k in c]; c_s_cl = [comp for comp in c_s_cl if comp]
  sub_g = g.subgraph([n for comp in c_s_cl for n in comp])
  try: mod = nx.community.modularity(sub_g, c_s_cl)
  except: mod = 0.0
  print(f"Модулярность отрисованных под-сообществ: {mod:.4f}\n")
  df_d = [{"Под-сообщество": str(k), "Узлы": r[k]['n'], "Внутр. ребра": r[k]['e_in'], "Внеш. ребра": r[k]['e_out'], "Плотность": round(r[k]['den'], 4), "Год": int(r[k]['y'])} for k in ks]
  display(pl.DataFrame(df_d))
  cm_p = plt.colormaps['tab20']; cl = [cm_p(i / len(ks) if len(ks) > 1 else 0) for i in range(len(ks))]; x_lbl = [str(k) for k in ks]
  plt.figure(figsize=(15, 5)); plt.bar(x_lbl, [r[k]['n'] for k in ks], color=cl); plt.title('Кол-во узлов', fontsize=14); plt.show(); plt.close()
  plt.figure(figsize=(15, 5)); plt.bar(x_lbl, [r[k]['e_in'] for k in ks], color=cl); plt.title('Внутренние ребра', fontsize=14); plt.show(); plt.close()
  plt.figure(figsize=(15, 5)); plt.bar(x_lbl, [r[k]['e_out'] for k in ks], color=cl); plt.title('Внешние ребра', fontsize=14); plt.show(); plt.close()
  plt.figure(figsize=(24, 20)); plt.imshow(m_e, cmap='Blues', interpolation='nearest'); plt.colorbar(); plt.xticks(range(len(ks)), x_lbl); plt.yticks(range(len(ks)), x_lbl)
  plt.title('Матрица связей (Под-сообщества)', fontsize=14)
  for i in range(len(ks)):
      for j in range(len(ks)): plt.text(j, i, str(m_e[i, j]), ha="center", va="center", color="white" if m_e[i, j] > m_e.max()/2 else "black")
  plt.show(); plt.close()
  print(f"\n{'ID':<5} | Ключевые концепты\n" + "-"*80)
  for k in ks: print(f"{str(k):<5} | " + ", ".join([f"{cw[0]} ({cw[1]})" for cw in r[k]['w']]))
  del sub_g, m_e, df_d; gc.collect()
  return r

def v_c(g, cm):
  '''
  Отрисовывает пространственную проекцию сообществ графа.
  g (nx.Graph или nx.DiGraph): Визуализируемый граф.
  cm (dict): Словарь распределения узлов по кластерам
  '''

  if g.number_of_nodes() == 0: return
  m_n = 1500
  if g.number_of_nodes() > m_n:
      t_n = sorted(dict(g.degree()).items(), key=lambda x: x[1], reverse=True)[:m_n]
      g = g.subgraph([n for n, d in t_n]).copy()
  else: g = g.copy()
  for u, v, d in g.edges(data=True): d['w_v'] = 10.0 if cm.get(u) == cm.get(v) else 0.1
  p = nx.spring_layout(g, k=4/np.sqrt(g.number_of_nodes()), weight='w_v', iterations=120)
  plt.figure(figsize=(14, 14)); all_c = sorted(list(set(cm.values())))
  c_u = sorted(list(set([cm.get(n) for n in g.nodes if n in cm])))
  cm_p = plt.colormaps['tab20']
  for c_i in c_u:
      c_idx = all_c.index(c_i); n_c = [n for n in g.nodes if cm.get(n) == c_i]
      if not n_c: continue
      nx.draw_networkx_nodes(g, p, nodelist=n_c, node_color=[cm_p(c_idx / len(all_c) if len(all_c) > 1 else 0)], node_size=45, alpha=0.9, label=f"Под-сообщество {c_i}")
  nx.draw_networkx_edges(g, p, alpha=0.15, edge_color='gray')
  plt.title('Внутренняя структура кластера (алгоритм Лувена)', fontsize=16)
  plt.legend(loc='best', fontsize=12, markerscale=1.5); plt.axis('off'); plt.show()




### Проверки качества сбо

## Feature Engineering

### Семантика

In [ ]:
def build_scibert_embeddings(d_p, g, raw_p, d_dv):
  '''
  Извлекает высокоразмерные (2304) сырые эмбеддинги SciBERT в FP16.
  d_p, g, raw_p — аналогично build_ae_embeddings.
  d_dv (torch.device): Устройство, на котором инициализируется модель SciBERT
  '''

  print(f"[{time.strftime('%H:%M:%S')}] Извлечение Title, Abstract, Concepts")
  ti_d = {}; ab_d = {}; ct_d = {}

  with open(raw_p, 'r', encoding='utf-8') as f:
      for ln in f:
          if not ln.strip(): continue
          x = json.loads(ln)
          if x['i'] in g.nodes():
              ti_d[x['i']] = x.get('ti', '') or ''
              ab_d[x['i']] = x.get('ab', '') or ''

  with open(d_p, 'r', encoding='utf-8') as f:
      for ln in f:
          if not ln.strip(): continue
          x = json.loads(ln)
          ct_d[x['i']] = x.get('ct_f', '') or ''

  nodes = list(g.nodes())
  ti_l = [ti_d.get(n, "") for n in nodes]
  ab_l = [ab_d.get(n, "") for n in nodes]
  ct_l = [ct_d.get(n, "") for n in nodes]

  print(f"[{time.strftime('%H:%M:%S')}] Очистка ОЗУ от гигантских словарей (Экономия ~2-3 GB)")
  del ti_d, ab_d, ct_d; gc.collect()

  print(f"[{time.strftime('%H:%M:%S')}] SciBERT Encoding (FP16)")
  st = ST('pritamdeka/S-SciBERT-snli-multinli-stsb').to(d_dv)

  with torch.autocast(device_type=d_dv.type, dtype=torch.float16):
      e_ti = st.encode(ti_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)
      e_ab = st.encode(ab_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)
      e_ct = st.encode(ct_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)

  del st, ti_l, ab_l, ct_l; torch.cuda.empty_cache(); gc.collect()

  x_raw = torch.cat([e_ti, e_ab, e_ct], dim=1).half().cpu()
  del e_ti, e_ab, e_ct; torch.cuda.empty_cache()

  return x_raw

def train_ae(m, data, d_dv, ep=30, b_sz=2048, lr=1e-3):
  '''
  Обучает автоэнкодер для сжатия признаков.
  m (nn.Module): Класс автоэнкодера (модель TextAE).
  data (torch.Tensor): Массив векторов SciBERT размерности 2304.
  d_dv (torch.device): Вычислительное устройство для обучения.
  ep (int, по умолчанию 30): Количество эпох обучения.
  b_sz (int, по умолчанию 2048): Размер батча.
  lr (float, по умолчанию 1e-3): Шаг обучения (learning rate) оптимизатора AdamW
  '''

  m = m.to(d_dv); o = op.AdamW(m.parameters(), lr=lr, weight_decay=1e-5); crit = nn.MSELoss()
  n_samples = data.size(0)
  print(f"[{time.strftime('%H:%M:%S')}] Обучение Autoencoder ({ep} эпох, батч {b_sz})")

  for e in range(ep):
      m.train(); tl = 0
      idx = torch.randperm(n_samples)
      for i in range(0, n_samples, b_sz):
          b_idx = idx[i:i+b_sz]
          x = data[b_idx].to(d_dv, dtype=torch.float32)
          o.zero_grad()
          out = m(x); l = crit(out, x)
          l.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); o.step()
          tl += l.item() * x.size(0)
      print(f"[{time.strftime('%H:%M:%S')}] epoch {e+1} | MSE: {tl/n_samples:.4f}")

  m.eval()
  c_l = []
  with torch.no_grad():
      for i in range(0, n_samples, b_sz):
          x = data[i:i+b_sz].to(d_dv, dtype=torch.float32)
          c_l.append(m.compress(x).cpu())
  return torch.cat(c_l, dim=0)

def build_ae_embeddings(d_p, g, raw_p, d_dv):
  '''
  Генерирует сжатые автоэнкодером латентные представления размерности 256.
  d_p (str): Путь к обработанному pruned JSONL-файлу.
  g (nx.DiGraph): Очищенный стабильный граф.
  raw_p (str): Путь к оригинальному сырому JSONL-файлу (для подтягивания названий и аннотаций).
  d_dv (torch.device): Рабочее устройство GPU/CPU
  '''

  print(f"[{time.strftime('%H:%M:%S')}] Извлечение Title, Abstract, Concepts")
  ti_d = {}; ab_d = {}; ct_d = {}

  with open(raw_p, 'r', encoding='utf-8') as f:
      for ln in f:
          if not ln.strip(): continue
          x = json.loads(ln)
          if x['i'] in g.nodes():
              ti_d[x['i']] = x.get('ti', '') or ''
              ab_d[x['i']] = x.get('ab', '') or ''

  with open(d_p, 'r', encoding='utf-8') as f:
      for ln in f:
          if not ln.strip(): continue
          x = json.loads(ln)
          ct_d[x['i']] = x.get('ct_f', '') or ''

  nodes = list(g.nodes())
  ti_l = [ti_d.get(n, "") for n in nodes]; ab_l = [ab_d.get(n, "") for n in nodes]; ct_l = [ct_d.get(n, "") for n in nodes]
  y_l = [g.nodes[n].get('y', 0) for n in nodes]

  print(f"[{time.strftime('%H:%M:%S')}] SciBERT Encoding (FP16)")
  st = ST('pritamdeka/S-SciBERT-snli-multinli-stsb').to(d_dv)

  with torch.autocast(device_type=d_dv.type, dtype=torch.float16):
      e_ti = st.encode(ti_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)
      e_ab = st.encode(ab_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)
      e_ct = st.encode(ct_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)

  del st; torch.cuda.empty_cache(); gc.collect()

  x_raw = torch.cat([e_ti, e_ab, e_ct], dim=1).float()
  del e_ti, e_ab, e_ct; torch.cuda.empty_cache()

  ae = TextAE(i_d=2304, h_d=1024, b_d=256)
  x_cmp = train_ae(ae, x_raw, d_dv, ep=30)

  print(f"[{time.strftime('%H:%M:%S')}] Нормализация года")
  yr_t = torch.tensor(y_l).float().view(-1, 1)
  yr_t = (yr_t - yr_t.mean()) / (yr_t.std() + 1e-8)

  x_f_red = torch.cat([x_cmp, yr_t], dim=1)

  print(f"[{time.strftime('%H:%M:%S')}] Индексы графа")
  n_to_i = {n: i for i, n in enumerate(nodes)}
  e_idx = torch.tensor([[n_to_i[u], n_to_i[v]] for u, v in g.edges()], dtype=torch.long).t()

  return x_f_red, e_idx, n_to_i


### Топология

In [ ]:
def compute_fastrp_hetero(G_m, dim=128):
  '''
  Рассчитывает Fast Random Projection на гетерогенном графе соавторства.
  G_m (nx.Graph): Маскированный гетерогенный граф (содержит только Train ребра).
  dim (int, по умолчанию 128): Выходная размерность топологического эмбеддинга FastRP
  '''

  print(f"[{time.strftime('%H:%M:%S')}] Вычисление FastRP на гетерогенном графе")
  n_v = G_m.number_of_nodes()
  n_list = list(G_m.nodes())
  n_map = {n: i for i, n in enumerate(n_list)}

  edges = [(n_map[u], n_map[v]) for u, v in G_m.edges()]
  row = np.array([e[0] for e in edges]); col = np.array([e[1] for e in edges])
  data = np.ones(len(row))

  A = sp.coo_matrix((data, (row, col)), shape=(n_v, n_v)).tocsr()
  deg = np.array(A.sum(axis=1)).flatten(); deg[deg == 0] = 1.0
  D_inv = sp.diags(1.0 / deg); T = D_inv.dot(A)

  S = np.random.choice([-1, 0, 1], size=(n_v, dim), p=[0.1, 0.8, 0.1]).astype(np.float32)
  H = np.zeros((n_v, dim), dtype=np.float32); curr = S

  weights = [0.0, 1.0, 2.0, 4.0]
  for w in weights:
      if w > 0: H += w * curr
      curr = T.dot(curr)

  return {n_list[i]: H[i] for i in range(n_v)}

def aggregate_author_topology(p_l, a_dict, topo_dict, dim=128):
  '''
  Агрегирует FastRP-эмбеддинги соавторов для инициализации признаков новых статей.
  p_l (list): Список идентификаторов статей.
  a_dict (dict): Словарь соавторов для каждой статьи {id_статьи: {авторы}}.
  topo_dict (dict): Словарь рассчитанных FastRP эмбеддингов для вершин-авторов.
  dim (int, по умолчанию 128): Размерность результирующего топологического вектора
  '''

  print(f"[{time.strftime('%H:%M:%S')}] Агрегация топологии авторов")
  res = np.zeros((len(p_l), dim), dtype=np.float32)
  for i, p in enumerate(p_l):
      aus = a_dict.get(p, set())
      if not aus: continue
      vecs = [topo_dict[a] for a in aus if a in topo_dict]
      if vecs: res[i] = np.mean(vecs, axis=0)
  return torch.tensor(res, dtype=torch.float32)


### Эвристики

In [ ]:
def build_eval_matrices(tr_p, au_d, n_v, d_dv):
  '''
  Генерирует разреженные тензоры PyTorch для GPU-вычислений сетевых эвристик.
  tr_p (torch.Tensor): Массив ребер обучающей выборки.
  au_d (dict): Словарь авторов статей.
  n_v (int): Суммарное число вершин в графе.
  d_dv (torch.device)
  '''

  print(f"[{time.strftime('%H:%M:%S')}] Сборка разреженных матриц эвристик")
  u_tr = tr_p[0]; v_tr = tr_p[1]
  u_sym = torch.cat([u_tr, v_tr]); v_sym = torch.cat([v_tr, u_tr])
  v_sym_val = torch.ones(u_sym.size(0), dtype=torch.float32)
  A_sym = torch.sparse_coo_tensor(torch.stack([u_sym, v_sym]), v_sym_val, (n_v, n_v)).coalesce()
  A_sym.values().clamp_(max=1.0)

  o_d = torch.zeros(n_v).scatter_add_(0, u_tr, torch.ones_like(u_tr, dtype=torch.float32))
  i_d = torch.zeros(n_v).scatter_add_(0, v_tr, torch.ones_like(v_tr, dtype=torch.float32))
  t_d = o_d + i_d
  w_aa = 1.0 / (torch.log(t_d + 1e-8) + 1e-8); w_aa[t_d <= 1] = 0
  A_aa = torch.sparse_coo_tensor(torch.stack([u_sym, v_sym]), w_aa[v_sym], (n_v, n_v)).coalesce()

  t_a = sum(len(a) for a in au_d.values())
  r = np.zeros(t_a, dtype=np.int32); c = np.zeros(t_a, dtype=np.int32)
  a_m = {}; idx = 0
  for p, a_set in au_d.items():
      for a in a_set:
          if a not in a_m: a_m[a] = len(a_m)
          r[idx] = a_m[a]; c[idx] = p; idx += 1
  n_a = len(a_m)
  r_t = torch.from_numpy(r).long(); c_t = torch.from_numpy(c).long()
  M_ap = torch.sparse_coo_tensor(torch.stack([c_t, r_t]), torch.ones(t_a, dtype=torch.float32), (n_v, n_a)).coalesce()
  M_ap_T = torch.sparse_coo_tensor(torch.stack([r_t, c_t]), torch.ones(t_a, dtype=torch.float32), (n_a, n_v)).coalesce()
  a_deg = torch.zeros(n_v, dtype=torch.float32)
  un_n, cnts = torch.unique(c_t, return_counts=True)
  a_deg[un_n] = cnts.float()

  return A_sym.to(d_dv), A_aa.to(d_dv), M_ap.to(d_dv), M_ap_T.to(d_dv), a_deg.to(d_dv)

def build_eval_matrices_csr(train_edges, authors_dict, num_nodes):
  '''
  Строит разреженные Scipy CSR-матрицы для CPU-вычислений эвристик.
  train_edges (torch.Tensor): Обучающие ребра графа.
  authors_dict (dict): Словарь соавторов статей.
  num_nodes (int): Общее количество вершин
  '''

  u_nodes = train_edges[0].numpy(); v_nodes = train_edges[1].numpy()
  data_ones = np.ones(len(u_nodes) * 2, dtype=np.float32)
  row_all = np.concatenate([u_nodes, v_nodes]); col_all = np.concatenate([v_nodes, u_nodes])
  adj_csr = sp.csr_matrix((data_ones, (row_all, col_all)), shape=(num_nodes, num_nodes))
  adj_csr.data = np.clip(adj_csr.data, 0, 1)

  total_degrees = np.array(adj_csr.sum(axis=1)).flatten()
  weights_aa = np.zeros(num_nodes, dtype=np.float32); mask_aa = total_degrees > 1
  weights_aa[mask_aa] = 1.0 / np.log(total_degrees[mask_aa] + 1e-8)
  adamic_adar_csr = sp.csr_matrix((weights_aa[col_all], (row_all, col_all)), shape=(num_nodes, num_nodes))

  total_authors = sum(len(a) for a in authors_dict.values())
  row_auth = np.zeros(total_authors, dtype=np.int32); col_auth = np.zeros(total_authors, dtype=np.int32)
  auth_map = {}; idx = 0
  for paper, authors in authors_dict.items():
      for author in authors:
          if author not in auth_map: auth_map[author] = len(auth_map)
          row_auth[idx] = auth_map[author]; col_auth[idx] = paper; idx += 1

  num_authors = len(auth_map)
  author_csr = sp.csr_matrix((np.ones(total_authors, dtype=np.float32), (col_auth, row_auth)), shape=(num_nodes, num_authors))
  author_degrees = np.array(author_csr.sum(axis=1)).flatten()

  return adj_csr, adamic_adar_csr, author_csr, author_degrees

def get_batch_heuristics(b_u, n_v, y_a, a_sym, a_aa, m_ap, m_ap_T, a_deg, d):
  '''
  Быстро вычисляет эвристики для пачки узлов-запросов (Retrieval) на GPU.
  b_u (torch.Tensor): Вектор индексов узлов-источников (батч запросов).
  n_v (int): Общее число вершин.
  y_a (torch.Tensor): Массив годов издания всех статей.
  a_sym (torch.sparse.Tensor): Разреженная симметричная матрица смежности.
  a_aa (torch.sparse.Tensor): Разреженная матрица для расчета индекса Адамика-Адар.
  m_ap, m_ap_T (torch.sparse.Tensor): Матрицы проекции «статья-автор» и обратно.
  a_deg (torch.Tensor): Вектор степеней вершин.
  d (torch.device): Устройство вычислений
  '''

  B = b_u.size(0); dt = y_a[b_u].unsqueeze(1) - y_a.unsqueeze(0)
  ind = torch.zeros(B, n_v, device=d); ind[torch.arange(B), b_u] = 1.0
  M_u = torch.sparse.mm(m_ap_T, ind.t()).t()
  I = torch.sparse.mm(m_ap, M_u.t()).t()
  U = a_deg[b_u].unsqueeze(1) + a_deg.unsqueeze(0) - I
  au_o = torch.where(U > 0, I / U, torch.zeros_like(I))
  A_u = torch.sparse.mm(a_sym, ind.t()).t()
  cn = torch.sparse.mm(a_sym, A_u.t()).t()
  aa = torch.sparse.mm(a_aa, A_u.t()).t()
  cn = torch.log(cn + 1.0); aa = torch.log(aa + 1.0)
  return torch.stack([dt, au_o, cn, aa], dim=-1)

def get_pair_heuristics(u, v, n_v, y_a, a_sym, a_aa, m_ap, m_ap_T, a_deg, d):
  '''
  Вычисляет попарные эвристики для конкретных пар ребер (u, v).
  u (torch.Tensor): Тензор индексов начальных вершин ребер.
  v (torch.Tensor): Тензор индексов конечных вершин ребер.
  (остальные параметры идентичны функции get_batch_heuristics)
  '''
  dt = y_a[u] - y_a[v]
  ind_u = torch.zeros(u.size(0), n_v, device=d); ind_u[torch.arange(u.size(0)), u] = 1.0
  ind_v = torch.zeros(v.size(0), n_v, device=d); ind_v[torch.arange(v.size(0)), v] = 1.0
  M_u = torch.sparse.mm(m_ap_T, ind_u.t()).t()
  M_v = torch.sparse.mm(m_ap_T, ind_v.t()).t()
  I = (M_u * M_v).sum(dim=1)
  U = a_deg[u] + a_deg[v] - I
  au_o = torch.where(U > 0, I / U, torch.zeros_like(I))
  A_u = torch.sparse.mm(a_sym, ind_u.t()).t(); A_v = torch.sparse.mm(a_sym, ind_v.t()).t()
  cn = (A_u * A_v).sum(dim=1)
  AA_u = torch.sparse.mm(a_aa, ind_u.t()).t()
  aa = (AA_u * A_v).sum(dim=1)
  cn = torch.log(cn + 1.0); aa = torch.log(aa + 1.0)
  return torch.stack([dt, au_o, cn, aa], dim=-1)


## Формирование обучающей, тестовой и валидационной выборок

In [ ]:
def build_hetero_graph_and_split(d_p, k_val=3, test_s=0.1, val_s=0.05):
  '''
  Преобразует исходные данные в гетерогенный граф соавторства-цитирования, выполняет k-core фильтрацию и делит узлы на случайные маскируемые выборки.
  d_p (str): Путь к файлу очищенных данных JSONL.
  k_val (int, по умолчанию 3): Параметр k-core фильтрации для двудольного подграфа "автор-статья".
  test_s (float, по умолчанию 0.1): Доля тестовой выборки (Test Split).
  val_s (float, по умолчанию 0.05): Доля валидационной выборки (Validation Split).
  '''
  p_l = []; y_dict = {}; c_dict = {}; a_dict = {}; cites_raw = []
  with open(d_p, 'r', encoding='utf-8') as f:
      for ln in f:
          if not ln.strip(): continue
          x = json.loads(ln); p_id = x.get('i')
          if p_id is None: continue
          p_l.append(p_id); y_dict[p_id] = x.get('y', 0)
          ct_raw = x.get('ct', ''); c_dict[p_id] = set(ct_raw.lower().split()) if ct_raw else set()
          au_raw = x.get('au', []); a_dict[p_id] = set(a for a in au_raw if a is not None)
          for v in x.get('rf', []):
              if v is not None: cites_raw.append((p_id, v))
  bg = nx.Graph()
  for p in p_l:
      for a in a_dict[p]: bg.add_edge(p, a)
  pg = nx.k_core(bg, k=k_val); v_n = set(pg.nodes())
  p_l = [p for p in p_l if p in v_n]
  y_dict = {p: y_dict[p] for p in p_l}; c_dict = {p: c_dict[p] for p in p_l}
  a_dict = {p: set(a for a in a_dict[p] if a in v_n) for p in p_l}
  cites_raw = [(u, v) for u, v in cites_raw if u in v_n and v in v_n]
  G = nx.DiGraph()
  for p in p_l:
      G.add_node(p, type='paper', y=y_dict[p])
      for a in a_dict[p]:
          G.add_node(a, type='author')
          G.add_edge(a, p, type='writes'); G.add_edge(p, a, type='writes')
  for u, v in cites_raw: G.add_edge(u, v, type='cites')
  G.remove_edges_from(list(nx.selfloop_edges(G)))
  random.seed(42); random.shuffle(p_l)
  n_te = int(len(p_l) * test_s); n_vl = int(len(p_l) * val_s)
  te_n = set(p_l[:n_te]); vl_n = set(p_l[n_te:n_te+n_vl]); tr_n = set(p_l[n_te+n_vl:])
  G_m = G.copy(); e_rm = []
  for u, v, d in G_m.edges(data=True):
      if d.get('type') == 'cites':
          if u in te_n or u in vl_n or v in te_n or v in vl_n: e_rm.append((u, v))
  G_m.remove_edges_from(e_rm)
  in_deg = {n: 0 for n in p_l}
  for u, v, d in G_m.edges(data=True):
      if d.get('type') == 'cites' and v in in_deg: in_deg[v] += 1
  return G, G_m, list(tr_n), list(vl_n), list(te_n), y_dict, c_dict, a_dict, in_deg

def generate_negative_edges(p_e, n_v, y_a):
  '''
  Генерирует случайные отрицательные пары под хронологическим фильтром.
  p_e (torch.Tensor): Тензор истинных (положительных) ребер.
  n_v (int): Общее количество вершин в графе.
  y_a (torch.Tensor): Года публикаций всех статей
  '''
  print(f"[{time.strftime('%H:%M:%S')}] Ген. негативных: {p_e.size(0)}")
  u = p_e[:, 0]; v_n = torch.randint(0, n_v, (p_e.size(0),), device=p_e.device)
  inv = (y_a[v_n] > y_a[u]) | (v_n == u)
  while inv.any():
      v_n[inv] = torch.randint(0, n_v, (inv.sum().item(),), device=p_e.device)
      inv = (y_a[v_n] > y_a[u]) | (v_n == u)
  return torch.stack([u, v_n], dim=1)

def generate_causal_negatives(p_l, G, y_d, p_i, n_r=1):
  '''
  Генерирует сбалансированные негативные примеры для двудольной структуры соавторства.
  p_l (list): Список вершин (статей), для которых подбираются негативы.
  G (nx.DiGraph): Глобальный граф соавторства и цитирований.
  y_d (dict): Словарь лет издания статей {id: year}.
  p_i (dict): Индексная мапа {id_строки: int_индекс}.
  n_r (int, по умолчанию 1): Коэффициент негативного сэмплирования (сколько негативных ребер генерировать на одно позитивное)
  '''

  print(f"[{time.strftime('%H:%M:%S')}] gen neg (integer indices safe)")
  p_e = []; n_e = []; a_p = list(y_d.keys())

  for u in p_l:
      c = [v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites']
      for v in c: p_e.append((p_i[u], p_i[v]))

      cnt = 0; atm = 0; max_atm = len(c) * n_r * 200
      while cnt < len(c) * n_r:
          atm += 1
          if atm > max_atm: raise RuntimeError(f"Edge case: Невозможно найти достаточно негативов для {u}")
          v_n = random.choice(a_p)
          if u != v_n and v_n not in c and y_d[v_n] <= y_d[u]:
              n_e.append((p_i[u], p_i[v_n])); cnt += 1

  e_l = p_e + n_e; y_l = [1.0] * len(p_e) + [0.0] * len(n_e)
  idx = np.random.permutation(len(e_l))
  return np.array(e_l, dtype=np.int32)[idx], np.array(y_l, dtype=np.float32)[idx]

def g_n_e_hard(p_e, x_f, y_a, n_v, d_dv, k=50):
  '''
  Сэмплирует семантически сложные негативные ребра на основе косинусного сходства SciBERT.
  p_e (torch.Tensor): Тензор истинных ребер.
  x_f (torch.Tensor): Матрица латентных векторов статей (размерность 256).
  y_a (torch.Tensor): Года публикаций.
  n_v (int): Общее количество вершин.
  d_dv (torch.device): Вычислительное устройство.
  k (int, по умолчанию 50): Размер пула случайных кандидатов, из которого выбирается один «наиболее семантически близкий» жесткий негатив
  '''

  print(f"[{time.strftime('%H:%M:%S')}] Семплирование Hard Negatives ({p_e.size(1)} ребер)")
  u = p_e[0].to(d_dv); v_pos = p_e[1].to(d_dv)
  v_n = torch.zeros_like(u)
  y_dev = y_a.to(d_dv); f_dev = x_f[:, :256].to(d_dv)

  b_sz = 2048
  for i in range(0, u.size(0), b_sz):
      b_u = u[i:i+b_sz]; b_v = v_pos[i:i+b_sz]; B = b_u.size(0)
      c = torch.randint(0, n_v, (B, k), device=d_dv)
      inv = (y_dev[c] > y_dev[b_u].unsqueeze(1)) | (c == b_u.unsqueeze(1)) | (c == b_v.unsqueeze(1))
      s = torch.cosine_similarity(f_dev[b_u].unsqueeze(1), f_dev[c], dim=2)
      s[inv] = -2.0

      b_idx = torch.argmax(s, dim=1); b_v_n = c[torch.arange(B), b_idx]

      f_inv = s[torch.arange(B), b_idx] == -2.0
      if f_inv.any():
          f_u = b_u[f_inv]; r = torch.randint(0, n_v, (f_u.size(0),), device=d_dv)
          m = (y_dev[r] > y_dev[f_u]) | (r == f_u) | (r == b_v[f_inv])
          sg = 0
          while m.any() and sg < 50:
              r[m] = torch.randint(0, n_v, (m.sum().item(),), device=d_dv)
              m = (y_dev[r] > y_dev[f_u]) | (r == f_u) | (r == b_v[f_inv])
              sg += 1
          b_v_n[f_inv] = r
      v_n[i:i+b_sz] = b_v_n
  return torch.stack([u.cpu(), v_n.cpu()], dim=0)


## Создание графа и быстрые проверки качества

In [ ]:
def run_stage2(p_s2):
  k_rw('w', name4, {'stage4_tensors.pt': d_s4}, t='Causal Features V5', p=p_s4)

    print(f"[{time.strftime('%H:%M:%S')}] СТАРТ ЭТАПА 2: Генерация структур")
    ld1 = k_rw('r', name1, p='./stage1_raw'); raw_p = ld1['d.jsonl']; del ld1; gc.collect()
    G, G_m, tr_n, vl_n, te_n, y_dict, c_dict, a_dict, in_deg = build_hetero_graph_and_split(raw_p, k_val=3)
    d_s2 = {'G': G, 'G_m': G_m, 'tr_n': tr_n, 'vl_n': vl_n, 'te_n': te_n, 'y_dict': y_dict, 'c_dict': c_dict, 'a_dict': a_dict, 'in_deg': in_deg}
    k_rw('w', name2, {'stage2_data.pkl': d_s2}, t='Hetero Core Graph Split V5', p=p_s2)

def sanity_check_training(p_s2):
    ld2 = k_rw('r', name2, p=p_s2); d_s2 = ld2['stage2_data.pkl']; G = d_s2['G']
    tr_set = set(d_s2['tr_n']); vl_set = set(d_s2['vl_n']); te_set = set(d_s2['te_n']); tr_e = vl_e = te_e = 0
    for u, v, d in G.edges(data=True):
        if d.get('type') == 'cites':
            if u in tr_set: tr_e += 1
            elif u in vl_set: vl_e += 1
            elif u in te_set: te_e += 1
    df = pl.DataFrame([
        {"Выборка": "Train", "Вершин": len(tr_set), "Ребер цитирования": tr_e},
        {"Выборка": "Val", "Вершин": len(vl_set), "Ребер цитирования": vl_e},
        {"Выборка": "Test", "Вершин": len(te_set), "Ребер цитирования": te_e}
    ])
    print(f"[{time.strftime('%H:%M:%S')}] SANITY CHECK ОБУЧЕНИЕ:\n{df}"); p_sample = list(tr_set)[0] if tr_set else None
    if p_sample: print(f"Пример вершины Train ({p_sample}): {G.nodes[p_sample]}")
    del ld2, d_s2, G, tr_set, vl_set, te_set, df; gc.collect()

def sanity_check_stage2(p_s2):
    ld = k_rw('r', name2, p=p_s2); d_s2 = ld['stage2_data.pkl']
    G = d_s2['G']; G_m = d_s2['G_m']
    tr_n = set(d_s2['tr_n']); vl_n = set(d_s2['vl_n']); te_n = set(d_s2['te_n'])

    tr_e = sum(1 for u, v, d in G.edges(data=True) if d.get('type') == 'cites' and u in tr_n)
    vl_e = sum(1 for u, v, d in G.edges(data=True) if d.get('type') == 'cites' and u in vl_n)
    te_e = sum(1 for u, v, d in G.edges(data=True) if d.get('type') == 'cites' and u in te_n)

    df_s = pl.DataFrame([
        {"Выборка": "Train", "Узлы": len(tr_n), "Исходящие цитирования": tr_e},
        {"Выборка": "Val", "Узлы": len(vl_n), "Исходящие цитирования": vl_e},
        {"Выборка": "Test", "Узлы": len(te_n), "Исходящие цитирования": te_e}
    ])

    e_cites = sum(1 for u, v, d in G.edges(data=True) if d.get('type') == 'cites')
    e_writes = sum(1 for u, v, d in G.edges(data=True) if d.get('type') == 'writes') // 2
    e_cites_m = sum(1 for u, v, d in G_m.edges(data=True) if d.get('type') == 'cites')
    df_g = pl.DataFrame([
        {"Граф": "Ядро Исходное", "Узлы": G.number_of_nodes(), "Цитирования": e_cites, "Авторства": e_writes},
        {"Граф": "Ядро Маскированное", "Узлы": G_m.number_of_nodes(), "Цитирования": e_cites_m, "Авторства": e_writes}
    ])

    print(f"[{time.strftime('%H:%M:%S')}] SANITY CHECK ЭТАП 2: Выборки\n{df_s}")
    print(f"[{time.strftime('%H:%M:%S')}] SANITY CHECK ЭТАП 2: Статистика графа\n{df_g}")

    p_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'paper']
    a_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'author']
    print(f"[{time.strftime('%H:%M:%S')}] SANITY CHECK ЭТАП 2: Примеры свойств вершин")
    if p_nodes: print(f"Статья (Paper): {p_nodes[0]} -> {G.nodes[p_nodes[0]]}")
    if a_nodes: print(f"Автор (Author): {a_nodes[0]} -> {G.nodes[a_nodes[0]]}")


def run_stage3_5(p_s35):
  '''
  Выполняет вычисление топологических эмбеддингов FastRP для авторов, агрегирует их для статей (для холодного старта) и строит объект HeteroData для PyG.
  p_s35 (str): Локальный путь для сохранения артефактов Этапа 3.5.
  '''
  print(f"[{time.strftime('%H:%M:%S')}] СТАРТ ЭТАПА 3.5: Топологические эмбеддинги")

  p_s2 = get_p(name2)
  ld2 = k_rw('r', name2, p=p_s2)
  d_s2 = ld2['stage2_data.pkl']
  G = d_s2['G']
  G_m = d_s2['G_m']
  y_dict = d_s2['y_dict']
  a_dict = d_s2['a_dict']
  del ld2; gc.collect()

  p_s3b = get_p(name3b)
  ld3b = k_rw('r', name3b, p=p_s3b)
  x_text = ld3b['stage3b_tensors.pt']['x_f']
  p_list_ordered = ld3b['stage3b_tensors.pt']['p_list_ordered']
  del ld3b; gc.collect()

  topo_dict = compute_fastrp_hetero(G_m, dim=128)

  z_topo = aggregate_author_topology(p_list_ordered, a_dict, topo_dict, dim=128)

  hd = HeteroData()

  hd['paper'].x = x_text.float()

  authors_list = sorted(list(set(a for p in p_list_ordered for a in a_dict.get(p, set()))))
  a_to_idx = {a: i for i, a in enumerate(authors_list)}

  a_feats = np.zeros((len(authors_list), 128), dtype=np.float32)
  for i, a in enumerate(authors_list):
      if a in topo_dict:
          a_feats[i] = topo_dict[a]
  hd['author'].x = torch.tensor(a_feats, dtype=torch.float)

  p_to_i = {p: i for i, p in enumerate(p_list_ordered)}

  w_src, w_dst = [], []
  for p in p_list_ordered:
      p_idx = p_to_i[p]
      for a in a_dict.get(p, set()):
          if a in a_to_idx:
              w_src.append(a_to_idx[a])
              w_dst.append(p_idx)

  hd['author', 'writes', 'paper'].edge_index = torch.tensor([w_src, w_dst], dtype=torch.long)
  hd['paper', 'rev_writes', 'author'].edge_index = torch.tensor([w_dst, w_src], dtype=torch.long)

  c_src, c_dst = [], []
  for u, v, d_edge in G_m.edges(data=True):
      if d_edge.get('type') == 'cites' and u in p_to_i and v in p_to_i:
          c_src.append(p_to_i[u])
          c_dst.append(p_to_i[v])

  hd['paper', 'cites', 'paper'].edge_index = torch.tensor([c_src, c_dst], dtype=torch.long)

  d_s35 = {
      'z_topo': z_topo,
      'hd': hd,
      'p_list_ordered': p_list_ordered,
      'p_to_i': p_to_i
  }
  k_rw('w', name3_5, {'stage3_5_data.pt': d_s35}, t='Topology Embs PyG V5', p=p_s35)
  print(f"[{time.strftime('%H:%M:%S')}] ЭТАП 3.5 УСПЕШНО ЗАВЕРШЕН")

def sanity_check_stage35(p_s35):
    ld = k_rw('r', name3_5, p=p_s35); d_s35 = ld['stage3_5_data.pt']
    df = pl.DataFrame([
        {"Компонент": "Paper Matrix (z_topo)", "Shape": str(list(d_s35['z_topo'].shape))},
        {"Компонент": "HeteroData Nodes Count", "Shape": str(d_s35['hd'].num_nodes)}
    ])
    print(f"[{time.strftime('%H:%M:%S')}] SANITY CHECK ЭТАП 3.5: Размеры структур\n{df}")
    print(f"[{time.strftime('%H:%M:%S')}] SANITY CHECK ЭТАП 3.5: Фрагмент z_topo [0, :5]")
    print(d_s35['z_topo'][0, :5].tolist())


def run_stage4(p_s4):
  '''
  Выполняет финальную сборку плоских признаковых пространств (эвристик, авторов, текстовых признаков) для GBDT (Этап 4).
  p_s4 (str): Локальный путь для сохранения артефактов Этапа 4.
  '''

  p_s2 = get_p(name2); ld2 = k_rw('r', name2, p=p_s2); d_s2 = ld2['stage2_data.pkl']
  G = d_s2['G']; G_m = d_s2['G_m']; tr_n = d_s2['tr_n']; vl_n = d_s2['vl_n']
  y_d = d_s2['y_dict']; c_d = d_s2['c_dict']; a_d = d_s2['a_dict']
  del ld2, d_s2; gc.collect()

  p_s3b = get_p(name3b); ld3b = k_rw('r', name3b, p=p_s3b)
  x_t = ld3b['stage3b_tensors.pt']['x_f'][:, :256].clone()
  del ld3b; gc.collect()

  p_s35 = get_p(name3_5); ld35 = k_rw('r', name3_5, p=p_s35)
  z_t = ld35['stage3_5_data.pt']['z_topo']
  p_l = ld35['stage3_5_data.pt']['p_list_ordered']
  p_i = {p: i for i, p in enumerate(p_l)}
  del ld35; gc.collect()

  in_d = {n: 0 for n in p_l}
  for u, v, d in G_m.edges(data=True):
      if d.get('type') == 'cites' and v in in_d: in_d[v] += 1

  tr_e, tr_y = generate_causal_negatives(tr_n, G, y_d, p_i, n_r=1)
  vl_e, vl_y = generate_causal_negatives(vl_n, G, y_d, p_i, n_r=1)
  del G, G_m, tr_n, vl_n; gc.collect()

  i_to_p = {v: k for k, v in p_i.items()}
  tr_e_str = np.array([[i_to_p[e[0]], i_to_p[e[1]]] for e in tr_e])
  vl_e_str = np.array([[i_to_p[e[0]], i_to_p[e[1]]] for e in vl_e])

  tr_X = extract_causal_features(tr_e_str, y_d, c_d, a_d, in_d, z_t, x_t, p_i, hub=True)
  vl_X = extract_causal_features(vl_e_str, y_d, c_d, a_d, in_d, z_t, x_t, p_i, hub=True)
  del tr_e_str, vl_e_str; gc.collect()

  d_s4 = {
      'tr_X': torch.from_numpy(tr_X), 'tr_y': torch.from_numpy(tr_y), 'tr_g': torch.from_numpy(tr_e[:, 0]),
      'vl_X': torch.from_numpy(vl_X), 'vl_y': torch.from_numpy(vl_y), 'vl_g': torch.from_numpy(vl_e[:, 0]),
      'in_deg_dict': in_d, 'p_to_i': p_i, 'p_list_ordered': p_l
  }

def sanity_check_stage4(p_s4):
    ld = k_rw('r', name4, p=p_s4); d_s4 = ld['stage4_tensors.pt']
    df_c = pl.DataFrame([
        {"Данные": "Train X", "Форма": str(list(d_s4['tr_X'].shape))},
        {"Данные": "Val X", "Форма": str(list(d_s4['vl_X'].shape))}
    ])
    print(df_c)


## Архитектуры, лоссы, предикторы

In [ ]:
class ASL(nn.Module):
  '''
  ASL (Asymmetric Loss)
  Функция потерь для компенсации экстремального дисбаланса классов (когда негативных примеров гораздо больше).
  Декоплирует параметры затухания для положительных и отрицательных примеров, жестче штрафуя ложные срабатывания.
  '''

  def __init__(self, gamma_pos=1.0, gamma_neg=2.0):
      super().__init__()
      self.gamma_pos = gamma_pos
      self.gamma_neg = gamma_neg

  def forward(self, p, y):
      p = torch.clamp(p, 1e-7, 1. - 1e-7)
      lp = -y * (1 - p)**self.gamma_pos * torch.log(p)
      ln = -(1 - y) * p**self.gamma_neg * torch.log(1 - p)
      return torch.mean(lp + ln)

class FL(nn.Module):
  '''
  FL (Focal Loss)
  Модификация классической кросс-энтропии.
  Динамически снижает вклад легко классифицируемых примеров в суммарную ошибку, заставляя модель фокусироваться на сложных случаях.
  '''

  def __init__(self, alpha=1, gamma=2):
      super().__init__()
      self.a = alpha
      self.g = gamma

  def forward(self, p, t):
      bce = nn.functional.binary_cross_entropy(p, t, reduction='none')
      p_t = p * t + (1 - p) * (1 - t)
      return (self.a * (1 - p_t) ** self.g * bce).mean()

class GAT(nn.Module):
  '''
  GAT (Graph Attention Network)
  Слой графового внимания.
  Рассчитывает веса связей между соседними вершинами на основе их признаков. Оптимизирован по памяти с использованием разреженных матриц PyTorch.
  '''

  def __init__(self, i_d, o_d):
      super().__init__()
      self.w = nn.Linear(i_d, o_d)
      self.a_l = nn.Linear(o_d, 1, bias=False)
      self.a_r = nn.Linear(o_d, 1, bias=False)
      self.a_bias = nn.Parameter(torch.zeros(1))
      self.l = nn.LeakyReLU(0.01)

  def forward(self, h, e):
      z = self.w(h)
      if e.size(0) == 0: return z
      u = torch.cat([e[:, 0], e[:, 1]])
      v = torch.cat([e[:, 1], e[:, 0]])
      z_l = self.a_l(z)
      z_r = self.a_r(z)
      a_e = self.l(z_l[u] + z_r[v] + self.a_bias).squeeze()
      e_w = torch.exp(a_e - a_e.max())
      s = torch.zeros(h.size(0), dtype=z.dtype, device=z.device)
      s.scatter_add_(0, u, e_w)
      s_u = s[u] + 1e-9
      a_n = e_w / s_u
      N = h.size(0)
      idx = torch.stack([u, v])
      A = torch.sparse_coo_tensor(idx, a_n, (N, N)).coalesce()
      r = torch.sparse.mm(A, z)
      return r + z

class LightGCN(nn.Module):
  '''
  Упрощенная версия графовой сверточной сети для рекомендательных систем.
  Не использует нелинейные активации и обучаемые матрицы весов при свертке, выполняя только линейное сглаживание признаков по ребрам.
  '''

  def __init__(self, i_d, o_d):
    super().__init__()
    self.w = nn.Linear(i_d, o_d)
    self.A = None; self.e_c = -1

  def forward(self, h, e):
      z = self.w(h)
      if e.size(1) == 0: return z
      if self.training or self.A is None or self.e_c != e.size(1):
          N = h.size(0); u = torch.cat([e[0], e[1]]); v = torch.cat([e[1], e[0]])
          s_l = torch.arange(N, device=h.device); u = torch.cat([u, s_l]); v = torch.cat([v, s_l])
          deg = torch.zeros(N, device=h.device).scatter_add_(0, u, torch.ones_like(u, dtype=torch.float))
          d_inv = deg.pow(-0.5); d_inv.masked_fill_(d_inv == float('inf'), 0); w = d_inv[u] * d_inv[v]
          self.A = torch.sparse_coo_tensor(torch.stack([u, v]), w, (N, N)).coalesce(); self.e_c = e.size(1)
      return torch.sparse.mm(self.A, z)

class SGC(nn.Module):
    '''
    SGC (Simplifying Graph Convolutional Networks)
    Модель, которая преобразует глубокую свертку в один шаг за счет предварительного вычисления K-степени нормализованной матрицы смежности графа.
    '''

    def __init__(self, i_d, o_d, k=2):
      super().__init__()
      self.w = nn.Linear(i_d, o_d)
      self.k = k; self.A = None
      self.e_c = -1

    def forward(self, h, e):
        if e.size(1) == 0: return self.w(h)
        if self.training or self.A is None or self.e_c != e.size(1):
            N = h.size(0); u = torch.cat([e[0], e[1], torch.arange(N, device=h.device)]); v = torch.cat([e[1], e[0], torch.arange(N, device=h.device)])
            deg = torch.zeros(N, device=h.device).scatter_add_(0, u, torch.ones_like(u, dtype=torch.float))
            d_inv = deg.pow(-0.5); d_inv.masked_fill_(d_inv == float('inf'), 0); w = d_inv[u] * d_inv[v]
            self.A = torch.sparse_coo_tensor(torch.stack([u, v]), w, (N, N)).coalesce(); self.e_c = e.size(1)
        x = h
        for _ in range(self.k): x = torch.sparse.mm(self.A, x)
        return self.w(x)

class SAGE(nn.Module):
  '''
  SAGE (GraphSAGE)
  Графовый слой, выполняющий конкатенацию собственных признаков вершины и агрегированных представлений ее локальной окрестности
  '''

  def __init__(self, i_d, o_d):
    super().__init__()
    self.w_s = nn.Linear(i_d, o_d)
    self.w_n = nn.Linear(i_d, o_d)
    self.A = None
    self.e_c = -1

  def forward(self, h, e):
      if e.size(1) == 0: return self.w_s(h)
      if self.training or self.A is None or self.e_c != e.size(1):
          N = h.size(0); u = torch.cat([e[0], e[1]]); v = torch.cat([e[1], e[0]])
          deg = torch.zeros(N, device=h.device).scatter_add_(0, u, torch.ones_like(u, dtype=torch.float))
          d_inv = deg.pow(-1.0); d_inv.masked_fill_(d_inv == float('inf'), 0); w = d_inv[u]
          self.A = torch.sparse_coo_tensor(torch.stack([u, v]), w, (N, N)).coalesce(); self.e_c = e.size(1)
      neigh_h = torch.sparse.mm(self.A, h)
      return torch.relu(self.w_s(h) + self.w_n(neigh_h))

class JKNet(nn.Module):
  '''
  JKNet (Jumping Knowledge Network)
  Архитектура, агрегирующая представления вершин с разных слоев (разных глубин окрестности) с помощью max-pooling, сохраняя локальные и глобальные свойства.
  '''

  def __init__(self, i_d, o_d, k=3):
      super().__init__()
      self.w = nn.Linear(i_d, o_d)
      self.k = k
      self.A = None
      self.e_c = -1

  def forward(self, h, e):
      z = self.w(h)
      if e.size(0) == 0: return z
      if self.A is None or self.e_c != e.size(0):
          N = h.size(0); u = torch.cat([e[:,0], e[:,1], torch.arange(N, device=h.device)])
          v = torch.cat([e[:,1], e[:,0], torch.arange(N, device=h.device)])
          deg = torch.zeros(N, device=h.device).scatter_add_(0, u, torch.ones_like(u, dtype=torch.float))
          d_inv = deg.pow(-0.5); d_inv.masked_fill_(d_inv == float('inf'), 0); w = d_inv[u] * d_inv[v]
          self.A = torch.sparse_coo_tensor(torch.stack([u, v]), w, (N, N)).coalesce(); self.e_c = e.size(0)
      l_o = [z]; x = z
      for _ in range(self.k): x = torch.sparse.mm(self.A, x); l_o.append(x)
      return torch.stack(l_o, dim=0).max(dim=0)[0]

class DirLightGCN(nn.Module):
  '''
  Модификация LightGCN для ориентированных сетей.
  Раздельно обрабатывает входящие и исходящие ребра цитирования
  '''

  def __init__(self, i_d, o_d):
    super().__init__()
    self.w = nn.Linear(i_d, o_d)
    self.A_in = None; self.A_out = None
    self.e_c = -1

  def forward(self, h, e):
      z = self.w(h)
      if e.size(1) == 0: return z
      if self.training or self.A_in is None or self.e_c != e.size(1):
          N = h.size(0); u = e[0]; v = e[1]
          d_out = torch.zeros(N, device=h.device).scatter_add_(0, u, torch.ones_like(u, dtype=torch.float))
          d_out_inv = d_out.pow(-1); d_out_inv.masked_fill_(d_out_inv == float('inf'), 0)
          self.A_out = torch.sparse_coo_tensor(torch.stack([u, v]), d_out_inv[u], (N, N)).coalesce()
          d_in = torch.zeros(N, device=h.device).scatter_add_(0, v, torch.ones_like(v, dtype=torch.float))
          d_in_inv = d_in.pow(-1); d_in_inv.masked_fill_(d_in_inv == float('inf'), 0)
          self.A_in = torch.sparse_coo_tensor(torch.stack([v, u]), d_in_inv[v], (N, N)).coalesce(); self.e_c = e.size(1)
      z_out = torch.sparse.mm(self.A_out, z); z_in = torch.sparse.mm(self.A_in, z)
      return z + z_out + z_in

class NeoGNN_Encoder(nn.Module):
  '''
  Гибридная архитектура, которая параллельно обрабатывает семантическую информацию (через LightGCN) и структурную схожесть графа смежности (через обучаемые эмбеддинги вершин), объединяя их на выходном слое.
  '''

  def __init__(self, n_v, i_d, o_d):
      super().__init__()
      self.feat_gcn = LightGCN(i_d, o_d // 2)
      self.struct_emb = nn.Embedding(n_v, o_d // 2)
      self.struct_gcn = LightGCN(o_d // 2, o_d // 2)

  def forward(self, h, v, e):
      z_f = self.feat_gcn(h, e)
      valid_mask = (v >= 0) & (v < self.struct_emb.num_embeddings)
      v_safe = torch.where(valid_mask, v, torch.zeros_like(v))
      s_e = self.struct_emb(v_safe)
      s_e = torch.where(valid_mask.unsqueeze(1), s_e, torch.zeros_like(s_e))
      z_s = self.struct_gcn(s_e, e)
      return torch.cat([z_f, z_s], dim=1)

class Standard_Predictor(nn.Module):
  '''Полносвязный классификатор ребер. Принимает эмбеддинги источника и цели, рассчитывает их произведения (Hadamard), абсолютную разность, конкатенирует с вектором эвристик и выдает вероятность связи через MLP'''

  def __init__(self, c_d, n_a, h_d=256):
      super().__init__()
      c_i = (c_d * 4) + n_a
      self.c = nn.Sequential(
          nn.Linear(c_i, h_d), nn.LayerNorm(h_d), nn.ReLU(), nn.Dropout(0.3),
          nn.Linear(h_d, h_d // 2), nn.LayerNorm(h_d // 2), nn.ReLU(), nn.Dropout(0.2),
          nn.Linear(h_d // 2, 1), nn.Sigmoid()
      )

  def forward(self, z_u, z_v, b_a):
      f = torch.cat([z_u, z_v, z_u * z_v, z_u - z_v], dim=1)
      return self.c(torch.cat([f, b_a], dim=1)).squeeze()

class BUDDY_Predictor(nn.Module):
  '''
  Late Fusion
  Семантические векторы и вектор топологических эвристик обрабатываются независимыми полносвязными блоками, результаты которых объединяются перед финальной классификацией
  '''

  def __init__(self, c_d, n_a, h_d=256):
      super().__init__()
      self.sem_mlp = nn.Sequential(nn.Linear(c_d * 2, h_d), nn.LayerNorm(h_d), nn.ReLU(), nn.Dropout(0.3), nn.Linear(h_d, h_d//2))
      self.str_mlp = nn.Sequential(nn.Linear(n_a, 32), nn.LayerNorm(32), nn.ReLU(), nn.Linear(32, 16))
      self.final = nn.Sequential(nn.Linear(h_d//2 + 16, 64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1), nn.Sigmoid())

  def forward(self, z_u, z_v, b_a):
      sem = self.sem_mlp(torch.cat([z_u, z_v], dim=1))
      st = self.str_mlp(b_a)
      return self.final(torch.cat([sem, st], dim=1)).squeeze()

class NCN_Predictor(nn.Module):
  '''
  Проекционная голова, адаптированная под архитектуру Neighborhood Common Neighbors.
  Опирается на попарное произведение векторов и топологические фичи.
  '''

  def __init__(self, c_d, n_a, h_d=256):
      super().__init__()
      self.mlp = nn.Sequential(
          nn.Linear(c_d + n_a, h_d), nn.LayerNorm(h_d), nn.ReLU(), nn.Dropout(0.3),
          nn.Linear(h_d, h_d//2), nn.LayerNorm(h_d//2), nn.ReLU(), nn.Dropout(0.2),
          nn.Linear(h_d//2, 1), nn.Sigmoid()
      )

  def forward(self, z_u, z_v, b_a):
      inter = z_u * z_v
      return self.mlp(torch.cat([inter, b_a], dim=1)).squeeze()

class GATv2(nn.Module):
  '''
  Улучшенная версия слоя GAT, устраняющая проблему статического внимания (в оригинальном GAT порядок внимания не зависел от центрального узла)
  '''

  def __init__(self, i_d, o_d):
      super().__init__()
      self.w_l = nn.Linear(i_d, o_d, bias=False)
      self.w_r = nn.Linear(i_d, o_d, bias=False)
      self.a = nn.Linear(o_d, 1, bias=False); self.l = nn.LeakyReLU(0.2)

  def forward(self, h, e):
      if e.size(0) == 0: return self.w_l(h)
      N = h.size(0); u = torch.cat([e[:,0], e[:,1], torch.arange(N, device=h.device)])
      v = torch.cat([e[:,1], e[:,0], torch.arange(N, device=h.device)])
      z_l = self.w_l(h); z_r = self.w_r(h); c = z_l[u] + z_r[v]; a_e = self.a(self.l(c)).squeeze()
      e_w = torch.exp(a_e - a_e.max()); s = torch.zeros(N, dtype=z_l.dtype, device=z_l.device)
      s.scatter_add_(0, u, e_w); a_n = e_w / (s[u] + 1e-9)
      A = torch.sparse_coo_tensor(torch.stack([u, v]), a_n, (N, N)).coalesce()
      return torch.sparse.mm(A, z_r) + z_l

class MarginLoss(nn.Module):
  '''
  Ранжирующий лосс.
  Обучает модель разносить оценки истинных и ложных связей на заданную величину зазора (margin)
  '''

  def __init__(self, margin=0.2):
      super().__init__()
      self.m = margin

  def forward(self, pr, y):
      p = pr[y == 1]; n = pr[y == 0]
      if len(p) > 0 and len(n) > 0:
          min_l = min(len(p), len(n)); p = p[:min_l]; n = n[torch.randperm(len(n), device=pr.device)[:min_l]]
          return torch.relu(n - p + self.m).mean()
      return torch.tensor(0.0, device=pr.device, requires_grad=True)

class BPRLoss(nn.Module):
  '''
  BPRLoss (Bayesian Personalized Ranking)
  Попарная функция потерь, оптимизирующая взаимное отношение между позитивными и негативными кандидатами. Широко применяется в рекомендательных задачах с неявной обратной связью
  '''

  def __init__(self): super().__init__()

  def forward(self, pr, y):
      p = pr[y == 1]; n = pr[y == 0]; m_l = min(len(p), len(n))
      if m_l == 0: return torch.tensor(0.0, device=pr.device, requires_grad=True)
      p = p[:m_l]; n = n[torch.randperm(len(n), device=pr.device)[:m_l]]
      return -(torch.log(torch.sigmoid(p - n) + 1e-8)).mean()

class InfoNCELoss(nn.Module):
  '''
  Контрастивный лосс
  Максимизирует косинусное сходство между истинной парой вершин и минимизирует его для случайно выбранных негативных кандидатов.
  '''

  def __init__(self, tau=0.1):
    super().__init__()
    self.t = tau

  def forward(self, pr, y):
      p = pr[y == 1]; n = pr[y == 0]
      if len(p) == 0 or len(n) == 0: return torch.tensor(0.0, device=pr.device, requires_grad=True)
      p = p / self.t; n = n / self.t; max_n = torch.max(n); sum_exp_n = torch.sum(torch.exp(n - max_n))
      lse = torch.log(torch.exp(p - max_n) + sum_exp_n) + max_n
      return -(p - lse).mean()

class GM(nn.Module):
  '''
  Конструктор класса-контейнера моделей GM.
  n_f (int): Входная размерность признаков вершин.
  n_v (int): Суммарное количество вершин в графе.
  n_a (int, по умолчанию 7): Размерность вектора эвристик, подаваемых на вход предиктора.
  o_d (int, по умолчанию 256): Выходная латентная размерность графового энкодера.
  g_type (str, по умолчанию 'none'): Тип графового энкодера ('lightgcn', 'sgc', 'sage', 'neognn', 'dirgcn').
  p_type (str, по умолчанию 'standard'): Тип предиктора связи ('standard', 'buddy', 'ncn').
  **kwargs: Свободные именованные аргументы, передаваемые в дочерние нейросетевые слои.
  '''

  def __init__(self, n_f, n_v, n_a=7, o_d=256, g_type='none', p_type='standard', **kwargs):
      super().__init__()
      self.g_type = g_type; self.p_type = p_type; self.p = nn.Linear(n_f, n_f); self.l = nn.Linear(n_f, o_d)
      if g_type == 'dirgcn': self.g = DirLightGCN(n_f, o_d)
      elif g_type == 'sgc': self.g = SGC(n_f, o_d)
      elif g_type == 'sage': self.g = SAGE(n_f, o_d)
      elif g_type == 'neognn': self.g = NeoGNN_Encoder(n_v, n_f, o_d)
      elif g_type == 'lightgcn': self.g = LightGCN(n_f, o_d)
      else: self.g = nn.Identity()
      self.norm = nn.LayerNorm(o_d); self.drop = nn.Dropout(0.3)
      if p_type == 'buddy': self.c = BUDDY_Predictor(o_d, n_a)
      elif p_type == 'ncn': self.c = NCN_Predictor(o_d, n_a)
      else: self.c = Standard_Predictor(o_d, n_a)

  def enc(self, x, v, a_e):
      h = self.p(x)
      if self.g_type == 'neognn': z = self.g(h, v, a_e)
      elif self.g_type != 'none': z = self.g(h, a_e)
      else: z = torch.relu(self.l(h))
      return self.drop(self.norm(z))

  def prd(self, z, b_e, b_a):
      u = b_e[:, 0]; v_i = b_e[:, 1]
      return self.c(z[u], z[v_i], b_a)

class GraphAlignerModel(nn.Module):
  '''
  Контрастивная модель, предназначенная для выравнивания представлений: приближает структурные эмбеддинги GNN к семантическим текстовым эмбеддингам SciBERT
  '''

  def __init__(self, c_dim=385, txt_dim=256, h_dim=256, z_dim=128):
      super().__init__()
      self.gnn = BaseSAGE(i_d=c_dim, h_d=h_dim, o_d=z_dim)
      self.aligner = AlignmentMLP(i_d=txt_dim, h_d=h_dim, o_d=z_dim)
      self.t = 0.1

  def enc_g(self, x, e_idx):
      z = self.gnn(x, e_idx)
      return torch.nn.functional.normalize(z, p=2, dim=1)

  def enc_t(self, x_txt):
      z = self.aligner(x_txt)
      return torch.nn.functional.normalize(z, p=2, dim=1)

class BaseSAGE(nn.Module):
  def __init__(self, i_d=385, h_d=256, o_d=128):
      super().__init__()
      self.c1 = SAGEConv(i_d, h_d); self.c2 = SAGEConv(h_d, o_d)
      self.drop = nn.Dropout(0.3); self.norm = nn.LayerNorm(h_d)
  def forward(self, x, e_idx):
      h = self.c1(x, e_idx); h = self.norm(h); h = torch.nn.functional.gelu(h); h = self.drop(h)
      return self.c2(h, e_idx)

class AlignmentMLP(nn.Module):
  def __init__(self, i_d=256, h_d=256, o_d=128):
      super().__init__()
      self.mlp = nn.Sequential(nn.Linear(i_d, h_d), nn.LayerNorm(h_d), nn.GELU(), nn.Dropout(0.2), nn.Linear(h_d, o_d))
  def forward(self, x): return self.mlp(x)

class TextAE(nn.Module):
  '''
  TextAE (Text Autoencoder)
  Полносвязный автоэнкодер.
  Сжимает конкатенированныеSciBERT-векторы исходной размерности 2304 в компактное латентное представление размерности 256 с минимизацией MSE
  '''

    def __init__(self, i_d=2304, h_d=1024, b_d=256):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(i_d, h_d), nn.LayerNorm(h_d), nn.GELU(), nn.Dropout(0.1), nn.Linear(h_d, b_d))
        self.dec = nn.Sequential(nn.Linear(b_d, h_d), nn.LayerNorm(h_d), nn.GELU(), nn.Dropout(0.1), nn.Linear(h_d, i_d))
    def forward(self, x): return self.dec(self.enc(x))
    def compress(self, x): return self.enc(x)

class HeteroEncoder(torch.nn.Module):
  '''
  Двудольный сверточный граф-энкодер, выполняющий агрегацию сообщений по разным типам связей (статья-автор, автор-статья, статья-статья)
  '''

  def __init__(self, in_dim, hid_dim):
      super().__init__()
      self.c1 = HeteroConv({
          ('author', 'writes', 'paper'): SAGEConv((in_dim, in_dim), hid_dim),
          ('paper', 'rev_writes', 'author'): SAGEConv((in_dim, in_dim), hid_dim),
          ('paper', 'cites', 'paper'): SAGEConv((in_dim, in_dim), hid_dim)
      }, aggr='mean')
      self.c2 = HeteroConv({
          ('author', 'writes', 'paper'): SAGEConv((hid_dim, hid_dim), hid_dim),
          ('paper', 'rev_writes', 'author'): SAGEConv((hid_dim, hid_dim), hid_dim),
          ('paper', 'cites', 'paper'): SAGEConv((hid_dim, hid_dim), hid_dim)
      }, aggr='mean')

  def forward(self, x_d, e_d):
      x_d = self.c1(x_d, e_d); x_d = {k: torch.nn.functional.relu(x) for k, x in x_d.items()}
      return self.c2(x_d, e_d)

class LinkCausalPredictor(torch.nn.Module):
  def __init__(self, dim):
      super().__init__()
      self.l1 = torch.nn.Linear(dim * 2, dim); self.l2 = torch.nn.Linear(dim, 1)

  def forward(self, z_s, z_d):
      x = torch.cat([z_s, z_d], dim=-1); x = torch.nn.functional.relu(self.l1(x))
      return self.l2(x).squeeze(-1)

class EdgePredictor(torch.nn.Module):
  def __init__(self, dim):
      super().__init__()
      self.l = torch.nn.Linear(dim * 2, 1)

  def forward(self, z_s, z_d):
      return self.l(torch.cat([z_s, z_d], dim=-1)).squeeze(-1)


## Обучение и оценка

In [ ]:
def b_g(d, m_n='pritamdeka/S-SciBERT-snli-multinli-stsb', b_s=1000, threshold=0.7):
  '''
  Строит семантический ориентированный граф сходства статей на основе векторов SciBERT под хронологическим фильтром.
  d (list): Список словарей с сырыми данными публикаций.
  m_n (str, по умолчанию 'pritamdeka/...'): Имя предобученной модели SentenceTransformer.
  b_s (int, по умолчанию 1000): Размер батча для попарного расчета косинусного сходства.
  threshold (float, по умолчанию 0.7): Порог косинусного сходства для проведения ребра.
  '''

  dv=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); m=ST(m_n).to(dv); g=nx.DiGraph()

  for x in d:
      g.add_node(x['i'], y=x['y'], na=x['na'], c=x['c'])

  ab=[x['ab'] for x in d]; ct=[x['ct'] for x in d]
  yr=torch.tensor([x['y'] for x in d], device=dv, dtype=torch.int16)

  e_a=m.encode(ab, convert_to_tensor=True, device=dv, batch_size=128).half()
  e_c=m.encode(ct, convert_to_tensor=True, device=dv, batch_size=128).half()

  n=len(d); rf=[set(x['rf']) for x in d]; au=[set(x['au']) for x in d]; i_l=[x['i'] for x in d]

  for i in range(0, n, b_s):
      i_e=min(i+b_s, n)
      y_b=yr[i:i_e]

      vl=y_b.unsqueeze(1) >= yr.unsqueeze(0)
      vl[torch.arange(i_e-i), torch.arange(i, i_e)] = False

      sim_a = ut.cos_sim(e_a[i:i_e], e_a)

      mask = (sim_a > threshold) & vl
      id_v = mask.nonzero(as_tuple=False)

      if id_v.numel() == 0: continue

      r_idx = id_v[:, 0]; c_idx = id_v[:, 1]
      s_a_vals = sim_a[r_idx, c_idx].cpu().float().numpy()
      s_c_vals = ut.cos_sim(e_c[i:i_e], e_c)[r_idx, c_idx].cpu().float().numpy()
      dt_vals = torch.abs(y_b[r_idx] - yr[c_idx]).cpu().numpy()

      e_l = []
      for k in range(len(r_idx)):
          u_idx = r_idx[k].item() + i
          v_idx = c_idx[k].item()

          a_u, a_v = au[u_idx], au[v_idx]
          common = len(a_u & a_v)
          w_au = common / len(a_u | a_v) if common > 0 else 0

          e_l.append((i_l[u_idx], i_l[v_idx], {
              'cit': 1 if i_l[v_idx] in rf[u_idx] else 0,
              'w_au': w_au,
              'w_sem': float(s_a_vals[k]),
              'w_cat': float(s_c_vals[k]),
              'w_dt': int(dt_vals[k])
          }))

      g.add_edges_from(e_l)

      del sim_a, mask, id_v, e_l
      torch.cuda.empty_cache()
      gc.collect()

  return g

def eval_model(y_t, y_p, p_n="metrics.png"):
  '''
  Рассчитывает метрики бинарной классификации (ROC-AUC, PR-AUC, F1), подбирает оптимальный порог и строит график PR-кривой.
  y_t (np.ndarray): Истинные метки классов (ground truth).
  y_p (np.ndarray): Предсказанные моделью вероятности связей.
  p_n (str, по умолчанию 'metrics.png'): Путь для сохранения графика PR-кривой.
  '''
  if len(np.unique(y_t)) < 2: raise ValueError("Требуется два класса")
  pr_c, rc_c, th_pr = sm.precision_recall_curve(y_t, y_p)
  f1_s = 2 * (pr_c * rc_c) / (pr_c + rc_c + 1e-8)
  o_i = np.argmax(f1_s); o_th = th_pr[o_i] if o_i < len(th_pr) else 0.5
  fpr, tpr, _ = sm.roc_curve(y_t, y_p); a_roc = sm.auc(fpr, tpr); a_pr = sm.auc(rc_c, pr_c)
  plt.figure(figsize=(6, 5)); plt.plot(rc_c, pr_c, color='indigo', lw=2)
  plt.title(f"PR-кривая (AUC={a_pr:.4f})"); plt.xlabel("Полнота"); plt.ylabel("Точность")
  plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig(p_n); plt.close()
  return o_th, a_roc, a_pr, f1_s[o_i]

def train_model(x, v, p_e, tr_e, tr_y, tr_a, val_e, val_y, val_a, m, kw, b_p, exp_n, loss_fn, b_sz=65536, epochs=150, lr=0.0005, p=12):
  '''
  x (torch.Tensor): Признаки вершин.
  v (torch.Tensor): Индексы вершин для NeoGNN.
  p_e (torch.Tensor): Тензор маскируемых ребер цитирования.
  tr_e (torch.Tensor): Тренировочные ребра.
  tr_y (torch.Tensor): Тренировочные таргеты.
  tr_a (torch.Tensor): Тренировочные эвристики.
  val_e, val_y, val_a: Валидационные выборки.
  m (nn.Module): Инициализированная модель.
  kw (dict): Параметры сборки модели.
  b_p (str): Путь проекта.
  exp_n (str): Имя эксперимента.
  loss_fn (nn.Module): Функция потерь (MarginLoss, BPR, InfoNCE).
  b_sz (int, по умолчанию 65536): Размер батча.
  epochs (int, по умолчанию 150): Максимальное число эпох.
  lr (float, по умолчанию 0.0005): Скорость обучения.
  p (int, по умолчанию 12): Patience для ранней остановки (Early Stopping)
  '''
  c_d = os.path.join(b_p, 'experiments', exp_n); os.makedirs(c_d, exist_ok=True)
  c_p = os.path.join(c_d, 'checkpoint.pt'); h_p = os.path.join(c_d, 'history.json')
  o = op.AdamW(m.parameters(), lr=lr, weight_decay=1e-4); crit = loss_fn.to(d)
  st_e = 0; b_l = float('inf'); s = 0; hs = {'tr': [], 'vl': []}; d_p = kw.get('d_p', 0.0)
  if os.path.exists(c_p):
      ck = torch.load(c_p, map_location=d, weights_only=False); m.load_state_dict(ck['m']); o.load_state_dict(ck['o'])
      st_e = ck['ep'] + 1; b_l = ck['b_l']; s = ck['s']; kw.update(ck.get('kw', {}))
      if os.path.exists(h_p):
          with open(h_p, 'r') as f: hs = json.load(f)
      print(f"[{time.strftime('%H:%M:%S')}] Восстановлено: {c_p}")
  for ep in range(st_e, epochs):
      m.train(); tr_l = 0; idx = torch.randperm(tr_e.size(0), device=d)
      if d_p > 0.0:
          mask = torch.rand(p_e.size(0), device=d) >= d_p; c_e = p_e[mask]
      else: c_e = p_e
      for i in range(0, tr_e.size(0), b_sz):
          b_i = idx[i:i+b_sz]; b_e = tr_e[b_i]; b_y = tr_y[b_i]; b_a = tr_a[b_i]
          o.zero_grad(); z = m.enc(x, v, c_e); pr = m.prd(z, b_e, b_a)
          l = crit(pr, b_y); l.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); o.step()
          tr_l += l.item() * b_e.size(0)
          if torch.isnan(l): raise RuntimeError("NaN batch")
      tr_l /= tr_e.size(0); m.eval()
      with torch.no_grad():
          z_v = m.enc(x, v, p_e); vl_l = 0
          for i in range(0, val_e.size(0), b_sz):
              b_e = val_e[i:i+b_sz]; b_y = val_y[i:i+b_sz]; b_a = val_a[i:i+b_sz]
              pr = m.prd(z_v, b_e, b_a); l = crit(pr, b_y); vl_l += l.item() * b_e.size(0)
          vl_l /= val_e.size(0)
      hs['tr'].append(tr_l); hs['vl'].append(vl_l)
      print(f"[{time.strftime('%H:%M:%S')}] Ep {ep+1} | Tr: {tr_l:.4f} | Vl: {vl_l:.4f}")
      if vl_l < b_l:
          b_l = vl_l; s = 0; torch.save({'m': m.state_dict(), 'o': o.state_dict(), 'ep': ep, 'b_l': b_l, 's': s, 'kw': kw}, c_p)
          with open(h_p, 'w') as f: json.dump(hs, f)
      else: s += 1
      if s >= p: print(f"[{time.strftime('%H:%M:%S')}] ES: {ep+1}"); break
  if os.path.exists(c_p): m.load_state_dict(torch.load(c_p, map_location=d, weights_only=False)['m'])
  return m

def detailed_evaluation(b_p, exp_n, te_e, te_y, te_p, o_th, d_pr_p, g_nodes):
  '''
  Проводит глубокий качественный анализ предсказаний и выводит тексты статей.
  b_p (str): Базовый путь к директории проекта.
  exp_n (str): Строковое имя текущего эксперимента.
  te_e (torch.Tensor): Тестовые ребра.
  te_y (torch.Tensor): Тестовые истинные метки.
  te_p (torch.Tensor): Предсказанные тестовые вероятности.
  o_th (float): Оптимальный порог бинаризации.
  d_pr_p (str): Путь к файлу очищенных данных JSONL.
  g_nodes (list): Список всех идентификаторов вершин графа.
  '''

  e_d = os.path.join(b_p, 'experiments', exp_n)
  h_p = os.path.join(e_d, 'history.json')
  m_p = os.path.join(e_d, 'metrics.json')

  print(f"{'='*100}\nОТЧЕТ ОБ ЭКСПЕРИМЕНТЕ: {exp_n}\n{'='*100}")

  if os.path.exists(m_p):
      with open(m_p, 'r') as f: m_d = json.load(f)
      df_m = pl.DataFrame([{"Метрика": k, "Значение": str(round(v, 4)) if isinstance(v, float) else str(v)} for k, v in m_d.items()])
      display(df_m)

  if os.path.exists(h_p):
      with open(h_p, 'r') as f: hs = json.load(f)
      plt.figure(figsize=(12, 6), dpi=300)
      plt.plot(hs['tr'], label='Focal Loss (Train)', color='royalblue', lw=2)
      plt.plot(hs['vl'], label='Focal Loss (Val)', color='crimson', lw=2)
      plt.title('Кривая обучения', fontsize=16)
      plt.xlabel('Эпоха', fontsize=12); plt.ylabel('Loss', fontsize=12)
      plt.grid(True, linestyle='--', alpha=0.6); plt.legend(fontsize=12)
      plt.tight_layout(); plt.show()

  print(f"\nСбор метаданных статей...")
  i_to_m = {i: nd for i, nd in enumerate(g_nodes)}
  ab_d = {}
  with open(d_pr_p, 'r', encoding='utf-8') as f:
      for ln in f:
          if not ln.strip(): continue
          x = json.loads(ln)
          ab_d[x['i']] = {'ct': x.get('ct_f', x.get('ct', '')), 'ab': x.get('ab', '').replace('\n', ' ')}

  te_p_np = te_p.numpy(); te_y_np = te_y.numpy(); te_e_np = te_e.numpy()
  tp_i = np.where((te_y_np == 1) & (te_p_np >= o_th))[0]
  fp_i = np.where((te_y_np == 0) & (te_p_np >= o_th))[0]

  np.random.shuffle(tp_i); np.random.shuffle(fp_i)

  def p_r(idx_list, t_name, max_n=10):
      print(f"\n{t_name} (Топ {max_n} случайных):\n{'-'*100}")
      for i in idx_list[:max_n]:
          u = i_to_m[te_e_np[i, 0]]; v = i_to_m[te_e_np[i, 1]]
          pr = te_p_np[i]
          u_d = ab_d.get(u, {'ct': '', 'ab': ''}); v_d = ab_d.get(v, {'ct': '', 'ab': ''})
          print(f"[{pr:.4f}] Исходящая: {u} | Теги: {u_d['ct']}\n    Текст: {u_d['ab'][:300]}...")
          print(f"         Целевая: {v} | Теги: {v_d['ct']}\n    Текст: {v_d['ab'][:300]}...\n")

  p_r(tp_i, "ПРАВИЛЬНЫЕ ПРЕДСКАЗАНИЯ (Верно предсказанные существующие цитирования)")
  p_r(fp_i, "НЕВЕРНЫЕ ПРЕДСКАЗАНИЯ (Модель предсказала цитирование, которого нет)")

def evaluate_full_ranking(m, x, v_t, tr_p_e, te_e, y_a, a_sym, a_aa, m_ap, m_ap_T, a_deg, kw, d, k=10, b_sz=16):
  '''
  Оценивает качество GNN ранжирования при холодном старте на тесте.
  k (int, по умолчанию 10): Глубина ранжирования для метрик Recall@K, Hits@K, NDCG@K.
  b_sz (int, по умолчанию 16): Размер батча запросов.
  (остальные параметры — модель, эмбеддинги и разреженные матрицы)
  '''
  print(f"[{time.strftime('%H:%M:%S')}] Full-ranking оценка")
  m.eval(); n_v = x.size(0); gt_d = collections.defaultdict(list)
  for u, v in te_e.tolist(): gt_d[int(u)].append(int(v))
  u_l = list(gt_d.keys()); te_u_c = torch.tensor(u_l, dtype=torch.long); k_p = kw.get('k_p', 5)

  idx_l = []; a_f_n = torch.nn.functional.normalize(x[:, :256].to(d), p=2, dim=1); y_a_d = y_a.to(d)
  B_r = 1024
  for i in range(0, te_u_c.size(0), B_r):
      b_u = te_u_c[i:i+B_r].to(d); u_f_n = torch.nn.functional.normalize(x[b_u, :256].to(d), p=2, dim=1)
      c = torch.mm(u_f_n, a_f_n.t())
      m_dt = y_a_d.unsqueeze(0) > y_a_d[b_u].unsqueeze(1)
      m_sl = torch.arange(n_v, device=d).unsqueeze(0) == b_u.unsqueeze(1)
      c[m_dt | m_sl] = -2.0
      _, b_idx = torch.topk(c, k_p, dim=1); idx_l.append(b_idx)
      del c, m_dt, m_sl, u_f_n, b_idx; torch.cuda.empty_cache()

  idx = torch.cat(idx_l, dim=0); te_u = te_u_c.to(d)
  p_e = torch.cat([te_u.unsqueeze(1).expand(-1, k_p).reshape(-1, 1), idx.reshape(-1, 1)], dim=1)
  c_e = torch.cat([tr_p_e.to(d), p_e], dim=0)
  with torch.no_grad(): z = m.enc(x.to(d), v_t.to(d), c_e)

  mrr_s = 0.0; rec_s = 0.0; hits_s = 0.0; ndcg_s = 0.0; N_u = len(u_l)
  if N_u == 0: raise ValueError("Нет позитивных примеров")

  with torch.no_grad():
      for i in range(N_u):
          u_i = u_l[i]; b_u = torch.tensor([u_i], dtype=torch.long, device=d)
          h = get_batch_heuristics(b_u, n_v, y_a, a_sym, a_aa, m_ap, m_ap_T, a_deg, d).squeeze(0)
          z_u = z[u_i].unsqueeze(0).expand(n_v, -1); z_v = z
          sc = m.c(z_u, z_v, h[:, :kw['n_a']])
          m_dt2 = y_a_d > y_a_d[u_i]
          sc[m_dt2] = -float('inf'); sc[u_i] = -float('inf')
          r = torch.argsort(sc, descending=True).cpu().numpy(); t_v = set(gt_d[u_i])
          pos = np.where(np.isin(r, list(t_v)))[0] + 1
          if len(pos) == 0: continue
          mrr_s += 1.0 / pos[0]; t_k = set(r[:k]); hits = len(t_v & t_k)
          rec_s += hits / len(t_v); hits_s += 1.0 if hits > 0 else 0.0
          dcg = np.sum([1.0 / math.log2(p + 1) for p in pos if p <= k])
          idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(t_v), k) + 1)])
          ndcg_s += dcg / idcg if idcg > 0 else 0.0
          del h, z_u, sc, m_dt2, r; torch.cuda.empty_cache()

  del idx_l, a_f_n, y_a_d, idx, p_e, c_e, z, te_u, te_u_c; torch.cuda.empty_cache(); import gc; gc.collect()
  return mrr_s / N_u, rec_s / N_u, hits_s / N_u, ndcg_s / N_u

def score_edges(e, a):
  '''
  Оценивает ребра на основе базовой косинусной семантики и авторского перекрытия.
  e (torch.Tensor): Оцениваемые ребра.
  a (torch.Tensor): Матрица эвристических признаков для ребер
  '''

  u = e[:, 0]; v = e[:, 1]
  e_ab_u = x_f[u, :768]; e_ab_v = x_f[v, :768]
  cos_ab = torch.cosine_similarity(e_ab_u, e_ab_v, dim=1)
  e_ct_u = x_f[u, 768:1536]; e_ct_v = x_f[v, 768:1536]
  cos_ct = torch.cosine_similarity(e_ct_u, e_ct_v, dim=1)
  au_o = a[:, 1]
  y_prob = 0.4 * cos_ab + 0.4 * cos_ct + 0.2 * au_o
  return y_prob

def g_s_b(e_tns, a_tns, x_f, b_sz=65536):
  '''
  Быстро рассчитывает семантические косинусы батчами на GPU.
  e_tns (torch.Tensor): Тензор ребер.
  a_tns (torch.Tensor): Тензор эвристик.
  x_f (torch.Tensor): Матрица латентных векторов статей.
  b_sz (int, по умолчанию 65536): Размер пачки для обработки
  '''

  s_l = []
  for i in range(0, e_tns.size(0), b_sz):
      b_e = e_tns[i:i+b_sz]; b_a = a_tns[i:i+b_sz]; u = b_e[:, 0]; v = b_e[:, 1]
      c_ab = torch.cosine_similarity(x_f[u, :768], x_f[v, :768], dim=1)
      c_ct = torch.cosine_similarity(x_f[u, 768:1536], x_f[v, 768:1536], dim=1)
      c_tp = torch.cosine_similarity(x_f[u, 1537:1665], x_f[v, 1537:1665], dim=1)
      cn_n = (b_a[:, 2] - b_a[:, 2].min()) / (b_a[:, 2].max() - b_a[:, 2].min() + 1e-8)
      s_l.append(0.2 * c_ab + 0.2 * c_ct + 0.3 * c_tp + 0.15 * b_a[:, 1] + 0.15 * cn_n)
  return torch.cat(s_l)

def build_tabular_data(edges, labels, x_f, y_a, a_sym, a_aa, m_ap, m_ap_T, a_deg, d):
  '''
  Генерирует плоский табличный датасет для классических GBDT на GPU.
  edges (torch.Tensor): Сэмплированные ребра (позитивные + негативные).
  labels (torch.Tensor): Метки классов (1 — цитирование, 0 — нет цитирования).
  x_f (torch.Tensor): Матрица латентных признаков статей.
  y_a (torch.Tensor): Года публикаций.
  (остальные параметры — разреженные GPU матрицы для расчета эвристик)
  '''

  print(f"[{time.strftime('%H:%M:%S')}] Сборка табличного датасета")
  u = edges[:, 0]; idx = torch.argsort(u)
  e_s = edges[idx]; y_s = labels[idx]
  u_s = e_s[:, 0]; v_s = e_s[:, 1]; B = 4096; f_l = []; n_v = x_f.size(0)
  for i in range(0, e_s.size(0), B):
      b_u_cpu = u_s[i:i+B]; b_v_cpu = v_s[i:i+B]
      b_u = b_u_cpu.to(d); b_v = b_v_cpu.to(d)
      h = get_pair_heuristics(b_u, b_v, n_v, y_a.to(d), a_sym, a_aa, m_ap, m_ap_T, a_deg, d)

      z_u_ab = torch.nn.functional.normalize(x_f[b_u_cpu, :768].to(d), p=2, dim=1)
      z_v_ab = torch.nn.functional.normalize(x_f[b_v_cpu, :768].to(d), p=2, dim=1)
      c_ab = (z_u_ab * z_v_ab).sum(dim=1, keepdim=True)

      z_u_ct = torch.nn.functional.normalize(x_f[b_u_cpu, 768:1536].to(d), p=2, dim=1)
      z_v_ct = torch.nn.functional.normalize(x_f[b_v_cpu, 768:1536].to(d), p=2, dim=1)
      c_ct = (z_u_ct * z_v_ct).sum(dim=1, keepdim=True)

      if x_f.size(1) > 1537:
          z_u_tp = torch.nn.functional.normalize(x_f[b_u_cpu, 1537:1665].to(d), p=2, dim=1)
          z_v_tp = torch.nn.functional.normalize(x_f[b_v_cpu, 1537:1665].to(d), p=2, dim=1)
          c_tp = (z_u_tp * z_v_tp).sum(dim=1, keepdim=True)
      else:
          c_tp = torch.zeros((b_u_cpu.size(0), 1), device=d)

      f_l.append(torch.cat([h, c_ab, c_ct, c_tp], dim=1).cpu())
      torch.cuda.empty_cache()
  X = torch.cat(f_l, dim=0).numpy(); y = y_s.numpy(); grp = u_s.numpy()
  return X, y, grp

def evaluate_two_stage(m_cb, x, te_p_e, y_a, a_sym, a_aa, m_ap, m_ap_T, a_deg, d, k_cand=200, k=10, b_sz=16):
  '''
  Оценивает двухстадийный пайплайн с CatBoostRanker на тесте.
  k_cand (int, по умолчанию 200): Количество кандидатов, отбираемых на первой стадии семантического поиска (Retrieval).
  '''
  print(f"[{time.strftime('%H:%M:%S')}] Two-Stage оценка (Retrieval K={k_cand})")
  gt_d = collections.defaultdict(list)
  for u, v in te_p_e.tolist(): gt_d[int(u)].append(int(v))
  u_l = list(gt_d.keys()); N_u = len(u_l); n_v = x.size(0)
  mrr_s = 0.0; rec_s = 0.0; hits_s = 0.0; ndcg_s = 0.0

  for i in range(0, N_u, b_sz):
      b_u_cpu = torch.tensor(u_l[i:i+b_sz], dtype=torch.long, device='cpu')
      b_u = b_u_cpu.to(d); B = b_u.size(0)

      z_u = torch.nn.functional.normalize(x[b_u_cpu, :768].to(d), p=2, dim=1)
      z_all = torch.nn.functional.normalize(x[:, :768].to(d), p=2, dim=1)
      sim = torch.mm(z_u, z_all.t())

      m_dt = y_a.unsqueeze(0).to(d) > y_a[b_u_cpu].unsqueeze(1).to(d)
      m_sl = torch.arange(n_v, device=d).unsqueeze(0) == b_u.unsqueeze(1)
      sim[m_dt | m_sl] = -2.0
      _, idx = torch.topk(sim, k_cand, dim=1)

      for j in range(B):
          u_i = u_l[i+j]; cands = idx[j]; cands_cpu = cands.cpu(); t_v = set(gt_d[u_i])
          u_tns = torch.full((k_cand,), u_i, dtype=torch.long, device=d)
          h_p = get_pair_heuristics(u_tns, cands, n_v, y_a.to(d), a_sym, a_aa, m_ap, m_ap_T, a_deg, d)

          z_u_c = z_u[j].unsqueeze(0); z_v_c = z_all[cands]
          c_ab = (z_u_c * z_v_c).sum(dim=1, keepdim=True)

          z_u_ct = torch.nn.functional.normalize(x[u_i, 768:1536].unsqueeze(0).to(d), p=2, dim=1)
          z_v_ct = torch.nn.functional.normalize(x[cands_cpu, 768:1536].to(d), p=2, dim=1)
          c_ct = (z_u_ct * z_v_ct).sum(dim=1, keepdim=True)

          if x.size(1) > 1537:
              z_u_tp = torch.nn.functional.normalize(x[u_i, 1537:1665].unsqueeze(0).to(d), p=2, dim=1)
              z_v_tp = torch.nn.functional.normalize(x[cands_cpu, 1537:1665].to(d), p=2, dim=1)
              c_tp = (z_u_tp * z_v_tp).sum(dim=1, keepdim=True)
          else:
              c_tp = torch.zeros((k_cand, 1), device=d)

          f_tns = torch.cat([h_p, c_ab, c_ct, c_tp], dim=1).cpu().numpy()
          sc = m_cb.predict(f_tns); r_idx = np.argsort(sc)[::-1]; s_cands = cands_cpu.numpy()[r_idx]

          pos = np.where(np.isin(s_cands, list(t_v)))[0] + 1
          if len(pos) == 0: continue
          mrr_s += 1.0 / pos[0]; t_k = set(s_cands[:k]); hits = len(t_v & t_k)
          rec_s += hits / len(t_v); hits_s += 1.0 if hits > 0 else 0.0
          dcg = np.sum([1.0 / math.log2(p + 1) for p in pos if p <= k])
          idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(t_v), k) + 1)])
          ndcg_s += dcg / idcg if idcg > 0 else 0.0

  return mrr_s / N_u, rec_s / N_u, hits_s / N_u, ndcg_s / N_u

def extract_features_opt(e, y_lbl, x_f, y_a, neighs, au_sets, w_aa_np, d_dv):
  '''
  Строит оптимизированную плоскую матрицу признаков на гибридном CPU-GPU конвейере.
  e (torch.Tensor): Сэмплированные ребра.
  y_lbl (torch.Tensor): Таргеты.
  x_f (torch.Tensor): Матрица признаков статей.
  y_a (torch.Tensor): Года публикаций.
  neighs (list): Списки смежности окрестностей для каждой вершины (для эвристики CN).
  au_sets (list): Множества ID авторов для каждой статьи (для перекрытия Jaccard).
  w_aa_np (numpy.ndarray): Веса Адамика-Адар для вершин.
  d_dv (torch.device): Вычислительное устройство для векторных операций
  '''
  u = e[0].numpy(); v = e[1].numpy()
  N_e = len(u)

  print(f"[{time.strftime('%H:%M:%S')}] Сборка эвристик (CPU)")
  h_res = np.zeros((N_e, 4), dtype=np.float32)
  y_a_np = y_a.numpy()

  for i in range(N_e):
      ui, vi = u[i], v[i]
      h_res[i, 0] = y_a_np[ui] - y_a_np[vi]

      su = au_sets[ui]; sv = au_sets[vi]
      un = len(su) + len(sv) - len(su & sv)
      if un > 0: h_res[i, 1] = len(su & sv) / un

      cn_s = neighs[ui] & neighs[vi]
      h_res[i, 2] = len(cn_s)
      if h_res[i, 2] > 0: h_res[i, 3] = sum(w_aa_np[x] for x in cn_s)

  h_tns = torch.tensor(h_res, dtype=torch.float32)
  h_tns[:, 2:] = torch.log(h_tns[:, 2:] + 1.0)

  print(f"[{time.strftime('%H:%M:%S')}] Векторные признаки (GPU)")
  B = 4096; f_l = []
  u_tns = e[0]; v_tns = e[1]

  for i in range(0, N_e, B):
      b_u = u_tns[i:i+B]; b_v = v_tns[i:i+B]

      z_u_txt = x_f[b_u, :256].to(d_dv); z_v_txt = x_f[b_v, :256].to(d_dv)
      z_u_grp = x_f[b_u, 257:].to(d_dv); z_v_grp = x_f[b_v, 257:].to(d_dv)

      c_txt = torch.cosine_similarity(z_u_txt, z_v_txt, dim=1).unsqueeze(1)
      c_grp = torch.cosine_similarity(z_u_grp, z_v_grp, dim=1).unsqueeze(1)

      had_txt = z_u_txt * z_v_txt
      had_grp = z_u_grp * z_v_grp

      b_f = torch.cat([c_txt, c_grp, had_txt, had_grp], dim=1).half().cpu()
      f_l.append(b_f)

  X_emb = torch.cat(f_l, dim=0)
  X = torch.cat([h_tns.half(), X_emb], dim=1)

  return X, y_lbl, e[0]

def evaluate_two_stage_cb(m_cb, x_f, te_p_e, y_a, neighs, au_sets, w_aa_np, d_dv, k_cand=200, k=10, b_sz=16):
  '''
  Оценивает двухстадийный пайплайн ранжирования с базовой моделью CatBoost (без стекинга GNN).
  m_cb (CatBoost): Обученная модель CatBoost.
  x_f (torch.Tensor): Семантическая матрица признаков статей.
  te_p_e (torch.Tensor): Тестовые положительные ребра цитирования.
  y_a (torch.Tensor): Массив годов издания.
  neighs (list): Списки смежности локальных окрестностей.
  au_sets (list): Множества ID соавторов для каждой статьи.
  w_aa_np (np.ndarray): Веса Адамика-Адар для вершин.
  d_dv (torch.device): Рабочее вычислительное устройство.
  k_cand (int, по умолчанию 200): Количество кандидатов, отбираемых на первой стадии Retrieval.
  k (int, по умолчанию 10): Глубина оценки ранжирования (например, NDCG@10).
  b_sz (int, по умолчанию 16): Размер батча для инференса.
  '''
  print(f"[{time.strftime('%H:%M:%S')}] Two-Stage Inference (Retrieval K={k_cand})")
  gt_d = collections.defaultdict(set)
  for u, v in te_p_e.tolist(): gt_d[int(u)].add(int(v))
  u_l = list(gt_d.keys()); N_u = len(u_l); n_v = x_f.size(0)
  mrr_s = 0.0; rec_s = 0.0; hits_s = 0.0; ndcg_s = 0.0

  y_a_np = y_a.numpy()
  x_txt_gpu = x_f[:, :256].to(d_dv)
  y_a_gpu = y_a.to(d_dv)

  for i in range(0, N_u, b_sz):
      b_u_cpu = torch.tensor(u_l[i:i+b_sz], dtype=torch.long)
      b_u = b_u_cpu.to(d_dv); B = b_u.size(0)

      z_u = x_txt_gpu[b_u]
      z_u_n = torch.nn.functional.normalize(z_u, p=2, dim=1)
      z_all_n = torch.nn.functional.normalize(x_txt_gpu, p=2, dim=1)
      sim = torch.mm(z_u_n, z_all_n.t())

      m_dt = y_a_gpu.unsqueeze(0) > y_a_gpu[b_u].unsqueeze(1)
      m_sl = torch.arange(n_v, device=d_dv).unsqueeze(0) == b_u.unsqueeze(1)
      sim[m_dt | m_sl] = -2.0

      _, idx = torch.topk(sim, k_cand, dim=1)
      cands_cpu = idx.cpu()

      for j in range(B):
          u_i = u_l[i+j]; t_v = gt_d[u_i]; cands = cands_cpu[j].numpy()

          h_res = np.zeros((k_cand, 4), dtype=np.float32)
          su = au_sets[u_i]; nu = neighs[u_i]
          for c_idx, vi in enumerate(cands):
              h_res[c_idx, 0] = y_a_np[u_i] - y_a_np[vi]
              sv = au_sets[vi]; un = len(su) + len(sv) - len(su & sv)
              if un > 0: h_res[c_idx, 1] = len(su & sv) / un
              cn_s = nu & neighs[vi]
              h_res[c_idx, 2] = len(cn_s)
              if h_res[c_idx, 2] > 0: h_res[c_idx, 3] = sum(w_aa_np[x] for x in cn_s)

          h_tns = torch.tensor(h_res, dtype=torch.float32)
          h_tns[:, 2:] = torch.log(h_tns[:, 2:] + 1.0)

          c_tns = torch.from_numpy(cands)
          z_u_txt = x_f[u_i, :256].unsqueeze(0); z_v_txt = x_f[c_tns, :256]
          z_u_grp = x_f[u_i, 257:].unsqueeze(0); z_v_grp = x_f[c_tns, 257:]

          c_txt = torch.cosine_similarity(z_u_txt, z_v_txt, dim=1).unsqueeze(1)
          c_grp = torch.cosine_similarity(z_u_grp, z_v_grp, dim=1).unsqueeze(1)
          had_txt = z_u_txt * z_v_txt
          had_grp = z_u_grp * z_v_grp

          X_cand = torch.cat([h_tns, c_txt, c_grp, had_txt, had_grp], dim=1).numpy()
          sc = m_cb.predict(X_cand)
          r_idx = np.argsort(sc)[::-1]; s_cands = cands[r_idx]

          pos = np.where(np.isin(s_cands, list(t_v)))[0] + 1
          if len(pos) == 0: continue

          mrr_s += 1.0 / pos[0]; t_k = set(s_cands[:k]); hits = len(t_v & t_k)
          rec_s += hits / len(t_v); hits_s += 1.0 if hits > 0 else 0.0
          dcg = np.sum([1.0 / math.log2(p + 1) for p in pos if p <= k])
          idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(t_v), k) + 1)])
          ndcg_s += dcg / idcg if idcg > 0 else 0.0

  return mrr_s / N_u, rec_s / N_u, hits_s / N_u, ndcg_s / N_u

def extract_features_stacking(edges_index, labels, gnn_embeddings, text_embeddings, years_tensor, adj_csr, adamic_adar_csr, author_csr, author_degrees, device):
  '''
  Строит глубокое признаковое пространство (1542 фичи) для двухстадийного CatBoost стекинга.
  edges_index (torch.Tensor): Ребра.
  labels (torch.Tensor): Метки.
  gnn_embeddings (torch.Tensor): Векторы, полученные на выходе GNN-модели.
  text_embeddings (torch.Tensor): Текстовые латентные представления (автоэнкодер).
  years_tensor (torch.Tensor): Года.
  (остальные параметры — CSR-матрицы для расчета сетевых эвристик)
  '''

  print(f"[{time.strftime('%H:%M:%S')}] Сортировка графа для группировки CatBoost...")
  sort_indices = np.argsort(edges_index[0].numpy())
  u = edges_index[0].numpy()[sort_indices]
  v = edges_index[1].numpy()[sort_indices]
  result_y = labels[sort_indices].numpy()
  result_groups = u

  num_edges = len(u)
  years_np = years_tensor.numpy()
  num_features = 1542

  print(f"[{time.strftime('%H:%M:%S')}] Выделение памяти под матрицу: {num_edges * num_features * 4 / 1024**2:.2f} MB")
  result_x = np.empty((num_edges, num_features), dtype=np.float32)

  batch_size = 32768
  u_tensor = edges_index[0][sort_indices]
  v_tensor = edges_index[1][sort_indices]

  for i in range(0, num_edges, batch_size):
      end_idx = min(i + batch_size, num_edges)
      b_u = u[i:end_idx]; b_v = v[i:end_idx]

      delta_time = years_np[b_u] - years_np[b_v]

      mat_u = author_csr[b_u]; mat_v = author_csr[b_v]
      intersect_auth = mat_u.multiply(mat_v).sum(axis=1).A1
      union_auth = author_degrees[b_u] + author_degrees[b_v] - intersect_auth
      author_overlap = np.zeros_like(intersect_auth, dtype=np.float32)
      mask_auth = union_auth > 0
      author_overlap[mask_auth] = intersect_auth[mask_auth] / union_auth[mask_auth]

      mat_adj_u = adj_csr[b_u]; mat_adj_v = adj_csr[b_v]
      common_neighbors = np.log(mat_adj_u.multiply(mat_adj_v).sum(axis=1).A1 + 1.0)

      mat_aa_u = adamic_adar_csr[b_u]
      adamic_adar = np.log(mat_aa_u.multiply(mat_adj_v).sum(axis=1).A1 + 1.0)

      heuristics = np.stack([delta_time, author_overlap, common_neighbors, adamic_adar], axis=1)

      b_u_cpu = u_tensor[i:end_idx]; b_v_cpu = v_tensor[i:end_idx]

      z_u_g = gnn_embeddings[b_u_cpu].to(device); z_v_g = gnn_embeddings[b_v_cpu].to(device)
      x_u_t = text_embeddings[b_u_cpu].to(device); x_v_t = text_embeddings[b_v_cpu].to(device)

      cos_z = torch.cosine_similarity(z_u_g, z_v_g, dim=1).unsqueeze(1); had_z = z_u_g * z_v_g; diff_z = torch.abs(z_u_g - z_v_g)
      cos_x = torch.cosine_similarity(x_u_t, x_v_t, dim=1).unsqueeze(1); had_x = x_u_t * x_v_t; diff_x = torch.abs(x_u_t - x_v_t)

      gpu_feat = torch.cat([cos_z, had_z, diff_z, cos_x, had_x, diff_x, z_u_g, z_v_g, x_u_t, x_v_t], dim=1).cpu().numpy().astype(np.float32)

      result_x[i:end_idx] = np.hstack([heuristics, gpu_feat])

      del z_u_g, z_v_g, x_u_t, x_v_t, cos_z, had_z, diff_z, cos_x, had_x, diff_x, gpu_feat, heuristics
      if torch.cuda.is_available(): torch.cuda.empty_cache()

  gc.collect()
  return result_x, result_y, result_groups

def evaluate_two_stage_cb_stacking(catboost_model, gnn_embeddings, text_embeddings, test_edges, years_tensor, adj_csr, adamic_adar_csr, author_csr, author_degrees, device, top_k_cands=150, eval_k=10, batch_size=256):
  '''
  Вычисляет ранжирующие метрики для стекинг-бустинга (GNN + тексты + эвристики).
  top_k_cands (int, по умолчанию 150): Число кандидатов для отбора на первой стадии.
  eval_k (int, по умолчанию 10): Метрика NDCG@K
  '''

  num_nodes = gnn_embeddings.size(0)

  gt_dict = collections.defaultdict(set)
  for u, v in test_edges.tolist(): gt_dict[int(u)].add(int(v))
  queries_list = list(gt_dict.keys()); num_queries = len(queries_list)
  mrr_sum = 0.0; rec_sum = 0.0; hits_sum = 0.0; ndcg_sum = 0.0

  years_np = years_tensor.numpy(); years_gpu = years_tensor.to(device)

  with torch.no_grad():
      gnn_gpu = gnn_embeddings.to(device)
      text_gpu = text_embeddings.to(device)

      text_gpu_norm = torch.nn.functional.normalize(text_gpu, p=2, dim=1)

      print(f"[{time.strftime('%H:%M:%S')}] Исправленный инференс Cold Start (Queries={num_queries})...")
      for i in range(0, num_queries, batch_size):
          b_u_cpu = torch.tensor(queries_list[i:i+batch_size], dtype=torch.long)
          b_u = b_u_cpu.to(device); b_len = b_u.size(0)

          sim = torch.mm(text_gpu_norm[b_u], text_gpu_norm.t())

          sim[years_gpu.unsqueeze(0) > years_gpu[b_u].unsqueeze(1)] = -2.0
          sim[torch.arange(b_len, device=device), b_u] = -2.0

          _, idx = torch.topk(sim, top_k_cands, dim=1)
          cands_cpu = idx.cpu().numpy()
          del sim, idx

          flat_q = np.repeat(queries_list[i:i+batch_size], top_k_cands)
          flat_c = cands_cpu.flatten()

          delta_time = years_np[flat_q] - years_np[flat_c]

          mat_u = author_csr[flat_q]; mat_v = author_csr[flat_c]
          intersect_auth = mat_u.multiply(mat_v).sum(axis=1).A1
          union_auth = author_degrees[flat_q] + author_degrees[flat_c] - intersect_auth
          author_overlap = np.zeros_like(intersect_auth, dtype=np.float32)
          mask_auth = union_auth > 0
          author_overlap[mask_auth] = intersect_auth[mask_auth] / union_auth[mask_auth]

          common_neighbors = np.zeros(len(flat_q), dtype=np.float32)
          adamic_adar = np.zeros(len(flat_q), dtype=np.float32)

          heuristics = np.stack([delta_time, author_overlap, common_neighbors, adamic_adar], axis=1, dtype=np.float32)

          q_tns = torch.from_numpy(flat_q).to(device)
          c_tns = torch.from_numpy(flat_c).to(device)

          z_q = gnn_gpu[q_tns]; z_c = gnn_gpu[c_tns]
          x_q = text_gpu[q_tns]; x_c = text_gpu[c_tns]

          cos_z = torch.cosine_similarity(z_q, z_c, dim=1).unsqueeze(1); had_z = z_q * z_c; diff_z = torch.abs(z_q - z_c)
          cos_x = torch.cosine_similarity(x_q, x_c, dim=1).unsqueeze(1); had_x = x_q * x_c; diff_x = torch.abs(x_q - x_c)

          gpu_feat = torch.cat([cos_z, had_z, diff_z, cos_x, had_x, diff_x, z_q, z_c, x_q, x_c], dim=1).half().cpu().numpy().astype(np.float32)

          cand_features = np.hstack([heuristics, gpu_feat])
          scores = catboost_model.predict(cand_features)

          scores_matrix = scores.reshape(b_len, top_k_cands)

          for j in range(b_len):
              query_node = queries_list[i+j]; truth_set = gt_dict[query_node]
              candidates = cands_cpu[j]

              sort_idx = np.argsort(scores_matrix[j])[::-1]
              sorted_cands = candidates[sort_idx]

              positions = np.where(np.isin(sorted_cands, list(truth_set)))[0] + 1
              if len(positions) == 0: continue

              mrr_sum += 1.0 / positions[0]; top_k_set = set(sorted_cands[:eval_k]); hits = len(truth_set & top_k_set)
              rec_sum += hits / len(truth_set); hits_sum += 1.0 if hits > 0 else 0.0
              dcg = np.sum([1.0 / math.log2(p + 1) for p in positions if p <= eval_k])
              idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(truth_set), eval_k) + 1)])
              ndcg_sum += dcg / idcg if idcg > 0 else 0.0

          del q_tns, c_tns, z_q, z_c, x_q, x_c, cos_z, had_z, diff_z, cos_x, had_x, diff_x, gpu_feat, cand_features

  del text_gpu_norm, years_gpu, gnn_gpu, text_gpu; gc.collect()
  return mrr_sum / num_queries, rec_sum / num_queries, hits_sum / num_queries, ndcg_sum / num_queries

def evaluate_two_stage_cb_stacking_cold(catboost_model, gnn_embeddings, text_embeddings, test_edges, years_tensor, adj_csr, adamic_adar_csr, author_csr, author_degrees, device, top_k_cands=150, eval_k=10, batch_size=256):
  '''
  Оценивает стекинг-бустинг в режиме жесткого холодного старта (с принудительным занулением Common Neighbors и Adamic-Adar).
  '''
  num_nodes = gnn_embeddings.size(0)

  gt_dict = collections.defaultdict(set)
  for u, v in test_edges.tolist(): gt_dict[int(u)].add(int(v))
  queries_list = list(gt_dict.keys()); num_queries = len(queries_list)
  mrr_sum = 0.0; rec_sum = 0.0; hits_sum = 0.0; ndcg_sum = 0.0

  years_np = years_tensor.numpy(); years_gpu = years_tensor.to(device)

  with torch.no_grad():
      gnn_gpu = gnn_embeddings.to(device)
      text_gpu = text_embeddings.to(device)
      text_gpu_norm = torch.nn.functional.normalize(text_gpu, p=2, dim=1)

      print(f"[{time.strftime('%H:%M:%S')}] Инференс Cold Start (Queries={num_queries})...")
      for i in range(0, num_queries, batch_size):
          b_u_cpu = torch.tensor(queries_list[i:i+batch_size], dtype=torch.long)
          b_u = b_u_cpu.to(device); b_len = b_u.size(0)

          sim = torch.mm(text_gpu_norm[b_u], text_gpu_norm.t())

          sim[years_gpu.unsqueeze(0) > years_gpu[b_u].unsqueeze(1)] = -2.0
          sim[torch.arange(b_len, device=device), b_u] = -2.0

          _, idx = torch.topk(sim, top_k_cands, dim=1)
          cands_cpu = idx.cpu().numpy()
          del sim, idx

          flat_q = np.repeat(queries_list[i:i+batch_size], top_k_cands)
          flat_c = cands_cpu.flatten()

          delta_time = years_np[flat_q] - years_np[flat_c]

          mat_u = author_csr[flat_q]; mat_v = author_csr[flat_c]
          intersect_auth = mat_u.multiply(mat_v).sum(axis=1).A1
          union_auth = author_degrees[flat_q] + author_degrees[flat_c] - intersect_auth
          author_overlap = np.zeros_like(intersect_auth, dtype=np.float32)
          mask_auth = union_auth > 0
          author_overlap[mask_auth] = intersect_auth[mask_auth] / union_auth[mask_auth]

          common_neighbors = np.zeros(len(flat_q), dtype=np.float32)
          adamic_adar = np.zeros(len(flat_q), dtype=np.float32)

          heuristics = np.stack([delta_time, author_overlap, common_neighbors, adamic_adar], axis=1, dtype=np.float32)

          q_tns = torch.from_numpy(flat_q).to(device)
          c_tns = torch.from_numpy(flat_c).to(device)

          z_q = gnn_gpu[q_tns]; z_c = gnn_gpu[c_tns]
          x_q = text_gpu[q_tns]; x_c = text_gpu[c_tns]

          cos_z = torch.cosine_similarity(z_q, z_c, dim=1).unsqueeze(1); had_z = z_q * z_c; diff_z = torch.abs(z_q - z_c)
          cos_x = torch.cosine_similarity(x_q, x_c, dim=1).unsqueeze(1); had_x = x_q * x_c; diff_x = torch.abs(x_q - x_c)

          gpu_feat = torch.cat([cos_z, had_z, diff_z, cos_x, had_x, diff_x, z_q, z_c, x_q, x_c], dim=1).half().cpu().numpy().astype(np.float32)

          cand_features = np.hstack([heuristics, gpu_feat])
          scores = catboost_model.predict(cand_features)

          scores_matrix = scores.reshape(b_len, top_k_cands)

          for j in range(b_len):
              query_node = queries_list[i+j]; truth_set = gt_dict[query_node]
              candidates = cands_cpu[j]

              sort_idx = np.argsort(scores_matrix[j])[::-1]
              sorted_cands = candidates[sort_idx]

              positions = np.where(np.isin(sorted_cands, list(truth_set)))[0] + 1
              if len(positions) == 0: continue

              mrr_sum += 1.0 / positions[0]; top_k_set = set(sorted_cands[:eval_k]); hits = len(truth_set & top_k_set)
              rec_sum += hits / len(truth_set); hits_sum += 1.0 if hits > 0 else 0.0
              dcg = np.sum([1.0 / math.log2(p + 1) for p in positions if p <= eval_k])
              idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(truth_set), eval_k) + 1)])
              ndcg_sum += dcg / idcg if idcg > 0 else 0.0

          del q_tns, c_tns, z_q, z_c, x_q, x_c, cos_z, had_z, diff_z, cos_x, had_x, diff_x, gpu_feat, cand_features

  del text_gpu_norm, years_gpu, gnn_gpu, text_gpu; gc.collect()
  if torch.cuda.is_available(): torch.cuda.empty_cache()
  return mrr_sum / num_queries, rec_sum / num_queries, hits_sum / num_queries, ndcg_sum / num_queries

def extract_causal_features(e, y_d, c_d, a_d, in_d, z_t, x_t, p_i, hub=True):
  '''
  Генерирует плоские признаки для финального CatBoost-ранжирования при холодном старте.
  e (np.ndarray): Названия ребер (строковые ID).
  y_d (dict): Года.
  c_d (dict): Словарь концептов.
  a_d (dict): Словарь авторов.
  in_d (dict): Входящие степени статей.
  z_t (torch.Tensor): Эмбеддинги FastRP соавторства.
  x_t (torch.Tensor): Текстовые эмбеддинги.
  p_i (dict): Индексная мапа.
  hub (bool, по умолчанию True): Флаг использования сетевой авторитетности («хабовости») целевой статьи в качестве обучающего признака.
  '''

  N = len(e); B = 8192; F = 390
  res = np.empty((N, F), dtype=np.float32)
  s_w = {'artificial', 'intelligence', 'machine', 'learning', 'computer', 'science'}
  print(f"[{time.strftime('%H:%M:%S')}] alloc {N}x{F}: {res.nbytes / 1024**2:.2f} MB")

  for i in range(0, N, B):
      e_idx = min(i + B, N); b_e = e[i:e_idx]
      h_b = np.zeros((len(b_e), 4), dtype=np.float32)
      u_i = np.empty(len(b_e), dtype=np.int32)
      v_i = np.empty(len(b_e), dtype=np.int32)

      for j, (u, v) in enumerate(b_e):
          u_i[j] = p_i[u]; v_i[j] = p_i[v]
          h_b[j, 0] = y_d[u] - y_d[v]

          au = a_d.get(u, set()); av = a_d.get(v, set())
          un_a = len(au | av)
          h_b[j, 1] = len(au & av) / un_a if un_a > 0 else 0.0

          cu = c_d.get(u, set()) - s_w; cv = c_d.get(v, set()) - s_w
          un_c = len(cu | cv)
          h_b[j, 2] = len(cu & cv) / un_c if un_c > 0 else 0.0

          if hub: h_b[j, 3] = np.log(in_d.get(v, 0) + 1.0)

      u_t = torch.from_numpy(u_i); v_t = torch.from_numpy(v_i)
      zt_u = z_t[u_t]; zt_v = z_t[v_t]
      xt_u = x_t[u_t]; xt_v = x_t[v_t]

      c_z = torch.cosine_similarity(zt_u, zt_v, dim=1).unsqueeze(1)
      h_z = zt_u * zt_v
      c_x = torch.cosine_similarity(xt_u, xt_v, dim=1).unsqueeze(1)
      h_x = xt_u * xt_v

      t_b = torch.cat([c_z, h_z, c_x, h_x], dim=1).numpy()
      res[i:e_idx] = np.hstack([h_b, t_b])

      del u_t, v_t, zt_u, zt_v, xt_u, xt_v, c_z, h_z, c_x, h_x, t_b, h_b
      if i % (B * 5) == 0:
          print(f"[{time.strftime('%H:%M:%S')}] feat: {e_idx}/{N}")
          gc.collect()

  return res

def evaluate_full_ranking_causal(m_cb, queries, gt_dict, y_dict, c_dict, a_dict, in_deg_dict, author_topology_embeddings_gpu, normalized_text_embeddings_gpu, text_embeddings_gpu, all_years_tensor_gpu, p_to_i, all_papers_numpy_array, use_hub=True, k_cand=100, k=10, batch_size=64):
  '''
  Оценивает CatBoostRanker при чистом холодном старте с агрегацией авторов по FastRP.
  use_hub (bool, по умолчанию True): Использовать ли предобученные хабовые фичи.
  k_cand (int, по умолчанию 100): Лимит семантического отбора кандидатов
  '''
  mrr_s = 0.0; rec_s = 0.0; hits_s = 0.0; ndcg_s = 0.0; n_v = normalized_text_embeddings_gpu.size(0); queries_evaluation_list = list(queries)
  stop_w = {'artificial', 'intelligence', 'machine', 'learning', 'computer', 'science'}

  for i in range(0, len(queries_evaluation_list), batch_size):
      batch_queries_subset = queries_evaluation_list[i:i+batch_size]; batch_length = len(batch_queries_subset)
      batch_indices_subset = [p_to_i[u] for u in batch_queries_subset]
      batch_years_tensor = torch.tensor([y_dict[u] for u in batch_queries_subset], dtype=torch.int32, device='cuda')

      similarity_matrix_gpu = torch.mm(normalized_text_embeddings_gpu[batch_indices_subset], normalized_text_embeddings_gpu.t())
      similarity_matrix_gpu[all_years_tensor_gpu.unsqueeze(0) > batch_years_tensor.unsqueeze(1)] = -2.0
      for idx_j, u_idx in enumerate(batch_indices_subset): similarity_matrix_gpu[idx_j, u_idx] = -2.0

      _, candidate_indices_gpu = torch.topk(similarity_matrix_gpu, min(k_cand, n_v), dim=1)
      candidate_indices_cpu = candidate_indices_gpu.cpu().numpy()

      query_indices_flat_gpu = torch.tensor(batch_indices_subset, device='cuda').repeat_interleave(k_cand)
      candidate_indices_flat_gpu = candidate_indices_gpu.flatten()

      query_topology_vectors = author_topology_embeddings_gpu[query_indices_flat_gpu]; candidate_topology_vectors = author_topology_embeddings_gpu[candidate_indices_flat_gpu]
      query_text_vectors = text_embeddings_gpu[query_indices_flat_gpu]; candidate_text_vectors = text_embeddings_gpu[candidate_indices_flat_gpu]

      cosine_similarity_topology = torch.cosine_similarity(query_topology_vectors, candidate_topology_vectors, dim=1).unsqueeze(1)
      hadamard_product_topology = query_topology_vectors * candidate_topology_vectors
      cosine_similarity_text = torch.cosine_similarity(query_text_vectors, candidate_text_vectors, dim=1).unsqueeze(1)
      hadamard_product_text = query_text_vectors * candidate_text_vectors

      gpu_computed_features_numpy = torch.cat([cosine_similarity_topology, hadamard_product_topology, cosine_similarity_text, hadamard_product_text], dim=1).cpu().numpy()

      candidate_features_batch_matrix = np.empty((batch_length * k_cand, 390), dtype=np.float32)
      candidate_features_batch_matrix[:, 4:] = gpu_computed_features_numpy

      idx = 0
      for j, u in enumerate(batch_queries_subset):
          au_u = a_dict.get(u, set()); c_u = c_dict.get(u, set()) - stop_w
          cands = all_papers_numpy_array[candidate_indices_cpu[j]]

          for v in cands:
              candidate_features_batch_matrix[idx, 0] = y_dict[u] - y_dict[v]
              au_v = a_dict.get(v, set()); un_a = len(au_u | au_v)
              candidate_features_batch_matrix[idx, 1] = len(au_u & au_v) / un_a if un_a > 0 else 0.0
              c_v = c_dict.get(v, set()) - stop_w; un_c = len(c_u | c_v)
              candidate_features_batch_matrix[idx, 2] = len(c_u & c_v) / un_c if un_c > 0 else 0.0
              candidate_features_batch_matrix[idx, 3] = np.log(in_deg_dict.get(v, 0) + 1.0) if use_hub else 0.0
              idx += 1

      predicted_scores_batch = m_cb.predict(candidate_features_batch_matrix)

      for j, u in enumerate(batch_queries_subset):
          t_v = gt_dict.get(u, set())
          if not t_v: continue
          cands = all_papers_numpy_array[candidate_indices_cpu[j]]
          sc = predicted_scores_batch[j * k_cand : (j + 1) * k_cand]
          sorted_rank_indices = np.argsort(sc)[::-1]; sorted_candidate_papers = cands[sorted_rank_indices]

          pos = np.where(np.isin(sorted_candidate_papers, list(t_v)))[0] + 1
          if len(pos) > 0:
              mrr_s += 1.0 / pos[0]
              t_k = set(sorted_candidate_papers[:k]); hits = len(t_v & t_k)
              rec_s += hits / len(t_v); hits_s += 1.0 if hits > 0 else 0.0
              dcg = np.sum([1.0 / math.log2(p + 1) for p in pos if p <= k])
              idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(t_v), k) + 1)])
              ndcg_s += dcg / idcg if idcg > 0 else 0.0

      del similarity_matrix_gpu, candidate_indices_gpu, candidate_indices_cpu, query_indices_flat_gpu, candidate_indices_flat_gpu, query_topology_vectors, candidate_topology_vectors, query_text_vectors, candidate_text_vectors
      del cosine_similarity_topology, hadamard_product_topology, cosine_similarity_text, hadamard_product_text, gpu_computed_features_numpy, candidate_features_batch_matrix, predicted_scores_batch; torch.cuda.empty_cache()

      if i % (batch_size * 20) == 0:
          mem = psutil.virtual_memory(); free_mb = mem.available / (1024 ** 2)
          print(f"[{time.strftime('%H:%M:%S')}] [RAM Check] Свободно: {free_mb:.2f} MB | Батч {i}/{len(queries_evaluation_list)}")
          if free_mb < 1000: gc.collect()

  n_q = len(queries)
  return mrr_s/n_q, rec_s/n_q, hits_s/n_q, ndcg_s/n_q

def train_and_eval_pipeline(d_p, x_text_tns, p_list_ordered, b_p):
  '''
  Выполняет полный цикл обучения и оценки CatBoostRanker на YetiRank.
  d_p (str): Путь к очищенному JSONL.
  x_text_tns (torch.Tensor): Матрица сжатых векторов статей (автоэнкодер).
  p_list_ordered (list): Упорядоченный список ID статей.
  b_p (str): Рабочая папка проекта
  '''
  G, G_m, tr_n, vl_n, te_n, y_dict, c_dict, a_dict, in_deg_dict = build_hetero_graph_and_split(d_p)
  topo_dict = compute_fastrp_hetero(G_m, dim=128)

  p_to_i = {p: i for i, p in enumerate(p_list_ordered)}
  z_topo = aggregate_author_topology(p_list_ordered, a_dict, topo_dict, dim=128)

  tr_e, tr_y = generate_causal_negatives(tr_n, G, y_dict)
  vl_e, vl_y = generate_causal_negatives(vl_n, G, y_dict)

  res_list = []
  for use_hub in [True, False]:
      exp_n = f"CatBoost_Hub_{use_hub}_{time.strftime('%H%M')}"
      c_d = os.path.join(b_p, 'experiments', exp_n); os.makedirs(c_d, exist_ok=True)
      m_p = os.path.join(c_d, 'model.cbm')

      tr_X = extract_causal_features(tr_e, y_dict, c_dict, a_dict, in_deg_dict, z_topo, x_text_tns, p_to_i, use_hub)
      vl_X = extract_causal_features(vl_e, y_dict, c_dict, a_dict, in_deg_dict, z_topo, x_text_tns, p_to_i, use_hub)

      tr_g = np.array([e[0] for e in tr_e])
      idx = np.argsort(tr_g); tr_X = tr_X[idx]; tr_y = tr_y[idx]; tr_g = tr_g[idx]

      vl_g = np.array([e[0] for e in vl_e])
      idx_v = np.argsort(vl_g); vl_X = vl_X[idx_v]; vl_y = vl_y[idx_v]; vl_g = vl_g[idx_v]

      tr_pool = cb.Pool(data=tr_X, label=tr_y, group_id=tr_g)
      vl_pool = cb.Pool(data=vl_X, label=vl_y, group_id=vl_g)

      print(f"\n{'='*50}\n[{time.strftime('%H:%M:%S')}] СТАРТ ТРЕНИРОВКИ: {exp_n}\n{'='*50}")
      m_cb = cb.CatBoostRanker(
          iterations=1500, depth=6, l2_leaf_reg=3.0, loss_function='YetiRank',
          task_type="GPU", early_stopping_rounds=50, verbose=False
      )

      if os.path.exists(m_p):
          m_cb.load_model(m_p)
          print(f"[{time.strftime('%H:%M:%S')}] Восстановлено из памяти: {m_p}")
      else:
          m_cb.fit(tr_pool, eval_set=vl_pool, callbacks=[cb_log(50)])
          m_cb.save_model(m_p)

          ev_res = m_cb.get_evals_result(); m_k = list(ev_res['validation'].keys())[0]
          plt.figure(figsize=(8, 4))
          if 'learn' in ev_res and m_k in ev_res['learn']: plt.plot(ev_res['learn'][m_k], label='Обучение', color='royalblue')
          plt.plot(ev_res['validation'][m_k], label='Валидация', color='crimson')
          plt.title(f"График обучения {exp_n}"); plt.xlabel("Эпоха"); plt.ylabel(m_k); plt.legend(); plt.grid(True, alpha=0.3)
          plt.tight_layout(); plt.savefig(os.path.join(c_d, 'loss_plot.png')); plt.close()

      gt_dict = {u: set(v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites') for u in te_n}
      mrr, rec, hits, ndcg = evaluate_full_ranking_causal(m_cb, te_n, gt_dict, y_dict, c_dict, a_dict, in_deg_dict, z_topo, x_text_tns, p_to_i, p_list_ordered, use_hub)

      res_list.append({"Эксп": exp_n, "Hubs": use_hub, "mrr": mrr, "rec10": rec, "hits10": hits, "ndcg10": ndcg})
      del m_cb, tr_X, vl_X, tr_pool, vl_pool; gc.collect()

  df_res = pd.DataFrame(res_list).sort_values("ndcg10", ascending=False)
  display(df_res.style.background_gradient(cmap='RdYlGn', subset=['mrr', 'rec10', 'hits10', 'ndcg10']))

def evaluate_hetero_full_graph(enc, pred, hd, te_n, gt_d, y_d, p_l, p_to_i, k=10, b_sz=2):
  '''
  Оценивает качество ранжирования гетерогенной PyTorch Geometric модели.
  enc (HeteroEncoder): Обученный двудольный энкодер.
  pred (LinkCausalPredictor): Классификатор ребер.
  hd (HeteroData): Графовая гетерогенная структура PyTorch Geometric
  '''
  enc.eval(); pred.eval(); mrr_s = 0.0; rec_s = 0.0; hits_s = 0.0; ndcg_s = 0.0; n_v = len(p_l); p_l_np = np.array(p_l)
  with torch.no_grad(): emb_d = enc(hd.x_dict, hd.edge_index_dict); z_p = emb_d['paper']
  y_t = torch.tensor([y_d[p] for p in p_l], dtype=torch.int32, device='cuda')
  for i in range(0, len(te_n), b_sz):
      b_q = te_n[i:i+b_sz]; b_len = len(b_q); b_idx = [p_to_i[u] for u in b_q]
      b_y = torch.tensor([y_d[u] for u in b_q], dtype=torch.int32, device='cuda')
      z_q = z_p[b_idx]; r_q = z_q.repeat_interleave(n_v, dim=0); r_c = z_p.repeat(b_len, 1)
      sc = pred(r_q, r_c).view(b_len, n_v); f_mask = y_t.unsqueeze(0) > b_y.unsqueeze(1); sc[f_mask] = -1e9
      for j, u_idx in enumerate(b_idx): sc[j, u_idx] = -1e9
      _, tk_idx_gpu = torch.topk(sc, k=100, dim=1); tk_idx_cpu = tk_idx_gpu.cpu().numpy()
      for j, u in enumerate(b_q):
          t_s = gt_d.get(u, set())
          if not t_s: continue
          preds = p_l_np[tk_idx_cpu[j]]; pos = np.where(np.isin(preds, list(t_s)))[0] + 1
          if len(pos) > 0:
              mrr_s += 1.0 / pos[0]; tk_p = set(preds[:k]); hits = len(t_s & tk_p)
              rec_s += hits / len(t_s); hits_s += 1.0 if hits > 0 else 0.0
              dcg = np.sum([1.0 / math.log2(p + 1) for p in pos if p <= k])
              idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(t_s), k) + 1)])
              ndcg_s += dcg / idcg if idcg > 0 else 0.0
      del sc, f_mask, tk_idx_gpu, r_q, r_c, b_y, z_q, b_idx, b_q; torch.cuda.empty_cache()
  return mrr_s/len(te_n), rec_s/len(te_n), hits_s/len(te_n), ndcg_s/len(te_n)

def run_hetero_gnn_experiments(b_p, p_s2, p_s35):
  '''
  Оркестрирует и запускает пайплайн обучения гетерогенных графовых нейросетей (HeteroGNN) на PyTorch Geometric.
  b_p (str): Базовый путь проекта.
  p_s2 (str): Путь к файлам Этапа 2 (структура графа).
  p_s35 (str): Путь к файлам Этапа 3.5 (топологические признаки).
  '''

  print(f"\n[{time.strftime('%H:%M:%S')}] СТАРТ ПАЙПЛАЙНА ОБУЧЕНИЯ ГЕТЕРОГЕННЫХ СЕТЕЙ")
  ld2 = k_rw('r', name2, p=p_s2); d_s2 = ld2['stage2_data.pkl']
  G = d_s2['G']; te_n = d_s2['te_n']; tr_n = d_s2['tr_n']; vl_n = d_s2['vl_n']; y_d = d_s2['y_dict']; del ld2, d_s2; gc.collect()
  ld35 = k_rw('r', name3_5, p=p_s35); d_s35 = ld35['stage3_5_data.pt']
  hd = d_s35['hd']; p_l = d_s35['p_list_ordered']; p_to_i = d_s35['p_to_i']; del ld35, d_s35; gc.collect()
  gt_d = {u: set(v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites' and v in p_to_i) for u in te_n}
  all_p_idx = np.array([p_to_i[p] for p in p_l]); y_arr = np.array([y_d[p] for p in p_l])
  tr_src = []; tr_dst = []
  for u in tr_n:
      for v in [v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites' and v in p_to_i]: tr_src.append(p_to_i[u]); tr_dst.append(p_to_i[v])
  vl_src = []; vl_dst = []
  for u in vl_n:
      for v in [v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites' and v in p_to_i]: vl_src.append(p_to_i[u]); vl_dst.append(p_to_i[v])
  exp_configs = [{"hid": 128, "lr": 0.001}, {"hid": 64, "lr": 0.003}]; res_list = []
  for cfg in exp_configs:
      exp_n = f"HeteroGNN_hid{cfg['hid']}_lr{cfg['lr']}"; c_d = os.path.join(b_p, 'experiments', exp_n); os.makedirs(c_d, exist_ok=True); chk_p = os.path.join(c_d, 'checkpoint.pt')
      enc = HeteroEncoder(256, cfg['hid']).to('cuda'); pred = LinkCausalPredictor(cfg['hid']).to('cuda'); opt = torch.optim.AdamW(list(enc.parameters()) + list(pred.parameters()), lr=cfg['lr'], weight_decay=1e-4); hd_gpu = hd.to('cuda')
      b_loss = float('inf'); s_ep = 0; tr_losses = []; vl_losses = []
      if os.path.exists(chk_p):
          ck = torch.load(chk_p, map_location='cuda'); enc.load_state_dict(ck['enc']); pred.load_state_dict(ck['pred']); opt.load_state_dict(ck['opt']); s_ep = ck['epoch'] + 1; b_loss = ck['loss']; tr_losses = ck['tr_l']; vl_losses = ck['vl_l']
          print(f"[{time.strftime('%H:%M:%S')}] Восстановлено из памяти: {chk_p} на эпохе {s_ep}")
      es_cnt = 0
      for epoch in range(s_ep, 200):
          ep_start = time.time(); enc.train(); pred.train(); opt.zero_grad(); z_p = enc(hd_gpu.x_dict, hd_gpu.edge_index_dict)['paper']
          t_neg_dst = []
          for u_idx in tr_src:
              y_u = y_arr[u_idx]
              while True:
                  v_neg = np.random.choice(all_p_idx)
                  if y_arr[v_neg] <= y_u: t_neg_dst.append(v_neg); break
          t_neg_t = torch.tensor(t_neg_dst, device='cuda'); p_sc = pred(z_p[tr_src], z_p[tr_dst]); n_sc = pred(z_p[tr_src], z_p[t_neg_t])
          tr_loss = torch.nn.functional.binary_cross_entropy_with_logits(p_sc, torch.ones_like(p_sc)) + torch.nn.functional.binary_cross_entropy_with_logits(n_sc, torch.zeros_like(n_sc)); tr_loss.backward(); opt.step()
          enc.eval(); pred.eval()
          with torch.no_grad():
              z_p_vl = enc(hd_gpu.x_dict, hd_gpu.edge_index_dict)['paper']; v_neg_dst = []
              for u_idx in vl_src:
                  y_u = y_arr[u_idx]
                  while True:
                      v_neg = np.random.choice(all_p_idx)
                      if y_arr[v_neg] <= y_u: v_neg_dst.append(v_neg); break
              v_neg_t = torch.tensor(v_neg_dst, device='cuda'); vp_sc = pred(z_p_vl[vl_src], z_p_vl[vl_dst]); vn_sc = pred(z_p_vl[vl_src], z_p_vl[v_neg_t])
              vl_loss = torch.nn.functional.binary_cross_entropy_with_logits(vp_sc, torch.ones_like(vp_sc)) + torch.nn.functional.binary_cross_entropy_with_logits(vn_sc, torch.zeros_like(vn_sc))
          tr_losses.append(tr_loss.item()); vl_losses.append(vl_loss.item()); t_p = time.time() - ep_start; h = int(t_p // 3600); m = int((t_p % 3600) // 60); s = int(t_p % 60)
          print(f'[{h:02d}:{m:02d}:{s:02d}] epoch {epoch}\n L Train: {tr_loss.item():.4f}\n L Val: {vl_loss.item():.4f}\n Time: {t_p:.2f}')
          if epoch % 5 == 0: torch.save({'epoch': epoch, 'enc': enc.state_dict(), 'pred': pred.state_dict(), 'opt': opt.state_dict(), 'loss': vl_loss.item(), 'tr_l': tr_losses, 'vl_l': vl_losses}, chk_p)
          if vl_loss.item() < b_loss:
              b_loss = vl_loss.item(); es_cnt = 0; torch.save({'epoch': epoch, 'enc': enc.state_dict(), 'pred': pred.state_dict(), 'opt': opt.state_dict(), 'loss': b_loss, 'tr_l': tr_losses, 'vl_l': vl_losses}, os.path.join(c_d, 'best_model.pt'))
          else:
              es_cnt += 1
              if es_cnt >= 20: print(f"[{time.strftime('%H:%M:%S')}] Early Stopping на эпохе {epoch}"); break
      plt.figure(figsize=(8, 4)); plt.plot(tr_losses, color='royalblue', label='Train'); plt.plot(vl_losses, color='crimson', label='Val'); plt.title(f"{exp_n}"); plt.xlabel("Эпоха"); plt.ylabel("Loss"); plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig(os.path.join(c_d, 'loss_plot.png')); plt.close()

      torch.cuda.empty_cache(); gc.collect()
      best_ck = torch.load(os.path.join(c_d, 'best_model.pt'), map_location='cuda'); enc.load_state_dict(best_ck['enc']); pred.load_state_dict(best_ck['pred'])
      mrr, rec, hits, ndcg = evaluate_hetero_full_graph(enc, pred, hd_gpu, te_n, gt_d, y_d, p_l, p_to_i, b_sz=2)
      res_list.append({"Эксп": exp_n, "mrr": mrr, "rec10": rec, "hits10": hits, "ndcg10": ndcg})
      del enc, pred, opt, hd_gpu, best_ck, z_p, z_p_vl, t_neg_t, v_neg_t; gc.collect(); torch.cuda.empty_cache()
  df_res = pd.DataFrame(res_list).sort_values("ndcg10", ascending=False)
  display(df_res.style.background_gradient(cmap='RdYlGn', subset=['mrr', 'rec10', 'hits10', 'ndcg10']))

def run_nn_experiments(b_p, name2, name3b, name3_5):
  '''
  Запускает масштабную сетку экспериментов по обучению гомогенных GNN-энкодеров и различных предикторов связи.
  b_p (str): Базовый путь проекта.
  name2 (str): Идентификатор датасета Этапа 2 на Kaggle.
  name3b (str): Идентификатор датасета Этапа 3b на Kaggle.
  name3_5 (str): Идентификатор датасета Этапа 3.5 на Kaggle.
  '''
  print(f"[{time.strftime('%H:%M:%S')}] СТАРТ СЕТКИ ЭКСПЕРИМЕНТОВ НЕЙРОСЕТЕЙ C COLD-START ТОПОЛОГИЕЙ")
  ld2 = k_rw('r', name2, p=get_p(name2)); d_s2 = ld2['stage2_data.pkl']
  G = d_s2['G']; G_m = d_s2['G_m']; tr_n = d_s2['tr_n']; vl_n = d_s2['vl_n']; te_n = d_s2['te_n']
  y_d = d_s2['y_dict']; c_d = d_s2['c_dict']; a_d = d_s2['a_dict']; in_deg = d_s2['in_deg']
  del ld2; gc.collect()

  ld3b = k_rw('r', name3b, p=get_p(name3b)); x_text = ld3b['stage3b_tensors.pt']['x_f'][:, :256].clone(); del ld3b; gc.collect()
  ld35 = k_rw('r', name3_5, p=get_p(name3_5)); d_s35 = ld35['stage3_5_data.pt']; p_l = d_s35['p_list_ordered']; p_i = d_s35['p_to_i']
  z_topo = d_s35['z_topo'].clone(); del ld35; gc.collect()

  n_v = len(p_l); gt_d = {u: set(v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites' and v in p_i) for u in te_n}
  p_l_np = np.array(p_l); y_arr = np.array([y_d[p] for p in p_l], dtype=np.int32)
  in_deg_arr = np.array([np.log(in_deg.get(p, 0) + 1.0) for p in p_l], dtype=np.float32)

  a_set = sorted(list(set(a for p in p_l for a in a_d.get(p, set())))); a_to_idx = {a: i for i, a in enumerate(a_set)}
  r_a, c_a = [], []
  for p in p_l:
      pidx = p_i[p]
      for a in a_d.get(p, set()): r_a.append(pidx); c_a.append(a_to_idx[a])
  M_A = sp.csr_matrix((np.ones(len(r_a), dtype=np.float32), (r_a, c_a)), shape=(n_v, len(a_set)))
  deg_A = np.array(M_A.sum(axis=1)).flatten()

  s_w = {'artificial', 'intelligence', 'machine', 'learning', 'computer', 'science'}
  c_set = sorted(list(set(c for p in p_l for c in c_d.get(p, set()) if c not in s_w))); c_to_idx = {c: i for i, c in enumerate(c_set)}
  r_c, c_c = [], []
  for p in p_l:
      pidx = p_i[p]
      for c in c_d.get(p, set()):
          if c in c_to_idx: r_c.append(pidx); c_c.append(c_to_idx[c])
  M_C = sp.csr_matrix((np.ones(len(r_c), dtype=np.float32), (r_c, c_c)), shape=(n_v, len(c_set)))
  deg_C = np.array(M_C.sum(axis=1)).flatten()

  tr_src, tr_dst = [], []
  for u in tr_n:
      for v in [v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites' and v in p_i]: tr_src.append(p_i[u]); tr_dst.append(p_i[v])
  vl_src, vl_dst = [], []
  for u in vl_n:
      for v in [v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites' and v in p_i]: vl_src.append(p_i[u]); vl_dst.append(p_i[v])

  c_src, c_dst = [], []
  for u, v, d in G_m.edges(data=True):
      if d.get('type') == 'cites' and u in p_i and v in p_i: c_src.append(p_i[u]); c_dst.append(p_i[v])
  homo_edge_index = torch.tensor([c_src, c_dst], dtype=torch.long, device='cuda')

  def get_heuristics_cpu(b_u, b_v, is_full=False):
      B = len(b_u)
      if is_full:
          dt = y_arr[b_u].reshape(-1, 1) - y_arr.reshape(1, -1)
          hub = np.repeat(in_deg_arr.reshape(1, -1), B, axis=0)
          I_A = M_A[b_u].dot(M_A.T).toarray()
          U_A = deg_A[b_u].reshape(-1, 1) + deg_A.reshape(1, -1) - I_A; au_o = np.where(U_A > 0, I_A / U_A, 0.0)
          I_C = M_C[b_u].dot(M_C.T).toarray()
          U_C = deg_C[b_u].reshape(-1, 1) + deg_C.reshape(1, -1) - I_C; cn_o = np.where(U_C > 0, I_C / U_C, 0.0)
      else:
          dt = (y_arr[b_u] - y_arr[b_v]).reshape(-1, 1); hub = in_deg_arr[b_v].reshape(-1, 1)
          I_A = M_A[b_u].multiply(M_A[b_v]).sum(axis=1).A1.reshape(-1, 1)
          U_A = deg_A[b_u].reshape(-1, 1) + deg_A[b_v].reshape(-1, 1) - I_A; au_o = np.where(U_A > 0, I_A / U_A, 0.0)
          I_C = M_C[b_u].multiply(M_C[b_v]).sum(axis=1).A1.reshape(-1, 1)
          U_C = deg_C[b_u].reshape(-1, 1) + deg_C[b_v].reshape(-1, 1) - I_C; cn_o = np.where(U_C > 0, I_C / U_C, 0.0)
      return torch.from_numpy(np.stack([dt, au_o, cn_o, hub], axis=-1).astype(np.float32)).to('cuda')

  g_types = ['none', 'lightgcn', 'sage', 'dirgcn']; p_types = ['standard', 'buddy', 'ncn']; hub_options = [True, False]; res_list = []
  all_p_idx = torch.arange(n_v, device='cuda'); x_full_gpu = torch.cat([x_text, z_topo], dim=1).to('cuda').float()
  empty_edge_index = torch.empty((2, 0), dtype=torch.long, device='cuda')

  for g_t, p_t, use_hub in itertools.product(g_types, p_types, hub_options):
      exp_n = f"NN_{g_t}_{p_t}_hub_{use_hub}"; print(f"\n{'='*70}\nЗапуск эксперимента: {exp_n}\n{'='*70}")
      c_d = os.path.join(b_p, 'experiments', exp_n); os.makedirs(c_d, exist_ok=True); chk_p = os.path.join(c_d, 'checkpoint.pt')

      n_a_dim = 4 if use_hub else 3
      m = GM(n_f=384, n_v=n_v, n_a=n_a_dim, o_d=128, g_type=g_t, p_type=p_t).to('cuda')
      opt = torch.optim.AdamW(m.parameters(), lr=0.001, weight_decay=1e-4); crit = torch.nn.BCELoss()

      st_epoch = 0; b_loss = float('inf'); es_cnt = 0; tr_losses = []; vl_losses = []

      if os.path.exists(chk_p):
          ck = torch.load(chk_p, map_location='cuda')
          m.load_state_dict(ck['m']); opt.load_state_dict(ck['opt']); st_epoch = ck['epoch'] + 1
          b_loss = ck['loss']; es_cnt = ck['es_cnt']; tr_losses = ck['tr_l']; vl_losses = ck['vl_l']
          print(f"Восстановлено состояние из чекпоинта. Возобновление с эпохи {st_epoch}")

      b_sz = 4096
      for epoch in range(st_epoch, 10):
          t0 = time.time(); m.train()
          t_neg = []
          for u_idx in tr_src:
              y_u = y_arr[u_idx]
              while True:
                  v_neg_idx = np.random.choice(all_p_idx.cpu().numpy())
                  if y_arr[v_neg_idx] <= y_u: t_neg.append(v_neg_idx); break

          idx = torch.randperm(len(tr_src)); ep_loss = 0.0
          for i in range(0, len(tr_src), b_sz):
              b_i = idx[i:i+b_sz]; b_u = np.array(tr_src)[b_i]; b_v_p = np.array(tr_dst)[b_i]; b_v_n = np.array(t_neg)[b_i]
              opt.zero_grad()

              z_connected = m.enc(x_full_gpu, all_p_idx, homo_edge_index)
              z_isolated = m.enc(x_full_gpu, all_p_idx, empty_edge_index)

              h_p = get_heuristics_cpu(b_u, b_v_p, is_full=False).view(len(b_u), 4)
              h_n = get_heuristics_cpu(b_u, b_v_n, is_full=False).view(len(b_u), 4)
              if not use_hub: h_p = h_p[:, :3]; h_n = h_n[:, :3]

              p_sc = m.c(z_isolated[b_u], z_connected[b_v_p], h_p).squeeze()
              n_sc = m.c(z_isolated[b_u], z_connected[b_v_n], h_n).squeeze()

              l = crit(p_sc, torch.ones_like(p_sc)) + crit(n_sc, torch.zeros_like(n_sc)); l.backward(); opt.step()
              ep_loss += l.item() * len(b_u)

          m.eval()
          with torch.no_grad():
              z_conn_vl = m.enc(x_full_gpu, all_p_idx, homo_edge_index)
              z_isol_vl = m.enc(x_full_gpu, all_p_idx, empty_edge_index)
              v_neg_vl = []
              for u_idx in vl_src:
                  y_u = y_arr[u_idx]
                  while True:
                      v_neg_idx = np.random.choice(all_p_idx.cpu().numpy())
                      if y_arr[v_neg_idx] <= y_u: v_neg_vl.append(v_neg_idx); break
              b_u_v = np.array(vl_src); b_v_p_v = np.array(vl_dst); b_v_n_v = np.array(v_neg_vl)
              h_p_v = get_heuristics_cpu(b_u_v, b_v_p_v, is_full=False).view(len(b_u_v), 4)
              h_n_v = get_heuristics_cpu(b_u_v, b_v_n_v, is_full=False).view(len(b_u_v), 4)
              if not use_hub: h_p_v = h_p_v[:, :3]; h_n_v = h_n_v[:, :3]

              vp_sc = m.c(z_isol_vl[b_u_v], z_conn_vl[b_v_p_v], h_p_v).squeeze()
              vn_sc = m.c(z_isol_vl[b_u_v], z_conn_vl[b_v_n_v], h_n_v).squeeze()
              vl_loss = crit(vp_sc, torch.ones_like(vp_sc)) + crit(vn_sc, torch.zeros_like(vn_sc))

          tr_losses.append(ep_loss / len(tr_src)); vl_losses.append(vl_loss.item())
          t_p = time.time() - t0; h_t = int(t_p // 3600); m_t = int((t_p % 3600) // 60); s_t = int(t_p % 60)
          print(f'[{h_t:02d}:{m_t:02d}:{s_t:02d}] epoch {epoch}\n L Train: {tr_losses[-1]:.4f}\n L Val: {vl_losses[-1]:.4f}\n Time: {t_p:.2f}')

          torch.save({'epoch': epoch, 'm': m.state_dict(), 'opt': opt.state_dict(), 'loss': vl_loss.item(), 'es_cnt': es_cnt, 'tr_l': tr_losses, 'vl_l': vl_losses}, chk_p)

          if vl_loss.item() < b_loss:
              b_loss = vl_loss.item(); es_cnt = 0
              torch.save({'epoch': epoch, 'm': m.state_dict(), 'opt': opt.state_dict(), 'loss': b_loss, 'es_cnt': es_cnt, 'tr_l': tr_losses, 'vl_l': vl_losses}, os.path.join(c_d, 'best_model.pt'))
          else:
              es_cnt += 1
              if es_cnt >= 3: print(f"Early Stopping на эпохе {epoch}"); break

      plt.figure(figsize=(8, 4)); plt.plot(tr_losses, color='royalblue', label='Обучение'); plt.plot(vl_losses, color='crimson', label='Валидация')
      plt.title(f"Кривая обучения: {exp_n}"); plt.xlabel("Эпоха"); plt.ylabel("Loss"); plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
      plt.savefig(os.path.join(c_d, 'loss_plot.png')); plt.show(); plt.close()

      torch.cuda.empty_cache(); gc.collect()
      best_ck = torch.load(os.path.join(c_d, 'best_model.pt'), map_location='cuda')
      m.load_state_dict(best_ck['m']); m.eval()
      mrr_s = rec_s = hits_s = ndcg_s = 0.0

      with torch.no_grad():
          z_conn_te = m.enc(x_full_gpu, all_p_idx, homo_edge_index)
          z_isol_te = m.enc(x_full_gpu, all_p_idx, empty_edge_index)
          y_t_gpu = torch.from_numpy(y_arr).to('cuda')

          for u in te_n:
              u_idx = p_i[u]; t_s = gt_d.get(u, set())
              if not t_s: continue

              h_full = get_heuristics_cpu([u_idx], None, is_full=True).view(n_v, 4)
              if not use_hub: h_full = h_full[:, :3]

              sc = m.c(z_isol_te[u_idx].expand(n_v, -1), z_conn_te, h_full).squeeze()
              sc[y_t_gpu > y_d[u]] = -1e9; sc[u_idx] = -1e9

              _, tk_idx = torch.topk(sc, k=100); preds = p_l_np[tk_idx.cpu().numpy()]
              pos = np.where(np.isin(preds, list(t_s)))[0] + 1
              if len(pos) > 0:
                  mrr_s += 1.0 / pos[0]; tk_p = set(preds[:10]); hits = len(t_s & tk_p)
                  rec_s += hits / len(t_s); hits_s += 1.0 if hits > 0 else 0.0
                  dcg = np.sum([1.0 / math.log2(p + 1) for p in pos if p <= 10])
                  idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(t_s), 10) + 1)])
                  ndcg_s += dcg / idcg if idcg > 0 else 0.0

      n_q = len([u for u in te_n if gt_d.get(u)])
      res_dict = {"Эксперимент": exp_n, "mrr": mrr_s/n_q, "rec10": rec_s/n_q, "hits10": hits_s/n_q, "ndcg10": ndcg_s/n_q}
      res_list.append(res_dict); print(f"Метрики Test для {exp_n}: MRR={res_dict['mrr']:.4f}, NDCG@10={res_dict['ndcg10']:.4f}")

      with open(os.path.join(c_d, 'metrics.json'), 'w', encoding='utf-8') as f_json: json.dump(res_dict, f_json, ensure_ascii=False, indent=4)
      update_scoreboard(res_dict, exp_n, b_p)
      del m, opt; torch.cuda.empty_cache(); gc.collect()

  df_res = pd.DataFrame(res_list).sort_values("ndcg10", ascending=False)
  from IPython.display import display; display(df_res.style.background_gradient(cmap='RdYlGn', subset=['mrr', 'rec10', 'hits10', 'ndcg10']))


# Данные

In [ ]:
name1   = 'kvdep1/ai-v2-stage1-raw-graph'
name2   = 'kvdep1/ai-v6-stage2-hetero-split'
name3a  = 'kvdep1/ai-v6-stage3a-scibert'
name3b  = 'kvdep1/ai-v6-stage3b-autoencoder'
name3_5 = 'kvdep1/ai-v6-stage3-5-topology'
name4   = 'kvdep1/ai-v6-stage4-final-touches'
name5   = 'kvdep1/ai-v6-stage5-gnn-aligner'

p_s1    = get_p(name1)
p_s2    = get_p(name2)
p_s3a   = get_p(name3a)
p_s3b   = get_p(name3b)
p_s35   = get_p(name3_5)
p_s4    = get_p(name4)

d_dv = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Создание графов

### test

In [ ]:
u = "https://api.openalex.org/works"
pm = {"filter": "has_abstract:true", "per-page": 1}
p = rq.get(u, params=pm, timeout=15)
if p.status_code != 200: raise RuntimeError(f"API Error {p.status_code}")

j = p.json(); works = j.get('results', [])
if not works: raise RuntimeError("API returned empty array")

w = works[0]; ai = w.get('abstract_inverted_index')
if not ai: raise RuntimeError("Missing abstract_inverted_index")

max_idx = max([pos for positions in ai.values() if isinstance(positions, list) for pos in positions], default=-1)
if max_idx == -1: raise RuntimeError("Empty or invalid abstract_inverted_index structure")

ab_arr = [""] * (max_idx + 1)
for word, positions in ai.items():
    if isinstance(positions, list):
        for pos in positions: ab_arr[pos] = word

ab = " ".join(ab_arr).strip()

extracted_node = {
    'i': w['id'],
    'y': w.get('publication_year'),
    'ti': w.get('title'),
    'na': len(w.get('authorships', [])),
    'c': w.get('cited_by_count', 0),
    'ab': ab,
    'au': [a['author']['id'] for a in w.get('authorships', []) if 'author' in a],
    'ct': " ".join([x['display_name'] for x in w.get('concepts', [])]),
    'rf': w.get('referenced_works', [])
}

print(json.dumps(extracted_node, indent=2, ensure_ascii=False))

### Сбор сырых данных

In [ ]:
f_r = False
p_s1 = get_p(name1)
if not f_r:
    try:
        ld = k_rw('r', name1, p=p_s1)
        d_raw, g_raw = ld['d.jsonl'], ld['g.pkl']
        import time; print(f"[{time.strftime('%H:%M:%S')}] load")
    except: f_r = True

if f_r:
    import time; import os; print(f"[{time.strftime('%H:%M:%S')}] gen")
    c_i = ['C119857082']

    if not os.path.exists('raw_data.jsonl'):
        print("Скачивание данных...")
        d_raw = f_ai(2_050_000, 10, c_ids=c_i, p_file='raw_data.jsonl')
    else:
        print("Файл raw_data.jsonl найден, пропускаем скачивание...")
        d_raw = 'raw_data.jsonl'

    g_raw = b_raw_g(d_raw)

    k_rw('w', name1, {'d.jsonl': d_raw, 'g.pkl': g_raw}, t='AI Stage 1 Raw Graph', p=p_s1)

### Прунинг и анализ

In [ ]:
f_r = False
p_s2 = get_p(name2);
try:
    ld = k_rw('r', name2, p=p_s2)
    if 'stage2_data.pkl' not in ld: f_r = True
    del ld; import gc; gc.collect()
except: f_r = True

if f_r: run_stage2(p_s2); import gc; gc.collect()
sanity_check_stage2(p_s2); import gc; gc.collect()

### Эмбеддинги

#### Scibert

In [ ]:
f_r_3a = False
p_s3a = get_p(name3a)
d_dv = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not f_r_3a:
    try:
        ld = k_rw('r', name3a, p=p_s3a); x_raw = ld['x_raw.pt']
    except: f_r_3a = True

if f_r_3a:
    p_s1 = get_p(name1); ld1 = k_rw('r', name1, p=p_s1); raw_p = ld1['d.jsonl']
    p_s2 = get_p(name2); ld2 = k_rw('r', name2, p=p_s2); d_s2 = ld2['stage2_data.pkl']
    G = d_s2['G']; c_dict = d_s2['c_dict']; y_dict = d_s2['y_dict']
    p_list_ordered = list(y_dict.keys())

    def b_sci_emb(G, raw_p, c_dict, p_list_ordered, d_dv):
        print(f"[{time.strftime('%H:%M:%S')}] Извлечение Title, Abstract, Concepts")
        ti_d = {}; ab_d = {}; p_set = set(p_list_ordered)
        with open(raw_p, 'r', encoding='utf-8') as f:
            for ln in f:
                if not ln.strip(): continue
                x = json.loads(ln)
                pid = x.get('i')
                if pid in p_set:
                    ti_d[pid] = x.get('ti', '') or ''
                    ab_d[pid] = x.get('ab', '') or ''

        ti_l = [ti_d.get(n, "") for n in p_list_ordered]
        ab_l = [ab_d.get(n, "") for n in p_list_ordered]
        ct_l = [" ".join(c_dict.get(n, set())) for n in p_list_ordered]
        del ti_d, ab_d; gc.collect()

        print(f"[{time.strftime('%H:%M:%S')}] SciBERT Encoding")
        st = ST('pritamdeka/S-SciBERT-snli-multinli-stsb').to(d_dv)
        with torch.autocast(device_type=d_dv.type, dtype=torch.float16):
            e_ti = st.encode(ti_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)
            e_ab = st.encode(ab_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)
            e_ct = st.encode(ct_l, batch_size=128, show_progress_bar=True, convert_to_tensor=True, device=d_dv)
        del st, ti_l, ab_l, ct_l; torch.cuda.empty_cache(); gc.collect()
        x_raw = torch.cat([e_ti, e_ab, e_ct], dim=1).half().cpu()
        return x_raw

    x_raw = b_sci_emb(G, raw_p, c_dict, p_list_ordered, d_dv)
    k_rw('w', name3a, {'x_raw.pt': x_raw}, t='Raw SciBERT Embs V5', p=p_s3a)
    del ld1, ld2, d_s2, G, c_dict, y_dict, x_raw; gc.collect()

#### Autoencoder

In [ ]:
f_r_3b = False
p_s3a = get_p(name3a)
p_s3b = get_p(name3b)

if not f_r_3b:
    try:
        ld = k_rw('r', name3b, p=p_s3b); tns_d = ld['stage3b_tensors.pt']
    except: f_r_3b = True

if f_r_3b:
    import time; import gc; import torch; import os
    print(f"[{time.strftime('%H:%M:%S')}] Очистка ОЗУ перед загрузкой 3B...")
    gc.collect(); torch.cuda.empty_cache()

    p_s2 = get_p(name2); ld2 = k_rw('r', name2, p=p_s2)
    y_dict = ld2['stage2_data.pkl']['y_dict']
    p_list_ordered = list(y_dict.keys())
    del ld2; gc.collect()

    x_raw_p = os.path.join(p_s3a, 'x_raw.pt')
    if os.path.exists(x_raw_p):
        x_raw = torch.load(x_raw_p, map_location='cpu', weights_only=False)
    else:
        ld3a = k_rw('r', name3a, p=p_s3a)
        x_raw = ld3a.get('x_raw.pt')

    ae = TextAE(i_d=2304, h_d=1024, b_d=256)
    x_cmp = train_ae(ae, x_raw, d_dv, ep=40, b_sz=2048, lr=1e-3)

    del x_raw; gc.collect(); torch.cuda.empty_cache()

    y_l = [y_dict[n] for n in p_list_ordered]
    yr_t = torch.tensor(y_l).float().view(-1, 1)
    yr_t = (yr_t - yr_t.mean()) / (yr_t.std() + 1e-8)
    x_f_red = torch.cat([x_cmp, yr_t], dim=1)

    tns_d = {'x_f': x_f_red, 'p_list_ordered': p_list_ordered}
    k_rw('w', name3b, {'stage3b_tensors.pt': tns_d}, t='Autoencoder Embs V5', p=p_s3b)

    del y_dict, x_cmp, yr_t, x_f_red, tns_d; gc.collect()
    print(f"[{time.strftime('%H:%M:%S')}] ЭТАП 3B УСПЕШНО ЗАВЕРШЕН")

### Топологические эмбеддинги и разбиение

In [ ]:
f_r = False
p_s35 = get_p(name3_5);
try:
    ld = k_rw('r', name3_5, p=p_s35)
    if 'stage3_5_data.pt' not in ld: f_r = True
    del ld; gc.collect()
except: f_r = True

if f_r: run_stage3_5(p_s35); gc.collect()
sanity_check_stage35(p_s35); gc.collect()

## Загрузка графов

In [ ]:
p_s4 = get_p(name4);
f_r = False
try:
    ld = k_rw('r', name4, p=p_s4)
    if 'stage4_tensors.pt' not in ld: f_r = True
    del ld; gc.collect()
except: f_r = True

if f_r: run_stage4(p_s4); gc.collect()
sanity_check_stage4(p_s4); gc.collect()

# Работа

## Анализ данных

In [ ]:
print(f"[{time.strftime('%H:%M:%S')}] Загрузка данных для расширенного анализа...")
ld = k_rw('r', name2, p='./stage2_hetero_split')
d_s2 = ld['stage2_data.pkl']

G = d_s2['G']            # Полный граф
G_m = d_s2['G_m']        # Маскированный граф (содержит только Train ребра цитирования)
tr_n = d_s2['tr_n']      # Вершины Train
vl_n = d_s2['vl_n']      # Вершины Val
te_n = d_s2['te_n']      # Вершины Test
y_dict = d_s2['y_dict']  # Словарь {id_статьи: год}

del ld; gc.collect()

print("="*50)
print("1. Количество вершин в выборках:")
print(f"Train: {len(tr_n)}")
print(f"Val:   {len(vl_n)}")
print(f"Test:  {len(te_n)}")

train_cites_edges = sum(1 for u, v, d in G_m.edges(data=True) if d.get('type') == 'cites')
print("\n2. Количество ребер цитирования (cites) в обучающем графе (Train):")
print(f"Train edges: {train_cites_edges}")
print("="*50)

print(f"[{time.strftime('%H:%M:%S')}] Построение графика распределения вершин по годам...")
tr_years = collections.Counter([y_dict.get(n) for n in tr_n if y_dict.get(n) is not None])
vl_years = collections.Counter([y_dict.get(n) for n in vl_n if y_dict.get(n) is not None])
te_years = collections.Counter([y_dict.get(n) for n in te_n if y_dict.get(n) is not None])

all_years = sorted(list(set(tr_years.keys()) | set(vl_years.keys()) | set(te_years.keys())))
all_years = [y for y in all_years if y > 1950]

tr_counts = [tr_years.get(y, 0) for y in all_years]
vl_counts = [vl_years.get(y, 0) for y in all_years]
te_counts = [te_years.get(y, 0) for y in all_years]

plt.figure(figsize=(16, 6))
bar_width = 0.25
r1 = np.arange(len(all_years))
r2 = [x + bar_width for x in r1]
r3 = [x + bar_width for x in r2]

plt.bar(r1, tr_counts, color='royalblue', width=bar_width, label='Train')
plt.bar(r2, vl_counts, color='orange', width=bar_width, label='Val')
plt.bar(r3, te_counts, color='crimson', width=bar_width, label='Test')

plt.xlabel('Год публикации', fontsize=12, fontweight='bold')
plt.ylabel('Количество статей (вершин)', fontsize=12, fontweight='bold')
plt.xticks([r + bar_width for r in range(len(all_years))], all_years, rotation=45)
plt.title('Распределение количества вершин по годам для Train, Val, Test', fontsize=15)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"[{time.strftime('%H:%M:%S')}] Построение Heatmap цитирований по годам...")
year_edges = collections.defaultdict(int)

for u, v, d in G.edges(data=True):
    if d.get('type') == 'cites':
        yu = y_dict.get(u)
        yv = y_dict.get(v)
        if yu is not None and yv is not None and yu > 1950 and yv > 1950:
            year_edges[(yu, yv)] += 1

if year_edges:
    valid_years = sorted(list(set([y for u, v in year_edges.keys() for y in (u, v)])))
    heatmap_data = pd.DataFrame(0, index=valid_years, columns=valid_years)

    for (yu, yv), count in year_edges.items():
        heatmap_data.at[yu, yv] = count

    plt.figure(figsize=(14, 12))

    from matplotlib.colors import LogNorm
    vmin = max(heatmap_data.values.min(), 1) # log(0)
    vmax = heatmap_data.values.max()

    sns.heatmap(heatmap_data, cmap='YlGnBu', norm=LogNorm(vmin=vmin, vmax=vmax),
                annot=True, cbar_kws={'label': 'Количество ребер (Логарифмическая шкала)'})

    plt.title('Heatmap цитирований по годам\n(Откуда цитируют -> Кого цитируют)', fontsize=16)
    plt.xlabel('Год цитируемой статьи (Target / Куда указывает ребро)', fontsize=12, fontweight='bold')
    plt.ylabel('Год цитирующей статьи (Source / Откуда исходит ребро)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Недостаточно данных о годах для построения Heatmap.")

In [ ]:
print(f"[{time.strftime('%H:%M:%S')}] Загрузка Stage 2 для анализа...")
p_s2 = get_p(name2); ld = k_rw('r', name2, p=p_s2)
G = ld['stage2_data.pkl']['G']

p_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'paper']
in_d = {n: G.in_degree(n) for n in p_nodes}
out_d = {n: G.out_degree(n) for n in p_nodes}

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(list(in_d.values()), bins=50, color='royalblue', alpha=0.8)
plt.title('Распределение входящих степеней')
plt.yscale('log')

plt.subplot(1, 2, 2)
plt.hist(list(out_d.values()), bins=50, color='seagreen', alpha=0.8)
plt.title('Распределение исходящих степеней')
plt.yscale('log')
plt.tight_layout()
plt.show()

y_d = collections.defaultdict(list)
for n in p_nodes:
    y = G.nodes[n].get('y')
    if y and y > 0: y_d[y].append(in_d[n] + out_d[n])

y_k = sorted(y_d.keys()); y_v = [np.mean(y_d[y]) for y in y_k]

plt.figure(figsize=(12, 5))
plt.bar(y_k, y_v, color='crimson', alpha=0.8, edgecolor='black', linewidth=0.5)
plt.title('Средняя степень вершины по годам')
plt.xticks(y_k, rotation=45)
plt.grid(True, linestyle='--', alpha=0.6, axis='y')
plt.tight_layout()
plt.show()

c_g = nx.Graph()
c_g.add_nodes_from(p_nodes)
cites_e = [(u, v) for u, v, d in G.edges(data=True) if d.get('type') == 'cites']
c_g.add_edges_from(cites_e)

print(f"[{time.strftime('%H:%M:%S')}] Запуск алгоритма Лувена...")
cm = nx.community.louvain_communities(c_g, seed=42)
cd = {n: i for i, c in enumerate(cm) for n in c}

r = a_c(G, cd); v_c(c_g, cd)

## Модель

#### baseline

In [ ]:
ld4 = k_rw('r', name4, p='./stage4_final'); tts = ld4['stage4_tensors.pt']
tr_p_e = tts['tr_p_e']; x_f = tts['x_f']
te_e = tts['te_e']; te_a = tts['te_a']; te_y = tts['te_y']

ld2 = k_rw('r', name2, p='./stage2_pruned'); g_f = ld2['g.pkl']
n_v = x_f.size(0)
au_d = {i: set(d.get('au', [])) for i, (n, d) in enumerate(g_f.nodes(data=True))}
y_a = torch.tensor([d.get('y', 0) for n, d in g_f.nodes(data=True)])

print(f"[{time.strftime('%H:%M:%S')}] BASELINE calc")
y_p_b = g_s_b(te_e, te_a, x_f)
y_p_b = (y_p_b - y_p_b.min()) / (y_p_b.max() - y_p_b.min() + 1e-8)

o_th, a_roc, a_pr, f1 = eval_model(te_y.numpy(), y_p_b.numpy(), "baseline_plot.png")

print(f"[{time.strftime('%H:%M:%S')}] Full-ranking оценка BASELINE")
te_p_e = te_e[te_y == 1]
gt_d = collections.defaultdict(list)
for u, v in te_p_e.tolist(): gt_d[int(u)].append(int(v))
u_l = list(gt_d.keys()); mrr_s = 0.0; rec_s = 0.0; hits_s = 0.0; ndcg_s = 0.0; N_u = len(u_l)

a_sym, a_aa, m_ap, m_ap_T, a_deg = build_eval_matrices(tr_p_e, au_d, n_v, d_dv)

b_sz = 64
for i in range(0, N_u, b_sz):
    b_u = torch.tensor(u_l[i:i+b_sz], dtype=torch.long, device=d_dv); B = b_u.size(0)
    h = get_batch_heuristics(b_u, n_v, y_a.to(d_dv), a_sym, a_aa, m_ap, m_ap_T, a_deg, d_dv)

    z_u_ab = torch.nn.functional.normalize(x_f[b_u, :768].to(d_dv), p=2, dim=1)
    z_v_ab = torch.nn.functional.normalize(x_f[:, :768].to(d_dv), p=2, dim=1)
    c_ab = torch.mm(z_u_ab, z_v_ab.t())

    z_u_ct = torch.nn.functional.normalize(x_f[b_u, 768:1536].to(d_dv), p=2, dim=1)
    z_v_ct = torch.nn.functional.normalize(x_f[:, 768:1536].to(d_dv), p=2, dim=1)
    c_ct = torch.mm(z_u_ct, z_v_ct.t())

    z_u_tp = torch.nn.functional.normalize(x_f[b_u, 1537:1665].to(d_dv), p=2, dim=1)
    z_v_tp = torch.nn.functional.normalize(x_f[:, 1537:1665].to(d_dv), p=2, dim=1)
    c_tp = torch.mm(z_u_tp, z_v_tp.t())

    cn_min = h[:, :, 2].min(dim=1, keepdim=True)[0]
    cn_max = h[:, :, 2].max(dim=1, keepdim=True)[0]
    cn_n = (h[:, :, 2] - cn_min) / (cn_max - cn_min + 1e-8)
    sc = 0.2 * c_ab + 0.2 * c_ct + 0.3 * c_tp + 0.15 * h[:, :, 1] + 0.15 * cn_n

    m_dt2 = y_a.to(d_dv).unsqueeze(0) > y_a[b_u].to(d_dv).unsqueeze(1)
    m_sl = torch.arange(n_v, device=d_dv).unsqueeze(0) == b_u.unsqueeze(1)
    sc[m_dt2 | m_sl] = -float('inf')
    r = torch.argsort(sc, dim=1, descending=True).cpu().numpy()

    for j in range(B):
        u_i = u_l[i + j]; t_v = set(gt_d[u_i]); u_r = r[j]
        pos = np.where(np.isin(u_r, list(t_v)))[0] + 1
        if len(pos) == 0: continue
        mrr_s += 1.0 / pos[0]; t_k = set(u_r[:10]); hits = len(t_v & t_k)
        rec_s += hits / len(t_v); hits_s += 1.0 if hits > 0 else 0.0
        dcg = np.sum([1.0 / math.log2(p + 1) for p in pos if p <= 10])
        idcg = np.sum([1.0 / math.log2(p + 1) for p in range(1, min(len(t_v), 10) + 1)])
        ndcg_s += dcg / idcg if idcg > 0 else 0.0

mrr, rec, hits, ndcg = mrr_s / N_u, rec_s / N_u, hits_s / N_u, ndcg_s / N_u
met = {"th": float(o_th), "roc": float(a_roc), "pr": float(a_pr), "f1": float(f1), "mrr": float(mrr), "rec10": float(rec), "hits10": float(hits), "ndcg10": float(ndcg), "cfg": "SciBERT + FastRP + Heuristics"}

exp_n = f"BASELINE_Topo_Heuristics_{time.strftime('%Y%m%d_%H%M')}"
n_exp_n = f"NDCG_{ndcg:.4f}-{exp_n}"
e_d = os.path.join(b_p, 'experiments', n_exp_n); os.makedirs(e_d, exist_ok=True)
shutil.copy("baseline_plot.png", os.path.join(e_d, "eval_plot.png"))
with open(os.path.join(e_d, 'metrics.json'), 'w', encoding='utf-8') as f: json.dump(met, f, ensure_ascii=False, indent=4)
update_scoreboard(met, n_exp_n, b_p)

#### GNN

In [ ]:
run_nn_experiments(b_p, name2, name3b, name3_5)

#### SEAL

#### GBDT

In [ ]:
p_s2 = get_p(name2); ld2 = k_rw('r', name2, p=p_s2); d_s2 = ld2['stage2_data.pkl']
G = d_s2['G']; te_n = d_s2['te_n']; y_dict = d_s2['y_dict']; c_dict = d_s2['c_dict']; a_dict = d_s2['a_dict']
del ld2, d_s2; gc.collect()

p_s3b = get_p(name3b); ld3b = k_rw('r', name3b, p=p_s3b); x_text_tns = ld3b['stage3b_tensors.pt']['x_f'][:, :256].clone(); del ld3b; gc.collect()
p_s35 = get_p(name3_5); ld35 = k_rw('r', name3_5, p=p_s35); z_topo = ld35['stage3_5_data.pt']['z_topo'].clone(); del ld35; gc.collect()

p_s4 = get_p(name4); ld4 = k_rw('r', name4, p=p_s4); d_s4 = ld4['stage4_tensors.pt']
tr_X_tns = d_s4['tr_X']; tr_y_tns = d_s4['tr_y']; tr_g_tns = d_s4['tr_g']
vl_X_tns = d_s4['vl_X']; vl_y_tns = d_s4['vl_y']; vl_g_tns = d_s4['vl_g']
in_deg_dict = d_s4['in_deg_dict']; p_to_i = d_s4['p_to_i']; p_list_ordered = d_s4['p_list_ordered']
del ld4, d_s4; gc.collect()

gt_dict = {u: set(v for _, v, d in G.out_edges(u, data=True) if d.get('type') == 'cites') for u in te_n}
all_papers_numpy_array = np.array(p_list_ordered)
normalized_text_embeddings_gpu = torch.nn.functional.normalize(x_text_tns, p=2, dim=1).to('cuda', dtype=torch.float32)
all_years_tensor_gpu = torch.tensor([y_dict[p] for p in p_list_ordered], dtype=torch.int32, device='cuda')
author_topology_embeddings_gpu = z_topo.to('cuda', dtype=torch.float32)
text_embeddings_gpu = x_text_tns.to('cuda', dtype=torch.float32)

tr_g_np = tr_g_tns.numpy(); tr_idx = np.argsort(tr_g_np)
tr_X_np = tr_X_tns.numpy()[tr_idx]; tr_y_np = tr_y_tns.numpy()[tr_idx]; tr_g_np = tr_g_np[tr_idx]
del tr_X_tns, tr_y_tns, tr_g_tns, tr_idx; gc.collect()

vl_g_np = vl_g_tns.numpy(); vl_idx = np.argsort(vl_g_np)
vl_X_np = vl_X_tns.numpy()[vl_idx]; vl_y_np = vl_y_tns.numpy()[vl_idx]; vl_g_np = vl_g_np[vl_idx]
del vl_X_tns, vl_y_tns, vl_g_tns, vl_idx; gc.collect()

depths = [6]; l2_regs = [3.0]; losses = ['YetiRank']; res_list = []

for use_hub in [True, False]:
    if not use_hub: tr_X_np[:, 3] = 0.0; vl_X_np[:, 3] = 0.0
    tr_pool = cb.Pool(data=tr_X_np, label=tr_y_np, group_id=tr_g_np)
    vl_pool = cb.Pool(data=vl_X_np, label=vl_y_np, group_id=vl_g_np)
    for dp, l2, lf in itertools.product(depths, l2_regs, losses):
        exp_n = f"CatBoost_HeteroGNN_{lf}_Hub_{use_hub}"
        c_d = os.path.join(b_p, 'experiments', exp_n); os.makedirs(c_d, exist_ok=True)
        m_p = os.path.join(c_d, 'model.cbm'); m_json = os.path.join(c_d, 'metrics.json')
        m_cb = cb.CatBoostRanker(iterations=1500, depth=dp, l2_leaf_reg=l2, loss_function=lf, task_type="GPU", early_stopping_rounds=50, metric_period=50)
        m_cb.fit(tr_pool, eval_set=vl_pool)
        m_cb.save_model(m_p)
        mrr, rec, hits, ndcg = evaluate_full_ranking_causal(m_cb, te_n, gt_dict, y_dict, c_dict, a_dict, in_deg_dict, author_topology_embeddings_gpu, normalized_text_embeddings_gpu, text_embeddings_gpu, all_years_tensor_gpu, p_to_i, all_papers_numpy_array, use_hub, k_cand=100, k=10, batch_size=64)
        res_list.append({"Эксперимент": exp_n, "mrr": mrr, "rec10": rec, "hits10": hits, "ndcg10": ndcg})
        del m_cb; gc.collect()
    del tr_pool, vl_pool; gc.collect()

del normalized_text_embeddings_gpu, all_years_tensor_gpu, author_topology_embeddings_gpu, text_embeddings_gpu, all_papers_numpy_array; gc.collect()
df_res = pd.DataFrame(res_list).sort_values("ndcg10", ascending=False)
from IPython.display import display; display(df_res.style.background_gradient(cmap='RdYlGn', subset=['mrr', 'rec10', 'hits10', 'ndcg10']))

# Скорборд всех моделей

In [ ]:
sb_p = os.path.join(b_p, 'scoreboard.csv')
if os.path.exists(sb_p):
    df_sb = pl.read_csv(sb_p).sort("ndcg10", descending=True)
    display(df_sb)
else:
    print("Скорборд пока пуст.")

## nuke leaderboard

In [ ]:
import os
sb_p = os.path.join(b_p, 'scoreboard.csv')
if os.path.exists(sb_p):
    os.remove(sb_p)
    print("Старый скорборд успешно удален.")
else:
    print("Скорборд не найден.")

# Инференс и оценка

## Загрузка данных

In [ ]:
d_dv = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sb_p = os.path.join(b_p, 'scoreboard.csv')
if not os.path.exists(sb_p): raise RuntimeError("Скорборд не найден")

df_sb = pl.read_csv(sb_p).sort("ndcg10", descending=True)
b_row = df_sb.filter(pl.col("Эксперимент").str.contains("BASELINE"))
b_ndcg = b_row["ndcg10"][0]

t_m = df_sb.filter(pl.col("ndcg10") > b_ndcg).head(2)
w_m = df_sb.filter(pl.col("ndcg10") < b_ndcg).tail(1)

m_d = {"BASELINE": {"exp": b_row["Эксперимент"][0], "ndcg10": b_ndcg}}
for r in t_m.iter_rows(named=True): m_d[f"TOP_{r['Эксперимент'].split('-')[-1]}"] = {"exp": r["Эксперимент"], "ndcg10": r["ndcg10"]}
for r in w_m.iter_rows(named=True): m_d[f"BAD_{r['Эксперимент'].split('-')[-1]}"] = {"exp": r["Эксперимент"], "ndcg10": r["ndcg10"]}

import time; print(f"[{time.strftime('%H:%M:%S')}] Загрузка графа и кэша Этапа 4...")
ld2 = k_rw('r', name2, p='./stage2_pruned'); g_f = ld2['g.pkl']
ld4 = k_rw('r', name4, p='./stage4_final'); tts = ld4['stage4_tensors.pt']

x_f = tts['x_f']; n_v = x_f.size(0); y_a = torch.tensor([d.get('y', 0) for n, d in g_f.nodes(data=True)])
au_d = {i: set(d.get('au', [])) for i, (n, d) in enumerate(g_f.nodes(data=True))}
i_to_id = {i: n for i, n in enumerate(g_f.nodes())}

mt_d = {}
with open('./stage1_raw/d.jsonl', 'r', encoding='utf-8') as f:
    for ln in f:
        if not ln.strip(): continue
        x = json.loads(ln); mt_d[x['i']] = {'y': x.get('y', 0), 'ti': x.get('ti', 'Без названия'), 'ct': x.get('ct', ''), 'ab': x.get('ab', '')}

tr_p_e = tts['tr_p_e']; u_tr = tr_p_e[:, 0]; v_tr = tr_p_e[:, 1]; val_tr = torch.ones(u_tr.size(0), dtype=torch.float32)
u_sym = torch.cat([u_tr, v_tr]); v_sym = torch.cat([v_tr, u_tr])
A_sym = torch.sparse_coo_tensor(torch.stack([u_sym, v_sym]), torch.ones_like(u_sym, dtype=torch.float32), (n_v, n_v)).coalesce()
A_sym.values().clamp_(max=1.0)
out_deg = torch.zeros(n_v).scatter_add_(0, u_tr, val_tr); in_deg = torch.zeros(n_v).scatter_add_(0, v_tr, val_tr)
tot_deg = out_deg + in_deg; w_aa = 1.0 / (torch.log(tot_deg + 1e-8) + 1e-8); w_aa[tot_deg <= 1] = 0
A_aa = torch.sparse_coo_tensor(torch.stack([u_sym, v_sym]), w_aa[v_sym], (n_v, n_v)).coalesce()


## Сравнение моделей

### На вершине из Test

In [ ]:
def g_f_1_vs_all(u):
    import torch
    dt_tns = y_a[u] - y_a; au_u = au_d[u]; au_l = []
    for v in range(n_v):
        au_v = au_d[v]; un = len(au_u | au_v)
        au_l.append(len(au_u & au_v) / un if un > 0 else 0.0)
    au_tns = torch.tensor(au_l, dtype=torch.float32)
    A_u = torch.index_select(A_sym, 0, torch.tensor([u])).to_dense()
    cn_tns = torch.sparse.mm(A_sym, A_u.t()).squeeze(); aa_tns = torch.sparse.mm(A_aa, A_u.t()).squeeze()
    res = torch.stack([dt_tns, au_tns, cn_tns, aa_tns], dim=1)
    res[:, 2:] = torch.log(res[:, 2:] + 1.0)
    return res

def p_a(idx, prf=""):
    u = i_to_id.get(idx, "?"); m = mt_d.get(u, {})
    ab_s = m.get('ab', '—'); ab_s = ab_s[:200] + "..." if len(ab_s) > 200 else ab_s
    print(f"{prf} URL: {u} | Год: {m.get('y', '?')}\n{prf} Название: {m.get('ti', '—')}\n{prf} Теги: {m.get('ct', '—')}\n{prf} Абстракт: {ab_s}\n")

te_p = tts['te_e'][tts['te_y'] == 1]; valid_us = torch.where(torch.bincount(te_p[:, 0]) >= 3)[0]
test_u = valid_us[torch.randint(0, len(valid_us), (1,)).item()].item()
t_c = te_p[te_p[:, 0] == test_u][:, 1].tolist()

print("="*80); print("ИССЛЕДУЕМАЯ СТАТЬЯ (ИСТОЧНИК):"); p_a(test_u); print("="*80)
print(f"ИСТИННЫЕ ЦИТИРОВАНИЯ (Ground Truth) - {len(t_c)} шт.:")
for i, v in enumerate(t_c): p_a(v, prf=f"[{i+1}]")

a_all = g_f_1_vs_all(test_u).to(d_dv); z_all = x_f.to(d_dv); z_u_exp = x_f[test_u].expand(n_v, -1).to(d_dv)

for m_n, cfg in m_d.items():
    print(f"\n---> Модель: {m_n} (NDCG@10: {cfg['ndcg10']:.4f})")
    if "BASELINE" in m_n:
        c_ab = torch.cosine_similarity(z_u_exp[:, :768], z_all[:, :768], dim=1)
        c_ct = torch.cosine_similarity(z_u_exp[:, 768:1536], z_all[:, 768:1536], dim=1)
        c_tp = torch.cosine_similarity(z_u_exp[:, 1537:1665], z_all[:, 1537:1665], dim=1) if z_all.shape[1] > 1537 else torch.zeros(n_v, device=d_dv)
        cn_n = (a_all[:, 2] - a_all[:, 2].min()) / (a_all[:, 2].max() - a_all[:, 2].min() + 1e-8)
        sc = 0.2*c_ab + 0.2*c_ct + 0.3*c_tp + 0.15*a_all[:, 1] + 0.15*cn_n
    else:
        w_p = os.path.join(b_p, 'experiments', cfg['exp'], 'checkpoint.pt')
        ck = torch.load(w_p, map_location=d_dv, weights_only=False); kw = ck['kw']
        m = GM(**kw).to(d_dv); m.load_state_dict(ck['m']); m.eval()
        with torch.no_grad():
            v_t = torch.arange(n_v).to(d_dv); c_x = x_f[:, :kw['n_f']].to(d_dv)
            z_e = m.enc(c_x, v_t, tr_p_e.to(d_dv)); z_u_e = z_e[test_u].expand(n_v, -1)
            sc = m.c(z_u_e, z_e, a_all[:, :kw['n_a']]).squeeze()

    sc[test_u] = -float('inf'); sc[y_a.to(d_dv) > y_a[test_u].to(d_dv)] = -float('inf')
    t_10 = torch.topk(sc, 10).indices.cpu().tolist()
    for i, v in enumerate(t_10):
        is_t = "TRUE" if v in t_c else "FALSE"
        u_id = i_to_id.get(v, '?'); m_i = mt_d.get(u_id, {})
        ab_s = m_i.get('ab', '—'); ab_s = ab_s[:100] + "..." if len(ab_s) > 100 else ab_s
        print(f"  {i+1}. [{sc[v]:.4f}] {is_t} | URL: {u_id} | Год: {m_i.get('y', '?')}\n     Название: {m_i.get('ti', '—')}\n     Теги: {m_i.get('ct', '—')}\n     Абстракт: {ab_s}")


### Создадим новую вершину вручную, и посмотрим, что предложит модель

In [ ]:
print("\n" + "="*80); print("НОВАЯ СТАТЬЯ (Холодный старт):")
n_a = {"ti": "Graph Neural Networks for Financial Fraud Detection", "ab": "In this paper, we review recent advancements in using deep learning and graph neural networks (GNNs) for detecting fraudulent transactions in banking systems.", "ct": "Computer science, Artificial intelligence, Machine learning, Fraud detection, Finance, Graph neural networks", "y": 2026, "au": ["User Author"]}
print(f"Название: {n_a['ti']}\nАбстракт: {n_a['ab']}\n")

st_m = SentenceTransformer('pritamdeka/S-SciBERT-snli-multinli-stsb').to(d_dv)
e_ab = st_m.encode([n_a['ab']], convert_to_tensor=True, device=d_dv); e_ct = st_m.encode([n_a['ct']], convert_to_tensor=True, device=d_dv)
e_tp = torch.zeros(1, 128, device=d_dv); yr_t = ((torch.tensor([[n_a['y']]], device=d_dv).float()) - y_a.float().mean().to(d_dv)) / (y_a.float().std().to(d_dv) + 1e-8)
z_n = torch.cat([e_ab, e_ct, yr_t, e_tp], dim=1); z_n_exp = z_n.expand(n_v, -1)

a_n = torch.stack([n_a['y'] - y_a.to(d_dv), torch.zeros(n_v, device=d_dv), torch.zeros(n_v, device=d_dv), torch.zeros(n_v, device=d_dv), torch.zeros(n_v, device=d_dv), torch.zeros(n_v, device=d_dv), in_deg.to(d_dv)], dim=1)
a_n[:, 2:] = torch.log(a_n[:, 2:] + 1.0)

for m_n, cfg in m_d.items():
    print(f"\n---> Холодный инференс: {m_n}")
    if "BASELINE" in m_n:
        c_ab = torch.cosine_similarity(z_n_exp[:, :768], z_all[:, :768], dim=1)
        c_ct = torch.cosine_similarity(z_n_exp[:, 768:1536], z_all[:, 768:1536], dim=1)
        c_tp = torch.cosine_similarity(z_n_exp[:, 1537:1665], z_all[:, 1537:1665], dim=1) if z_all.shape[1] > 1537 else torch.zeros(n_v, device=d_dv)
        sc = 0.2*c_ab + 0.2*c_ct + 0.3*c_tp + 0.15*a_n[:, 1]
    else:
        w_p = os.path.join(b_p, 'experiments', cfg['exp'], 'checkpoint.pt')
        ck = torch.load(w_p, map_location=d_dv, weights_only=False); kw = ck['kw']
        m = GM(**kw).to(d_dv); m.load_state_dict(ck['m']); m.eval()
        with torch.no_grad():
            c_x = x_f[:, :kw['n_f']].to(d_dv); c_z_n = z_n[:, :kw['n_f']].to(d_dv)
            z_e = m.enc(c_x, torch.arange(n_v).to(d_dv), tr_p_e.to(d_dv))
            z_n_e = m.enc(c_z_n, torch.tensor([-1]).to(d_dv), torch.empty((0,2), dtype=torch.long).to(d_dv))
            z_n_e_exp = z_n_e.expand(n_v, -1)
            sc = m.c(z_n_e_exp, z_e, a_n[:, :kw['n_a']]).squeeze()

    sc[y_a.to(d_dv) > n_a['y']] = -float('inf')
    t_10 = torch.topk(sc, 10).indices.cpu().tolist()
    for i, v in enumerate(t_10):
        u_id = i_to_id.get(v, '?'); m_i = mt_d.get(u_id, {})
        ab_s = m_i.get('ab', '—'); ab_s = ab_s[:100] + "..." if len(ab_s) > 100 else ab_s
        print(f"  {i+1}. [{sc[v]:.4f}] URL: {u_id} | Год: {m_i.get('y', '?')}\n     Название: {m_i.get('ti', '—')}\n     Теги: {m_i.get('ct', '—')}\n     Абстракт: {ab_s}")